# Ultimate Downloader

This notebook embeds a local copy of `ultimate_downloader.py` — includes the atomic session-save fix, the cleaned-up episode-detection regex, and TorBox torrent folder-structure preservation (multi-file torrents keep their internal folder layout under `Downloads/<Torrent Name>/...` when Auto-organise is off). Nothing is fetched from GitHub at runtime.

Just run the cell below. The UI will appear automatically once Drive is mounted.

**Tips**
- ⚙️ Settings — enter Gofile / Real-Debrid / TorBox tokens, or store them as Colab Secrets (recommended).
- Uncheck **Auto-organise** to keep original filenames instead of Plex-style renaming.
- Check **Upload to Drive via API (bypass the mount)** for reliable, verified uploads — especially for large torrent batches.

In [ ]:
script = 'import os\nimport re\nimport json\nimport requests\nimport subprocess\nimport shutil\nimport time\nimport difflib\nimport queue\nfrom typing import Optional, Tuple, List, Dict, Any, Callable\nfrom dataclasses import dataclass, field, fields, asdict\nfrom datetime import datetime\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nfrom threading import Lock, RLock, local\nfrom uuid import uuid4\nimport ipywidgets as widgets\nfrom IPython.display import display, clear_output\nfrom urllib.parse import urlparse, unquote, parse_qs\ntry:\n    from google.colab import drive\nexcept ImportError:\n    drive = None  # Running outside Colab (local testing) — Drive mount disabled\n\n# --- COLAB SECRETS HELPER ---\ndef get_colab_secret(key: str, default: str = "") -> str:\n    """Retrieve a secret from Colab secrets, return default if not found."""\n    try:\n        from google.colab import userdata\n        return userdata.get(key)\n    except (ImportError, ModuleNotFoundError):\n        return default\n    except Exception as e:\n        # This catches SecretNotFoundError and NotebookAccessError\n        return default\n\n# --- SETTINGS CHANGE TRACKING ---\n# _loading_settings: True while load_dir_settings/check_and_load_secrets write widget\n# values programmatically, so the save-on-change observers neither rewrite settings.json\n# mid-load (which would persist half-loaded state over the file) nor mistake those\n# writes for user edits. _user_touched_settings: widgets the user actually changed this\n# session — the post-mount settings reload leaves these alone, so a stored value never\n# overrides a live choice.\n_loading_settings = False\n_user_touched_settings: set = set()\n\n\ndef check_and_load_secrets():\n    """Re-check secrets and populate fields if they were empty on initial load."""\n    try:\n        from google.colab import userdata\n    except (ImportError, ModuleNotFoundError):\n        return\n    # (widget, secret name) pairs — widgets are defined below in the UI section\n    secret_fields = [\n        (token_rd, \'RD_TOKEN\'),\n        (token_tb, \'TB_TOKEN\'),\n        (token_gf, \'GOFILE_TOKEN\'),\n        (token_tmdb, \'TMDB_API_KEY\'),\n        (token_fshare_email, \'FSHARE_EMAIL\'),\n        (token_fshare_password, \'FSHARE_PASSWORD\'),\n    ]\n    global _loading_settings\n    _loading_settings = True  # secret writes are programmatic, not user edits\n    try:\n        for widget, secret_name in secret_fields:\n            if widget.value:\n                continue\n            try:\n                val = userdata.get(secret_name)\n                if val:\n                    widget.value = val\n                    print(f"🔑 {secret_name} loaded from Colab Secrets")\n            except Exception:\n                pass\n    finally:\n        _loading_settings = False\n\n# --- CONFIGURATION ---\nCOLAB_ROOT = "/content/"\nDRIVE_BASE = f"{COLAB_ROOT}drive/My Drive/"\nUD_CONFIG_PATH = f"{DRIVE_BASE}Ultimate Downloader/"  # Config folder for session & history files\nDRIVE_TV_PATH = "TV Shows"\nDRIVE_MOVIE_PATH = "Movies"\nDRIVE_YOUTUBE_PATH = "YouTube"\nDRIVE_DOWNLOADS_PATH = "Downloads"\nDRIVE_ANIME_SERIES_PATH = "Anime Series"\nDRIVE_ANIME_MOVIES_PATH = "Anime Movies"\nMIN_FILE_SIZE_MB = 10\nKEEP_EXTENSIONS = {\'.srt\', \'.ass\', \'.sub\', \'.vtt\'}\nSESSION_FILE = f"{UD_CONFIG_PATH}session.json"\nHISTORY_FILE = f"{UD_CONFIG_PATH}history.json"\nSETTINGS_FILE = f"{UD_CONFIG_PATH}settings.json"\nFSHARE_COOKIE_FILE = f"{UD_CONFIG_PATH}fshare_cookies.json"\nTMDB_CACHE_FILE = f"{UD_CONFIG_PATH}tmdb_cache.json"\nCOOKIE_PATH = f"{COLAB_ROOT}cookies.txt"\nMAX_CONCURRENT_DEFAULT = 3\n\n# API Configuration\nREQUEST_TIMEOUT = 30  # Default timeout for HTTP requests\nGOFILE_WEBSITE_TOKEN = "4fd6sg89d7s6"  # Website token for Gofile API - update if authentication fails\nTORBOX_API_BASE = "https://api.torbox.app/v1/api"  # TorBox API base URL\nTMDB_API_BASE = "https://api.themoviedb.org/3"  # TMDB v3 API (metadata matching)\nTMDB_MATCH_THRESHOLD = 0.60  # Minimum title similarity to accept a search result\nTMDB_QUERY_CACHE_MAX = 500   # Persistent query cache cap (oldest dropped first)\nTMDB_CACHE_VERSION = 2       # Bump when matching gets smarter so cached misses are retried\nTMDB_CLEARED = {\'cleared\': True}  # DownloadTask.tmdb_override value meaning "force regex, no TMDB"\n\n# Known resolution values (for filename parsing)\nKNOWN_RESOLUTIONS = {360, 480, 540, 720, 1080, 1440, 2160, 4320}\nYEAR_RANGE = range(1900, 2100)\n\n# Debrid-supported file hosts (route through selected debrid service when token available)\nDEBRID_SUPPORTED_HOSTS = {\n    \'1fichier.com\', \'4shared.com\', \'alfafile.net\', \'clicknupload.org\', \'ddownload.com\',\n    \'dailymotion.com\', \'dropbox.com\', \'filefactory.com\', \'hexupload.net\', \'hitfile.net\',\n    \'k2s.cc\', \'keep2share.cc\', \'mediafire.com\', \'mega.nz\', \'mixdrop.co\', \'nitroflare.com\',\n    \'oboom.com\', \'rapidgator.net\', \'redtube.com\', \'scribd.com\', \'sendspace.com\',\n    \'solidfiles.com\', \'soundcloud.com\', \'streamtape.com\', \'turbobit.net\', \'ulozto.net\',\n    \'upload.ee\', \'uploaded.net\', \'uptobox.com\', \'userscloud.com\', \'vidoza.net\',\n    \'vimeo.com\', \'wetransfer.com\', \'wipfiles.net\', \'worldbytez.com\', \'youporn.com\',\n}\nRD_SUPPORTED_HOSTS = DEBRID_SUPPORTED_HOSTS  # Backward-compatible alias\n\n# --- DOWNLOAD TASK DATACLASS ---\n@dataclass\nclass DownloadTask:\n    url: str  # Direct download URL (may be resolved API URL)\n    filename: str\n    source: str\n    link_type: str  # gofile, pixeldrain, direct, youtube, mega, rd, tb\n    id: str = field(default_factory=lambda: str(uuid4()))  # Unique ID for tracking\n    status: str = "pending"  # pending, downloading, moving (downloaded, Drive move pending), done, failed, skipped\n    error: Optional[str] = None\n    cookie: Optional[str] = None\n    original_url: Optional[str] = None  # Original user-provided URL (for re-resolving on resume)\n    tmdb_override: Optional[dict] = None  # Manual TMDB correction (persisted); {\'cleared\': True} forces regex\n    season_override: Optional[int] = None  # Manual season number (persisted); None = detect from filename/TMDB\n    episode_override: Optional[int] = None  # Manual episode number (persisted); set by queue 🔢 Renumber\n    episode_end_override: Optional[int] = None  # Manual range end (persisted); with episode_override N, names the file ENN-Eend\n    name_override: Optional[str] = None  # Manual name for organise (persisted); set by queue ✏️ Force Name\n    year_override: Optional[str] = None  # Manual year accompanying name_override (persisted)\n    route_override: Optional[str] = None  # Manual destination category (persisted); set by queue 🎯 Route as\n    part_override: Optional[int] = None  # Manual part suffix (persisted); N = force -ptN, 0 = force none\n    relative_path: Optional[str] = None  # Torrent-internal subfolder (e.g. "Season 1"), \'\' = torrent root, None = not from a torrent\n    torrent_name: Optional[str] = None  # Parent torrent/container name; used as the top-level folder when preserving structure\n\n_TASK_FIELDS = {f.name for f in fields(DownloadTask)}\n\n# Link types with dedicated sequential processors. Every other link type (gofile,\n# pixeldrain, direct, rd, tb, fshare, mediafire, 1fichier, ...) downloads through\n# the parallel aria2 pool — partitioning by exclusion so new resolver types are\n# never silently dropped.\nSEQUENTIAL_LINK_TYPES = {\'youtube\', \'mega\', \'magnet\', \'magnet_file\', \'tb_magnet_file\'}\n\ndef task_from_dict(data: Dict[str, Any]) -> DownloadTask:\n    """Build a DownloadTask from a session dict, ignoring unknown keys from older versions."""\n    return DownloadTask(**{k: v for k, v in data.items() if k in _TASK_FIELDS})\n\n# --- THREAD SAFETY ---\nprogress_lock = Lock()\nprint_lock = Lock()  # Prevent interleaved print output from parallel threads\n_session_save_lock = Lock()  # session.json is written from the main thread AND the Drive mover thread\ndownload_stats: Dict[str, Dict[str, float]] = {}  # task_id -> {\'pct\': 0-100, \'speed_mbs\': MB/s}\n# Sentinel returned by download_with_aria2 when the file is already in Drive (a skip,\n# not a failure). Callers must check `is DUPLICATE_SKIP` before treating a falsy result\n# as failed, so resume doesn\'t retry already-complete files forever.\nDUPLICATE_SKIP = "__duplicate_skip__"\nstop_monitor = False  # Flag to stop progress monitor thread\nbatch_start_time: Optional[float] = None  # Track when batch started for overall ETA\nlast_display_speed: float = 0.0  # Persist last known speed to prevent flickering\n\n# --- BATCH CANCELLATION ---\n# Stop works via the kernel interrupt (the ■ button next to the running cell / Runtime\n# → Interrupt), which raises KeyboardInterrupt on the main thread. The pipeline catches\n# it, terminates the active aria2/megadl subprocesses, and saves the session so\n# Resume/Retry can continue. Workers also poll _cancel_requested so anything not yet\n# started stays "pending" and in-flight items become "failed (Cancelled by user)".\n# (A widget Stop *button* can\'t work while a synchronous download blocks the kernel\'s\n# shell thread — the click would never be delivered until the batch already finished.)\n_cancel_requested = False\n_active_procs: Dict[str, Any] = {}  # key -> running subprocess.Popen\n_active_procs_lock = Lock()\n\ndef cancel_requested() -> bool:\n    return _cancel_requested\n\ndef stop_active_downloads():\n    """Signal cancellation and terminate every registered subprocess. Called from the\n    pipeline\'s KeyboardInterrupt handlers when the user interrupts the kernel."""\n    global _cancel_requested\n    _cancel_requested = True\n    with _active_procs_lock:\n        procs = list(_active_procs.values())\n    for p in procs:\n        try:\n            p.terminate()\n        except Exception:\n            pass\n\ndef _register_proc(key: str, process):\n    with _active_procs_lock:\n        _active_procs[key] = process\n\ndef _unregister_proc(key: str):\n    with _active_procs_lock:\n        _active_procs.pop(key, None)\n\ndef _reset_cancel_state():\n    """Clear the cancel flag and process registry between batches."""\n    global _cancel_requested, _disk_stall_abort\n    _cancel_requested = False\n    _disk_stall_abort = False\n    with _active_procs_lock:\n        _active_procs.clear()\n\n# --- DISK-SPACE GUARD ---\n# Colab hard-terminates the runtime when the local disk fills up. Debrid\n# downloads are faster than the FUSE move to Drive, so a big batch of large\n# files can fill /content before the mover drains it. The guard delays new\n# downloads (and pauses running ones) until moves free enough space, instead\n# of letting the session die.\nDISK_FLOOR_GB = 4.0         # Pause a running download when free space drops below this\nDISK_START_GB = 10.0        # Don\'t start a new download with less than this free\nDISK_CHECK_SECS = 5         # How often a running download polls free space\nDISK_WAIT_POLL_SECS = 10    # How often a blocked download re-checks free space\nDISK_WAIT_STALL_SECS = 600  # Give up when space stops improving and nothing can free it\n\n# --- RATE-LIMITED HOSTS ---\n# Hosts that meter by request count, not just bandwidth. aria2\'s stock behaviour\n# here (16 connections, and a 1M split floor) fires a burst of fresh Range\n# requests as a file nears completion — worst on resume, where the whole\n# remainder is small and still gets split every way at once. TorBox answers that\n# burst with HTTP 429, so the last few percent of a file never lands, and the\n# partial it leaves behind is exactly the state that triggers the burst again.\n#\n# Connection counts here are deliberately small: TorBox sells concurrency by tier\n# (the entry plan allows 3 slots), so the budget that matters is\n# Parallel DLs x connections-per-file, not either number alone. Two connections\n# leaves headroom at any slider setting, and costs no throughput — a throttled\n# transfer runs far slower than a clean single-stream one.\n# Measured on the entry tier: 3 parallel x 2 connections (6 at once) runs clean,\n# so a "slot" meters transfers, not TCP connections. Don\'t tighten below this\n# without evidence; the old 16-connection default was the actual fault.\nRATE_LIMITED_HOSTS = (\'torbox.app\', \'tb-cdn.io\')\nTHROTTLE_BACKOFF_SECS = 30  # Cool-off after a 429 (multiplied by the wait count)\nMAX_THROTTLE_WAITS = 4      # 429 cool-offs allowed before a download really fails\n\n_disk_guard_lock = Lock()\n_moves_in_flight = 0        # Drive transfers currently copying — each frees local space when done\n_disk_stall_abort = False   # A wait already timed out this batch; fail fast instead of re-stalling\n\n# --- DRIVE TRANSFER THROUGHPUT ---\n# Measured on the Drive API path: 3 concurrent uploads sustained ~45 MB/s each, so\n# per-stream throughput holds as streams are added and aggregate scales with the\n# mover count. Where it stops scaling is empirical, hence the slider and this\n# summary: raise movers until the aggregate figure stops moving. (Through the FUSE\n# mount none of this applies — that path caps out around 24 MB/s in total.)\nDRIVE_MOVERS_DEFAULT = 3    # Concurrent Drive uploads in overlap mode\nMOVE_QUEUE_DEPTH = 2        # Finished files allowed to queue on top of the in-flight ones\n\n_drive_xfer_lock = Lock()\n_drive_xfer_bytes = 0          # Bytes pushed to Drive this batch\n_drive_xfer_secs = 0.0         # Summed per-file transfer time (> wall clock when parallel)\n_drive_xfer_files = 0\n_drive_xfer_first = None       # Earliest transfer start / latest end, for wall-clock rate\n_drive_xfer_last = None\n\ndef _reset_drive_xfer_stats():\n    """Clear the per-batch Drive throughput counters."""\n    global _drive_xfer_bytes, _drive_xfer_secs, _drive_xfer_files\n    global _drive_xfer_first, _drive_xfer_last\n    with _drive_xfer_lock:\n        _drive_xfer_bytes = 0\n        _drive_xfer_secs = 0.0\n        _drive_xfer_files = 0\n        _drive_xfer_first = None\n        _drive_xfer_last = None\n\ndef _record_drive_transfer(nbytes: int, started: float, ended: float):\n    """Record one completed Drive transfer (called from every mover thread)."""\n    global _drive_xfer_bytes, _drive_xfer_secs, _drive_xfer_files\n    global _drive_xfer_first, _drive_xfer_last\n    with _drive_xfer_lock:\n        _drive_xfer_bytes += nbytes\n        _drive_xfer_secs += max(ended - started, 0.0)\n        _drive_xfer_files += 1\n        if _drive_xfer_first is None or started < _drive_xfer_first:\n            _drive_xfer_first = started\n        if _drive_xfer_last is None or ended > _drive_xfer_last:\n            _drive_xfer_last = ended\n\ndef _drive_xfer_summary() -> Optional[str]:\n    """One-line Drive throughput report, or None if nothing was transferred.\n\n    Aggregate = bytes / wall clock spent transferring — the number that actually\n    got faster. Per-stream = bytes / summed per-file time, which more movers do\n    not change. Their ratio is the parallel speed-up the mover count bought, so\n    comparing two batches at different mover counts is a direct A/B."""\n    with _drive_xfer_lock:\n        if not _drive_xfer_files or _drive_xfer_first is None:\n            return None\n        total_mb = _drive_xfer_bytes / (1024 * 1024)\n        wall = max(_drive_xfer_last - _drive_xfer_first, 0.001)\n        stream_secs = max(_drive_xfer_secs, 0.001)\n        files = _drive_xfer_files\n    agg = total_mb / wall\n    per = total_mb / stream_secs\n    return (f"📤 Drive transfers: {files} file(s), {total_mb / 1024:.2f} GB in {wall / 60:.1f} min "\n            f"— {agg:.1f} MB/s aggregate, {per:.1f} MB/s per stream ({agg / per:.1f}x from parallelism)")\ndef _disk_free_gb() -> float:\n    """Free space on the Colab local disk in GB (inf when unmeasurable, e.g. locally)."""\n    try:\n        return shutil.disk_usage(COLAB_ROOT).free / (1024 ** 3)\n    except Exception:\n        return float(\'inf\')\n\ndef _move_in_flight() -> bool:\n    return _moves_in_flight > 0\n\ndef _downloads_running() -> bool:\n    """True while any registered download subprocess is alive — it will either\n    finish (and free space via its Drive move) or pause itself at the floor."""\n    with _active_procs_lock:\n        return bool(_active_procs)\n\ndef _mark_disk_gave_up(task_id):\n    """Tag a task\'s stats so download_worker reports an accurate failure reason."""\n    if task_id and not _cancel_requested:\n        with progress_lock:\n            download_stats.setdefault(task_id, {\'pct\': 0.0, \'speed_mbs\': 0.0})[\'disk_gave_up\'] = True\n\ndef _wait_for_disk_space(needed_gb: float, label: str = "", task_id: Optional[str] = None) -> bool:\n    """Block until at least needed_gb is free on the local disk.\n\n    Returns True once enough space is free. Returns False when cancelled, or when\n    free space stops improving with nothing left that could free it (no Drive move\n    in flight, no download still running) — waiting longer would deadlock, e.g.\n    every worker paused on a huge partial. The caller fails that one download and\n    the session survives; partials stay on disk for Resume/Retry.\n    """\n    global _disk_stall_abort\n    free = _disk_free_gb()\n    if free >= needed_gb:\n        _disk_stall_abort = False  # Space recovered — future waits are worth trying again\n        return True\n    if _disk_stall_abort and not _move_in_flight() and not _downloads_running():\n        with print_lock:\n            print(f"   ⏭️ Skipped — local disk still full ({free:.1f} GB free) and nothing is freeing space")\n        return False\n    with print_lock:\n        print(f"   ⏸️ Low disk: {free:.1f} GB free (need {needed_gb:.0f} GB) — waiting for Drive moves to free space...")\n    if task_id:\n        with progress_lock:\n            stats = download_stats.setdefault(task_id, {\'pct\': 0.0, \'speed_mbs\': 0.0})\n            stats[\'speed_mbs\'] = 0.0\n            stats[\'disk_wait\'] = True\n    try:\n        best = free\n        last_gain = time.time()\n        while not _cancel_requested:\n            for _ in range(DISK_WAIT_POLL_SECS):\n                time.sleep(1)\n                if _cancel_requested:\n                    return False\n            free = _disk_free_gb()\n            if free >= needed_gb:\n                _disk_stall_abort = False\n                with print_lock:\n                    print(f"   ▶️ Disk space recovered ({free:.1f} GB free) — resuming {label or \'download\'}")\n                return True\n            if free > best + 0.5 or _move_in_flight() or _downloads_running():\n                # Space is draining, or something is still running that will free\n                # space when it completes — keep waiting\n                best = max(best, free)\n                last_gain = time.time()\n            elif time.time() - last_gain > DISK_WAIT_STALL_SECS:\n                _disk_stall_abort = True\n                with print_lock:\n                    print(f"   ❌ Still only {free:.1f} GB free after {DISK_WAIT_STALL_SECS // 60} min with nothing freeing space — giving up on {label or \'this download\'}")\n                    print(f"      💡 Partial files are kept: Retry re-uses them. For very large files, lower Parallel DLs.")\n                return False\n        return False  # Cancelled while waiting\n    finally:\n        if task_id:\n            with progress_lock:\n                stats = download_stats.get(task_id)\n                if stats:\n                    stats.pop(\'disk_wait\', None)\n\n# --- COLAB KEEP-ALIVE ---\n_keep_alive_stop = False\n_keep_alive_thread = None  # Handle to the running keep-alive thread (only one at a time)\n_rd_magnet_delay = 0  # Adaptive delay (seconds) between RD addMagnet calls; auto-set when rate-limited\n\ndef _keep_alive_worker(interval: int = 120):\n    """Background thread: simulate Colab interaction to prevent idle timeout."""\n    from IPython.display import display as ipy_display, Javascript\n    while not _keep_alive_stop:\n        try:\n            ipy_display(Javascript(\'\'\'\n                (function() {\n                    // Click the connect button to reset Colab\'s idle timer\n                    var btn = document.querySelector("colab-connect-button");\n                    if (btn) { btn.click(); }\n                    console.log("Colab keep-alive: " + new Date().toLocaleTimeString());\n                })();\n            \'\'\'))\n        except Exception:\n            pass\n        # Sleep in short increments so stop_keep_alive() takes effect promptly\n        waited = 0\n        while waited < interval and not _keep_alive_stop:\n            time.sleep(5)\n            waited += 5\n\ndef start_keep_alive():\n    """Start background keep-alive thread to prevent Colab idle disconnection.\n    Idempotent: does nothing if a keep-alive thread is already running."""\n    global _keep_alive_stop, _keep_alive_thread\n    if _keep_alive_thread is not None and _keep_alive_thread.is_alive():\n        return\n    _keep_alive_stop = False\n    import threading\n    _keep_alive_thread = threading.Thread(target=_keep_alive_worker, daemon=True)\n    _keep_alive_thread.start()\n\ndef stop_keep_alive():\n    """Stop the background keep-alive thread."""\n    global _keep_alive_stop\n    _keep_alive_stop = True\n\n# --- UI ELEMENTS ---\ntoken_gf = widgets.Text(description=\'Gofile:\', placeholder=\'Optional\', value=get_colab_secret(\'GOFILE_TOKEN\'), style={\'description_width\': \'130px\'}, layout=widgets.Layout(width=\'320px\'))\ntoken_tmdb = widgets.Text(description=\'TMDB:\', placeholder=\'API Key (optional)\', value=get_colab_secret(\'TMDB_API_KEY\'), style={\'description_width\': \'130px\'}, layout=widgets.Layout(width=\'320px\'))\ntmdb_enabled_checkbox = widgets.Checkbox(value=True, description=\'TMDB matching\', tooltip=\'Match filenames against TMDB for canonical names, years, and season mapping\', indent=False, layout=widgets.Layout(width=\'150px\'))\ntoken_rd = widgets.Text(description=\'RD Token:\', placeholder=\'Real-Debrid API Key\', value=get_colab_secret(\'RD_TOKEN\'), style={\'description_width\': \'130px\'}, layout=widgets.Layout(width=\'320px\'))\ntoken_tb = widgets.Text(description=\'TB Token:\', placeholder=\'TorBox API Key\', value=get_colab_secret(\'TB_TOKEN\'), style={\'description_width\': \'130px\'}, layout=widgets.Layout(width=\'320px\'))\n# Debrid sits second in its row, after the 280px auto-organise checkbox — the same slot\n# the 280px Name field occupies in the row below. Same 50px label width as Year:, so the\n# dropdown and the Year box line up without margin tweaks.\ndebrid_service_toggle = widgets.Dropdown(\n    options=[\'None\', \'Real-Debrid\', \'TorBox\'],\n    value=\'None\',\n    description=\'Debrid:\',\n    tooltip=\'Select debrid service for premium links & magnets\',\n    style={\'description_width\': \'75px\'},  # matches Auto Retry so both label+field pairs align\n    layout=widgets.Layout(width=\'215px\')\n)\ntoken_fshare_email = widgets.Text(description=\'FShare:\', placeholder=\'Email\', value=get_colab_secret(\'FSHARE_EMAIL\'), style={\'description_width\': \'130px\'}, layout=widgets.Layout(width=\'320px\'))\ntoken_fshare_password = widgets.Password(description=\'Password:\', placeholder=\'FShare Password\', value=get_colab_secret(\'FSHARE_PASSWORD\'), style={\'description_width\': \'130px\'}, layout=widgets.Layout(width=\'320px\'))\n# Season match (default): bare episode numbers on TMDB-matched shows are converted to\n# SxxEyy via per-season counts (e.g. "One Piece - 1085" → S19E..). Absolute: keep the\n# number as-is (S01E1085) for libraries scanned with absolute ordering. Only affects\n# TMDB-matched shows whose filenames carry no explicit SxxExx/NxN season marker, so it\n# lives in ⚙️ Settings beside the TMDB controls rather than in the per-batch input row.\nepisode_numbering_toggle = widgets.Dropdown(\n    options=[\'Season match\', \'Absolute\'],\n    value=\'Season match\',\n    description=\'Episode numbering:\',\n    tooltip=\'Season match: convert absolute episode numbers to SxxEyy using TMDB season data. Absolute: keep the absolute number as-is (e.g. S01E1085). Applies only to TMDB-matched shows with no season marker in the filename.\',\n    style={\'description_width\': \'130px\'},  # matches the settings key-field label column\n    layout=widgets.Layout(width=\'270px\')\n)\nplaylist_selection = widgets.Text(description=\'Playlist:\', placeholder=\'e.g. 1,3,5-10 (Empty=All)\', style={\'description_width\': \'60px\'}, layout=widgets.Layout(width=\'220px\'))\nconcurrent_slider = widgets.IntSlider(value=MAX_CONCURRENT_DEFAULT, min=1, max=5, description=\'Parallel DLs:\', style={\'description_width\': \'80px\'}, layout=widgets.Layout(width=\'280px\'))\nauto_retry_input = widgets.Text(description=\'Auto Retry:\', placeholder=\'e.g. 3 (Empty=Off)\', tooltip=\'Automatically re-run 🔁 Retry Failed when a batch ends with failures, up to this many extra passes. Empty or 0 = off. Stops early once nothing is left failed; a kernel interrupt cancels the chain.\', style={\'description_width\': \'75px\'}, layout=widgets.Layout(width=\'215px\'))\n# Opt-in: finished files move to Drive on a dedicated mover thread so the pool starts\n# the next download immediately. Off = each worker moves its own file before taking\n# another, which doubles as backpressure when Colab disk or debrid slots are tight.\nasync_moves_checkbox = widgets.Checkbox(value=False, description=\'Overlap Drive moves with downloads\', tooltip=\'Finished files move to Drive in the background while the next download starts immediately. Uses more local disk (up to 3 finished files can queue) and keeps more debrid slots busy — leave off if Colab disk space or your provider\\\'s concurrent slots are tight.\', indent=False, layout=widgets.Layout(width=\'320px\'))\n# Opt-in: send finished files to Drive over the REST API instead of writing them\n# through the mount. The mount is a write-back cache, so a write there returns\n# before Drive actually has the file; the API returns only when the upload is done.\ndrive_api_checkbox = widgets.Checkbox(value=False, description=\'Upload to Drive via API (bypass the mount)\', tooltip=\'Writes through the Drive mount are staged locally and uploaded in the background, so a file can be reported complete minutes before it reaches Drive — and is lost outright if the runtime ends first. This uploads over the Drive API instead: slower to report, but "complete" means complete. Falls back to the mount automatically if the API is unavailable.\', indent=False, layout=widgets.Layout(width=\'360px\'))\n# How many Drive uploads run at once in overlap mode. Separate from Parallel DLs:\n# that slider is bounded by the debrid plan\'s concurrent slots, this one by how many\n# uploads Drive will take and how much local disk the pending backlog needs.\ndrive_movers_slider = widgets.IntSlider(value=DRIVE_MOVERS_DEFAULT, min=1, max=8, description=\'Drive movers:\', tooltip=\'Concurrent Drive uploads when overlap is on. Only meaningful with the Drive API enabled — writes to the mount are drained by a single background uploader no matter how many movers run. Each in-flight upload holds one finished file on local disk, so higher values need more free space. Raise it until the aggregate MB/s printed at the end of a batch stops improving.\', style={\'description_width\': \'95px\'}, layout=widgets.Layout(width=\'300px\'))\n# Auto-organisation checkbox for main UI. Width matches the Parallel DLs slider\n# (280px) so the Debrid and Auto Retry label+field pairs align across both rows.\nauto_organize_checkbox = widgets.Checkbox(value=True, description=\'Auto-organise\', tooltip=\'Auto-rename and organise files. Uncheck to save with original filenames to Downloads.\', indent=False, layout=widgets.Layout(width=\'280px\'))\n\ntext_area = widgets.Textarea(description=\'Links:\', placeholder=\'Paste Links Here (Transfer.it, Mega, YouTube, etc.)...\', layout=widgets.Layout(width=\'98%\', height=\'150px\'))\nbtn = widgets.Button(description="Resolve Links", button_style=\'success\', icon=\'search\')\nbtn_quick = widgets.Button(description="Quick Download", button_style=\'primary\', icon=\'bolt\', tooltip=\'Download immediately without queue preview\', layout=widgets.Layout(width=\'140px\'))\nbtn_resume = widgets.Button(description="Resume Previous Session", button_style=\'warning\', icon=\'play\', layout=widgets.Layout(display=\'none\', width=\'180px\'))\nbtn_restart = widgets.Button(description="🔄 Restart Runtime", button_style=\'danger\', tooltip=\'Restart runtime then Resume Previous Session\', layout=widgets.Layout(display=\'none\'))\n# Stop is done via the kernel interrupt (see BATCH CANCELLATION), so this is a hint,\n# not a button — a widget button can\'t be clicked while a synchronous download blocks\n# the kernel. Shown only during an active batch.\nstop_hint = widgets.HTML(value="", layout=widgets.Layout(display=\'none\'))\nbtn_retry = widgets.Button(description="🔁 Retry Failed", button_style=\'warning\', tooltip=\'Retry failed downloads from the saved session\', layout=widgets.Layout(display=\'none\', width=\'130px\'))\nbtn_history = widgets.Button(description="📜 History", button_style=\'\', tooltip=\'View Download History\', layout=widgets.Layout(width=\'100px\'))\nbtn_settings = widgets.Button(description="⚙️ Settings", button_style=\'\', tooltip=\'Settings & Manage Files\', layout=widgets.Layout(width=\'105px\'))\nbtn_about = widgets.Button(description="ℹ️ About", button_style=\'\', tooltip=\'About\', layout=widgets.Layout(width=\'90px\'))\nprogress_bar = widgets.FloatProgress(value=0.0, min=0.0, max=100.0, description=\'Idle\', bar_style=\'info\', layout=widgets.Layout(width=\'98%\'))\nstatus_label = widgets.HTML(value="")\n\n# --- PER-DOWNLOAD PROGRESS BARS ---\n# Individual bars for each parallel download, shown in a collapsible accordion.\n_per_task_bars: Dict[str, widgets.FloatProgress] = {}  # task_id -> bar widget\n_per_task_done_at: Dict[str, float] = {}  # task_id -> completion timestamp (for linger)\n_PER_TASK_LINGER = 2.0  # seconds a completed bar stays visible before removal\n\n_per_task_box = widgets.VBox([], layout=widgets.Layout(width=\'100%\'))\n_per_task_accordion = widgets.Accordion(children=[_per_task_box])\n_per_task_accordion.set_title(0, \'📥 Downloads\')\n_per_task_accordion.selected_index = None  # collapsed by default\n_per_task_accordion.layout = widgets.Layout(width=\'98%\', display=\'none\')  # hidden until needed\n\n# --- SETTINGS/MANAGEMENT UI ---\nbtn_clear_history = widgets.Button(description="Clear Download History", button_style=\'warning\', tooltip=\'Delete history.json\', layout=widgets.Layout(width=\'180px\'))\nbtn_clear_ytarchive = widgets.Button(description="Clear YT Archive", button_style=\'warning\', tooltip=\'Delete yt_history.txt (allows re-downloading videos)\', layout=widgets.Layout(width=\'150px\'))\nbtn_clear_session = widgets.Button(description="Clear Session", button_style=\'warning\', tooltip=\'Delete session.json\', layout=widgets.Layout(width=\'120px\'))\nbtn_settings_close = widgets.Button(description="Close", button_style=\'\', layout=widgets.Layout(width=\'70px\'))\nsettings_status = widgets.HTML(value="")\n\n# Cookie UI (experimental)\nbtn_upload_cookies = widgets.Button(description="📤 Upload Cookies", button_style=\'info\', tooltip=\'Upload cookies.txt for YouTube Premium (experimental)\', layout=widgets.Layout(width=\'140px\'))\nbtn_clear_cookies = widgets.Button(description="🗑️ Clear Cookies", button_style=\'warning\', tooltip=\'Delete cookies.txt (fixes format errors)\', layout=widgets.Layout(width=\'130px\'))\ncookie_status = widgets.HTML(value="")\n\n# Quick Download subtitle settings\nquick_dl_subs_checkbox = widgets.Checkbox(value=False, description=\'Include Subtitles in Quick Downloads\', indent=False, layout=widgets.Layout(width=\'250px\'))\nquick_dl_subtitle_langs = widgets.SelectMultiple(\n    options=[(\'English\', \'en\'), (\'Vietnamese\', \'vi\'), (\'Chinese\', \'zh\'), (\'Japanese\', \'ja\'), (\'Korean\', \'ko\'),\n             (\'Thai\', \'th\'), (\'Indonesian\', \'id\'), (\'Spanish\', \'es\'), (\'French\', \'fr\'), (\'German\', \'de\'), (\'Portuguese\', \'pt\'), (\'Russian\', \'ru\')],\n    value=[\'en\', \'vi\'],\n    description=\'Languages:\',\n    rows=6,  # whole rows only — a fixed pixel height clipped the last visible item\n    layout=widgets.Layout(width=\'300px\')\n)\n\n# Embedded subtitle extraction settings — auto-extract toggle + retroactive library\n# scan tool (merged from the standalone Smart Subtitle Extractor)\nauto_extract_subs_checkbox = widgets.Checkbox(value=False, description=\'Extract embedded subs after download\', indent=False, layout=widgets.Layout(width=\'250px\'))\nextract_sub_langs = widgets.SelectMultiple(\n    options=[(\'English\', \'en\'), (\'Vietnamese\', \'vi\'), (\'Chinese\', \'zh\'), (\'Japanese\', \'ja\'), (\'Korean\', \'ko\'),\n             (\'Thai\', \'th\'), (\'Indonesian\', \'id\'), (\'Spanish\', \'es\'), (\'French\', \'fr\'), (\'German\', \'de\'), (\'Portuguese\', \'pt\'), (\'Russian\', \'ru\')],\n    value=[\'en\', \'vi\'],\n    description=\'Languages:\',\n    rows=6,  # whole rows only — a fixed pixel height clipped the last visible item\n    layout=widgets.Layout(width=\'300px\')\n)\nextract_any_track_checkbox = widgets.Checkbox(value=False, description=\'Fall back to first text track when no language matches\', indent=False, layout=widgets.Layout(width=\'380px\'))\nextract_dir_input = widgets.Text(value=DRIVE_TV_PATH, layout=widgets.Layout(width=\'200px\'))\nbtn_browse_extract = widgets.Button(description=\'📁\', tooltip=\'Browse Drive folders\', layout=widgets.Layout(width=\'35px\'))\nbtn_extract_library = widgets.Button(description=\'📑 Extract from Library\', button_style=\'info\',\n                                     tooltip=\'Scan the folder for videos with embedded subtitle tracks and write missing .srt sidecars\',\n                                     layout=widgets.Layout(width=\'180px\'))\n\n# Secrets status UI\nsecrets_status = widgets.HTML(value="")\n\n# Confirmation UI elements\nconfirm_message = widgets.HTML(value="")\nbtn_confirm_yes = widgets.Button(description="Yes, Delete", button_style=\'danger\', layout=widgets.Layout(width=\'100px\'))\nbtn_confirm_cancel = widgets.Button(description="Cancel", button_style=\'\', layout=widgets.Layout(width=\'80px\'))\nconfirm_box = widgets.HBox([confirm_message, btn_confirm_yes, btn_confirm_cancel], \n                           layout=widgets.Layout(display=\'none\', padding=\'5px\', border=\'1px solid #f0ad4e\', margin=\'5px 0\'))\n\n# Track which action is pending confirmation\npending_action = {\'type\': None}\n\n# Directory configuration widgets with browse buttons\ndir_tv_input = widgets.Text(value=DRIVE_TV_PATH, layout=widgets.Layout(width=\'200px\'))\ndir_movie_input = widgets.Text(value=DRIVE_MOVIE_PATH, layout=widgets.Layout(width=\'200px\'))\ndir_youtube_input = widgets.Text(value=DRIVE_YOUTUBE_PATH, layout=widgets.Layout(width=\'200px\'))\ndir_downloads_input = widgets.Text(value=DRIVE_DOWNLOADS_PATH, layout=widgets.Layout(width=\'200px\'))\ndir_anime_series_input = widgets.Text(value=DRIVE_ANIME_SERIES_PATH, layout=widgets.Layout(width=\'200px\'))\ndir_anime_movies_input = widgets.Text(value=DRIVE_ANIME_MOVIES_PATH, layout=widgets.Layout(width=\'200px\'))\n\nbtn_browse_tv = widgets.Button(description=\'📁\', tooltip=\'Browse Drive folders\', layout=widgets.Layout(width=\'35px\'))\nbtn_browse_movie = widgets.Button(description=\'📁\', tooltip=\'Browse Drive folders\', layout=widgets.Layout(width=\'35px\'))\nbtn_browse_youtube = widgets.Button(description=\'📁\', tooltip=\'Browse Drive folders\', layout=widgets.Layout(width=\'35px\'))\nbtn_browse_downloads = widgets.Button(description=\'📁\', tooltip=\'Browse Drive folders\', layout=widgets.Layout(width=\'35px\'))\nbtn_browse_anime_series = widgets.Button(description=\'📁\', tooltip=\'Browse Drive folders\', layout=widgets.Layout(width=\'35px\'))\nbtn_browse_anime_movies = widgets.Button(description=\'📁\', tooltip=\'Browse Drive folders\', layout=widgets.Layout(width=\'35px\'))\n\n# Folder browser state\nbrowser_state = {\'current_path\': \'\', \'target_widget\': None, \'active\': False}\n\n# Browser UI widgets\nbrowser_path_label = widgets.HTML("")\nbrowser_folder_list = widgets.Select(options=[], description=\'\', layout=widgets.Layout(width=\'320px\', height=\'120px\'))\nbtn_browser_up = widgets.Button(description=\'⬆️ Up\', layout=widgets.Layout(width=\'60px\'))\nbtn_browser_open = widgets.Button(description=\'� Open\', layout=widgets.Layout(width=\'70px\'))\nbtn_browser_select = widgets.Button(description=\'✓ Select\', button_style=\'success\', layout=widgets.Layout(width=\'70px\'))\nbtn_browser_close = widgets.Button(description=\'✕\', button_style=\'danger\', layout=widgets.Layout(width=\'35px\'))\nnew_folder_input = widgets.Text(placeholder=\'New folder name\', layout=widgets.Layout(width=\'150px\'))\nbtn_create_folder = widgets.Button(description=\'➕ Create\', button_style=\'info\', layout=widgets.Layout(width=\'90px\'))\n\nbrowser_ui = widgets.VBox([\n    widgets.HBox([browser_path_label, btn_browser_close]),\n    browser_folder_list,\n    widgets.HBox([btn_browser_up, btn_browser_open, btn_browser_select]),\n    widgets.HBox([new_folder_input, btn_create_folder])\n], layout=widgets.Layout(display=\'none\', border=\'1px solid #888\', padding=\'5px\', margin=\'5px 0\'))\n\ndef get_folders_in_path(path):\n    """Get list of folders in the given path."""\n    folders = []\n    try:\n        full_path = os.path.join(DRIVE_BASE, path) if path else DRIVE_BASE\n        for item in os.listdir(full_path):\n            item_path = os.path.join(full_path, item)\n            if os.path.isdir(item_path) and not item.startswith(\'.\'):\n                folders.append(item)\n        folders.sort()\n    except Exception:\n        pass\n    return folders\n\ndef update_browser_ui():\n    """Update the browser UI with current path contents."""\n    path = browser_state[\'current_path\']\n    display_path = f"📁 /{path}" if path else "📁 / (Drive Root)"\n    browser_path_label.value = f"<b>{display_path}</b>"\n    folders = get_folders_in_path(path)\n    browser_folder_list.options = folders if folders else [\'(empty)\']\n    browser_folder_list.value = folders[0] if folders else None\n\ndef open_browser(target_widget):\n    """Open the folder browser for the given input widget."""\n    def handler(b):\n        # Check if Drive is mounted\n        if not os.path.exists(DRIVE_BASE):\n            dir_status.value = "<small style=\'color:orange\'>⚠️ Mount Drive first (run a download)</small>"\n            return\n        browser_state[\'target_widget\'] = target_widget\n        browser_state[\'current_path\'] = \'\'\n        browser_state[\'active\'] = True\n        browser_ui.layout.display = \'block\'\n        update_browser_ui()\n        dir_status.value = "<small>Navigate folders, then click ✓ Select</small>"\n    return handler\n\ndef on_browser_up(b):\n    """Navigate up one directory level."""\n    path = browser_state[\'current_path\']\n    if path:\n        parent = os.path.dirname(path)\n        browser_state[\'current_path\'] = parent\n        update_browser_ui()\n\ndef on_browser_open(b):\n    """Navigate into the selected folder."""\n    selected = browser_folder_list.value\n    if selected and selected != \'(empty)\':\n        path = browser_state[\'current_path\']\n        new_path = os.path.join(path, selected) if path else selected\n        browser_state[\'current_path\'] = new_path\n        update_browser_ui()\n\ndef on_browser_select(b):\n    """Select the current folder or selected subfolder."""\n    selected = browser_folder_list.value\n    path = browser_state[\'current_path\']\n    # If a folder is selected, use that; otherwise use current path\n    if selected and selected != \'(empty)\':\n        final_path = os.path.join(path, selected) if path else selected\n    else:\n        final_path = path\n    if browser_state[\'target_widget\']:\n        browser_state[\'target_widget\'].value = final_path\n    browser_ui.layout.display = \'none\'\n    browser_state[\'active\'] = False\n    dir_status.value = f"<small style=\'color:green\'>✓ Set to: {final_path}</small>"\n\ndef on_browser_close(b):\n    """Close the folder browser."""\n    browser_ui.layout.display = \'none\'\n    browser_state[\'active\'] = False\n    dir_status.value = ""\n\ndef on_create_folder(b):\n    """Create new folder in current browsing path."""\n    folder_name = new_folder_input.value.strip()\n    if not folder_name:\n        dir_status.value = "<small style=\'color:orange\'>⚠️ Enter a folder name</small>"\n        return\n    try:\n        path = browser_state[\'current_path\']\n        base = os.path.join(DRIVE_BASE, path) if path else DRIVE_BASE\n        new_path = os.path.join(base, folder_name)\n        if os.path.exists(new_path):\n            dir_status.value = f"<small style=\'color:orange\'>⚠️ \'{folder_name}\' already exists</small>"\n            return\n        os.makedirs(new_path)\n        new_folder_input.value = ""\n        update_browser_ui()\n        # Select the new folder\n        browser_folder_list.value = folder_name\n        dir_status.value = f"<small style=\'color:green\'>✅ Created \'{folder_name}\'</small>"\n    except Exception as e:\n        dir_status.value = f"<small style=\'color:red\'>❌ Error: {e}</small>"\n\nbtn_browse_tv.on_click(open_browser(dir_tv_input))\nbtn_browse_movie.on_click(open_browser(dir_movie_input))\nbtn_browse_youtube.on_click(open_browser(dir_youtube_input))\nbtn_browse_downloads.on_click(open_browser(dir_downloads_input))\nbtn_browse_anime_series.on_click(open_browser(dir_anime_series_input))\nbtn_browse_anime_movies.on_click(open_browser(dir_anime_movies_input))\nbtn_browse_extract.on_click(open_browser(extract_dir_input))\nbtn_browser_up.on_click(on_browser_up)\nbtn_browser_open.on_click(on_browser_open)\nbtn_browser_select.on_click(on_browser_select)\nbtn_browser_close.on_click(on_browser_close)\nbtn_create_folder.on_click(on_create_folder)\n\n# Organised folder config (shown when auto-organise is enabled). Labels sit on the\n# same right-aligned 130px column as the key fields above — wide enough that the\n# long ones (Anime Series/Movies) always show fully.\ndef _dir_label(text):\n    return widgets.HTML(f"<div style=\'text-align:right; padding-right:8px\'><small><b>{text}</b></small></div>", layout=widgets.Layout(width=\'130px\'))\n\norganized_dir_config = widgets.VBox([\n    widgets.HBox([_dir_label(\'TV Shows:\'), dir_tv_input, btn_browse_tv]),\n    widgets.HBox([_dir_label(\'Movies:\'), dir_movie_input, btn_browse_movie]),\n    widgets.HBox([_dir_label(\'YouTube:\'), dir_youtube_input, btn_browse_youtube]),\n    widgets.HBox([_dir_label(\'Anime Series:\'), dir_anime_series_input, btn_browse_anime_series]),\n    widgets.HBox([_dir_label(\'Anime Movies:\'), dir_anime_movies_input, btn_browse_anime_movies]),\n    browser_ui\n])\n\n# Simple downloads folder config (shown when auto-organise is disabled)\ndownloads_dir_config = widgets.VBox([\n    widgets.HBox([_dir_label(\'Downloads:\'), dir_downloads_input, btn_browse_downloads]),\n    browser_ui\n], layout=widgets.Layout(display=\'none\'))\n\ndir_config_row = widgets.VBox([\n    organized_dir_config,\n    downloads_dir_config\n])\ndir_status = widgets.HTML("")\n\nsettings_buttons = widgets.HBox([btn_clear_history, btn_clear_ytarchive, btn_clear_session, btn_settings_close])\ncookie_row = widgets.HBox([btn_upload_cookies, btn_clear_cookies, cookie_status])\napi_keys_row = widgets.HBox([token_gf, token_rd, token_tb], layout=widgets.Layout(flex_flow=\'row wrap\'))\ntmdb_row = widgets.HBox([token_tmdb, tmdb_enabled_checkbox], layout=widgets.Layout(flex_flow=\'row wrap\'))\n# Numbering toggle sits under the TMDB controls because it only affects TMDB-matched\n# shows (absolute-episode → SxxEyy conversion); its 130px label matches the key fields.\ntmdb_numbering_row = widgets.HBox([episode_numbering_toggle])\nfshare_keys_row = widgets.HBox([token_fshare_email, token_fshare_password], layout=widgets.Layout(flex_flow=\'row wrap\'))\n# 55px indent = the settings label column (130px) minus Debrid\'s own 75px label, so\n# the dropdown box lines up under the key fields. (The toggle keeps 75px to stay\n# aligned with Auto Retry in the main UI — it\'s the same widget instance in both places.)\ndebrid_row = widgets.HBox([debrid_service_toggle], layout=widgets.Layout(margin=\'0 0 0 55px\'))\nsettings_ui = widgets.VBox([\n    widgets.HTML("<b>⚙️ Settings & File Management</b>"),\n    widgets.HTML("<div style=\'margin-top:10px\'><small><b>🔑 API Keys:</b></small></div>"),\n    api_keys_row,\n    debrid_row,\n    tmdb_row,\n    tmdb_numbering_row,\n    widgets.HTML("<div style=\'margin-top:10px\'><small><b>🇻🇳 FShare Account:</b></small></div>"),\n    fshare_keys_row,\n    secrets_status,\n    widgets.HTML("<div style=\'margin-top:10px\'><small><b>📁 Download Directories (relative to Google Drive):</b></small></div>"),\n    dir_config_row,\n    dir_status,\n    widgets.HTML("<div style=\'margin-top:10px\'><small><b>🍪 YouTube Cookies (Experimental):</b></small></div>"),\n    cookie_row,\n    widgets.HTML("<div style=\'margin-top:10px\'><small><b>🚀 Performance:</b></small></div>"),\n    widgets.HBox([async_moves_checkbox]),\n    widgets.HBox([drive_api_checkbox]),\n    widgets.HBox([drive_movers_slider]),\n    widgets.HTML("<div style=\'margin-top:10px\'><small><b>⚡ Quick Download Options:</b></small></div>"),\n    widgets.HBox([quick_dl_subs_checkbox, quick_dl_subtitle_langs]),\n    widgets.HTML("<div style=\'margin-top:10px\'><small><b>📑 Embedded Subtitles:</b></small></div>"),\n    widgets.HBox([auto_extract_subs_checkbox, extract_sub_langs]),\n    widgets.HBox([extract_any_track_checkbox]),\n    widgets.HBox([_dir_label(\'Library folder:\'), extract_dir_input, btn_browse_extract, btn_extract_library]),\n    widgets.HTML("<div style=\'margin-top:10px\'><small><b>🗑️ Clear Data:</b></small></div>"),\n    settings_buttons,\n    confirm_box,\n    settings_status\n], layout=widgets.Layout(display=\'none\', padding=\'10px\', border=\'1px solid #ccc\', margin=\'5px 0\'))\n\n# --- ABOUT UI ---\nbtn_about_close = widgets.Button(description="Close", button_style=\'\', layout=widgets.Layout(width=\'80px\'))\nabout_ui = widgets.VBox([\n    widgets.HTML("""\n        <div style=\'padding: 10px;\'>\n            <h3>ℹ️ About Ultimate Downloader</h3>\n            <p><strong>Version:</strong> 6.8</p>\n            <p><strong>Author:</strong> xersbtt</p>\n            <p><strong>Repository:</strong> <a href=\'https://github.com/xersbtt/ultimate-downloader-colab\' target=\'_blank\'>github.com/xersbtt/ultimate-downloader-colab</a></p>\n            <hr>\n            <p><strong>Copyright © 2025-2026 xersbtt</strong></p>\n            <p><small>\n                Permission is hereby granted, free of charge, to any person obtaining a copy\n                of this software and associated documentation files (the "Software"), to deal\n                in the Software without restriction, including without limitation the rights\n                to use, copy, modify, merge, publish, distribute, sublicense, and/or sell\n                copies of the Software, and to permit persons to whom the Software is\n                furnished to do so, subject to the following conditions:<br><br>\n                The above copyright notice and this permission notice shall be included in all\n                copies or substantial portions of the Software.<br><br>\n                THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\n                IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\n                FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT.\n            </small></p>\n            <p><strong>Licence:</strong> MIT</p>\n            <p><small>Metadata provided by <a href=\'https://www.themoviedb.org\' target=\'_blank\'>TMDB</a>. This product uses the TMDB API but is not endorsed or certified by TMDB.</small></p>\n        </div>\n    """),\n    btn_about_close\n], layout=widgets.Layout(display=\'none\', padding=\'10px\', border=\'1px solid #ccc\', margin=\'5px 0\'))\n\ndef toggle_about(b=None):\n    """Toggle about panel visibility."""\n    if about_ui.layout.display == \'none\':\n        about_ui.layout.display = \'block\'\n        settings_ui.layout.display = \'none\'  # Close settings if open\n    else:\n        about_ui.layout.display = \'none\'\n\nbtn_about.on_click(toggle_about)\nbtn_about_close.on_click(lambda b: setattr(about_ui.layout, \'display\', \'none\'))\n\n# Helper functions to get current directory paths from widgets\ndef get_tv_path():\n    """Get TV shows path from widget or default."""\n    return dir_tv_input.value.strip() or DRIVE_TV_PATH\n\ndef get_movie_path():\n    """Get movies path from widget or default."""\n    return dir_movie_input.value.strip() or DRIVE_MOVIE_PATH\n\ndef get_youtube_path():\n    """Get YouTube path from widget or default."""\n    return dir_youtube_input.value.strip() or DRIVE_YOUTUBE_PATH\n\ndef get_downloads_path():\n    """Get Downloads path from widget or default."""\n    return dir_downloads_input.value.strip() or DRIVE_DOWNLOADS_PATH\n\ndef get_anime_series_path():\n    """Get Anime Series path from widget or default."""\n    return dir_anime_series_input.value.strip() or DRIVE_ANIME_SERIES_PATH\n\ndef get_anime_movies_path():\n    """Get Anime Movies path from widget or default."""\n    return dir_anime_movies_input.value.strip() or DRIVE_ANIME_MOVIES_PATH\n\ndef is_auto_organize_enabled():\n    """Check if auto-organization is enabled."""\n    return auto_organize_checkbox.value\n\n# --- SETTINGS PERSISTENCE ---\ndef save_dir_settings():\n    """Save directory settings to settings.json."""\n    try:\n        if not os.path.exists(UD_CONFIG_PATH):\n            os.makedirs(UD_CONFIG_PATH, exist_ok=True)\n        settings = {\n            \'tv_path\': dir_tv_input.value.strip(),\n            \'movie_path\': dir_movie_input.value.strip(),\n            \'youtube_path\': dir_youtube_input.value.strip(),\n            \'downloads_path\': dir_downloads_input.value.strip(),\n            \'anime_series_path\': dir_anime_series_input.value.strip(),\n            \'anime_movies_path\': dir_anime_movies_input.value.strip(),\n            \'auto_organize\': auto_organize_checkbox.value,\n            \'quick_dl_subs\': quick_dl_subs_checkbox.value,\n            \'tmdb_enabled\': tmdb_enabled_checkbox.value,\n            \'quick_dl_langs\': list(quick_dl_subtitle_langs.value),\n            \'auto_retry\': auto_retry_input.value.strip(),\n            \'async_moves\': async_moves_checkbox.value,\n            \'drive_api\': drive_api_checkbox.value,\n            \'drive_movers\': drive_movers_slider.value,\n            \'episode_numbering\': episode_numbering_toggle.value,\n            # FShare password is intentionally NOT saved — plaintext credentials\n            # don\'t belong on Drive. Use Colab Secrets (FSHARE_PASSWORD) instead.\n            \'fshare_email\': token_fshare_email.value.strip(),\n            \'debrid_service\': debrid_service_toggle.value,\n            \'auto_extract_subs\': auto_extract_subs_checkbox.value,\n            \'extract_langs\': list(extract_sub_langs.value),\n            \'extract_any_track\': extract_any_track_checkbox.value,\n            \'extract_dir\': extract_dir_input.value.strip()\n        }\n        with open(SETTINGS_FILE, \'w\') as f:\n            json.dump(settings, f)\n    except Exception as e:\n        # Expected before Drive is mounted; warn only if Drive is available but the write failed\n        if os.path.exists(DRIVE_BASE):\n            print(f"⚠️ Could not save settings: {e}")\n\ndef load_dir_settings():\n    """Restore settings.json into the widgets — directory paths and UI state alike.\n    A widget the user has already changed this session always keeps its value\n    (_user_touched_settings), so the post-mount reload in setup_environment restores\n    sticky settings without overriding in-progress choices. Runs at startup too,\n    where it is a no-op unless Drive happens to be mounted already."""\n    global _loading_settings\n    try:\n        if not os.path.exists(SETTINGS_FILE):\n            return  # normal before the Drive mount — setup_environment reloads later\n        with open(SETTINGS_FILE, \'r\') as f:\n            settings = json.load(f)\n        def keep(widget):  # True when the user\'s in-session change outranks the file\n            return widget in _user_touched_settings\n        _loading_settings = True  # suppress save-on-change while widgets are written\n        try:\n            if settings.get(\'tv_path\') and not keep(dir_tv_input):\n                dir_tv_input.value = settings[\'tv_path\']\n            if settings.get(\'movie_path\') and not keep(dir_movie_input):\n                dir_movie_input.value = settings[\'movie_path\']\n            if settings.get(\'youtube_path\') and not keep(dir_youtube_input):\n                dir_youtube_input.value = settings[\'youtube_path\']\n            if settings.get(\'downloads_path\') and not keep(dir_downloads_input):\n                dir_downloads_input.value = settings[\'downloads_path\']\n            if settings.get(\'anime_series_path\') and not keep(dir_anime_series_input):\n                dir_anime_series_input.value = settings[\'anime_series_path\']\n            if settings.get(\'anime_movies_path\') and not keep(dir_anime_movies_input):\n                dir_anime_movies_input.value = settings[\'anime_movies_path\']\n            if settings.get(\'fshare_email\') and not keep(token_fshare_email):\n                token_fshare_email.value = settings[\'fshare_email\']\n            # Legacy: older versions stored the password in settings.json.\n            # Still read it so existing users aren\'t locked out, but it is\n            # no longer written back (use Colab Secrets FSHARE_PASSWORD).\n            if settings.get(\'fshare_password\') and not token_fshare_password.value:\n                token_fshare_password.value = settings[\'fshare_password\']\n            if \'auto_organize\' in settings and not keep(auto_organize_checkbox):\n                auto_organize_checkbox.value = settings[\'auto_organize\']\n            if \'quick_dl_subs\' in settings and not keep(quick_dl_subs_checkbox):\n                quick_dl_subs_checkbox.value = settings[\'quick_dl_subs\']\n            if \'tmdb_enabled\' in settings and not keep(tmdb_enabled_checkbox):\n                tmdb_enabled_checkbox.value = settings[\'tmdb_enabled\']\n            if \'quick_dl_langs\' in settings and not keep(quick_dl_subtitle_langs):\n                quick_dl_subtitle_langs.value = tuple(settings[\'quick_dl_langs\'])\n            if \'auto_extract_subs\' in settings and not keep(auto_extract_subs_checkbox):\n                auto_extract_subs_checkbox.value = bool(settings[\'auto_extract_subs\'])\n            if \'extract_langs\' in settings and not keep(extract_sub_langs):\n                extract_sub_langs.value = tuple(settings[\'extract_langs\'])\n            if \'extract_any_track\' in settings and not keep(extract_any_track_checkbox):\n                extract_any_track_checkbox.value = bool(settings[\'extract_any_track\'])\n            if settings.get(\'extract_dir\') and not keep(extract_dir_input):\n                extract_dir_input.value = settings[\'extract_dir\']\n            if settings.get(\'debrid_service\') and not keep(debrid_service_toggle):\n                debrid_service_toggle.value = settings[\'debrid_service\']\n            if \'auto_retry\' in settings and not keep(auto_retry_input):\n                auto_retry_input.value = str(settings[\'auto_retry\'])\n            if \'async_moves\' in settings and not keep(async_moves_checkbox):\n                async_moves_checkbox.value = bool(settings[\'async_moves\'])\n            if \'drive_api\' in settings and not keep(drive_api_checkbox):\n                drive_api_checkbox.value = bool(settings[\'drive_api\'])\n            if \'drive_movers\' in settings and not keep(drive_movers_slider):\n                # Clamped: a settings.json from a build with a wider range must not\n                # push the slider outside its own min/max (ipywidgets raises).\n                drive_movers_slider.value = max(drive_movers_slider.min,\n                                                min(drive_movers_slider.max, int(settings[\'drive_movers\'])))\n            if settings.get(\'episode_numbering\') in (\'Season match\', \'Absolute\') and not keep(episode_numbering_toggle):\n                episode_numbering_toggle.value = settings[\'episode_numbering\']\n        finally:\n            _loading_settings = False\n        update_main_ui_visibility()\n    except Exception as e:\n        print(f"⚠️ Could not load settings (using defaults): {e}")\n\ndef update_folder_config_visibility():\n    """Show/hide appropriate folder config based on auto-organise checkbox."""\n    if auto_organize_checkbox.value:\n        organized_dir_config.layout.display = \'block\'\n        downloads_dir_config.layout.display = \'none\'\n    else:\n        organized_dir_config.layout.display = \'none\'\n        downloads_dir_config.layout.display = \'block\'\n\ndef update_main_ui_visibility():\n    """Refresh visibility of settings that depend on the auto-organise checkbox."""\n    update_folder_config_visibility()\n\ndef on_auto_organize_change(change):\n    """Handle auto-organise checkbox change."""\n    if change[\'type\'] == \'change\' and change[\'name\'] == \'value\':\n        update_main_ui_visibility()\n        if not _loading_settings:\n            _user_touched_settings.add(change[\'owner\'])\n            save_dir_settings()\n\n# Auto-save when an observed setting changes — real user edits only: programmatic\n# writes during load_dir_settings/check_and_load_secrets are ignored, and the widget\n# is remembered as user-touched so a later reload never overrides the user\'s choice.\ndef on_dir_change(change):\n    """Save settings when the user changes any observed input."""\n    if change[\'type\'] == \'change\' and change[\'name\'] == \'value\' and not _loading_settings:\n        _user_touched_settings.add(change[\'owner\'])\n        save_dir_settings()\n\nauto_organize_checkbox.observe(on_auto_organize_change, names=\'value\')\ndir_tv_input.observe(on_dir_change, names=\'value\')\ndir_movie_input.observe(on_dir_change, names=\'value\')\ndir_youtube_input.observe(on_dir_change, names=\'value\')\ndir_downloads_input.observe(on_dir_change, names=\'value\')\ndir_anime_series_input.observe(on_dir_change, names=\'value\')\ndir_anime_movies_input.observe(on_dir_change, names=\'value\')\nquick_dl_subs_checkbox.observe(on_dir_change, names=\'value\')\ntmdb_enabled_checkbox.observe(on_dir_change, names=\'value\')\nquick_dl_subtitle_langs.observe(on_dir_change, names=\'value\')\ntoken_fshare_email.observe(on_dir_change, names=\'value\')\ntoken_fshare_password.observe(on_dir_change, names=\'value\')\ndebrid_service_toggle.observe(on_dir_change, names=\'value\')\nauto_retry_input.observe(on_dir_change, names=\'value\')\nasync_moves_checkbox.observe(on_dir_change, names=\'value\')\ndrive_api_checkbox.observe(on_dir_change, names=\'value\')\ndrive_movers_slider.observe(on_dir_change, names=\'value\')\nepisode_numbering_toggle.observe(on_dir_change, names=\'value\')\nauto_extract_subs_checkbox.observe(on_dir_change, names=\'value\')\nextract_sub_langs.observe(on_dir_change, names=\'value\')\nextract_any_track_checkbox.observe(on_dir_change, names=\'value\')\nextract_dir_input.observe(on_dir_change, names=\'value\')\n\n# Try to load settings on startup (will work if Drive already mounted)\nload_dir_settings()\n\n\n# --- QUEUE MANAGEMENT UI ---\nqueue_list = widgets.SelectMultiple(options=[], description=\'Queue:\', layout=widgets.Layout(width=\'98%\', height=\'200px\'))\nbtn_queue_up = widgets.Button(description="▲ Up", button_style=\'\', layout=widgets.Layout(width=\'80px\'))\nbtn_queue_down = widgets.Button(description="▼ Down", button_style=\'\', layout=widgets.Layout(width=\'80px\'))\nbtn_queue_select_all = widgets.Button(description="Select All", button_style=\'\', layout=widgets.Layout(width=\'80px\'))\nbtn_queue_select_none = widgets.Button(description="None", button_style=\'\', layout=widgets.Layout(width=\'60px\'))\nbtn_queue_remove = widgets.Button(description="Remove", button_style=\'danger\', layout=widgets.Layout(width=\'70px\'))\nbtn_queue_sort = widgets.Button(description="Sort A-Z", button_style=\'\', icon=\'sort-alpha-asc\', tooltip=\'Sort queue alphabetically by filename\', layout=widgets.Layout(width=\'90px\'))\nqueue_sort_ascending = True  # Track current sort direction\nbtn_queue_start = widgets.Button(description="▶ Start Download", button_style=\'success\', layout=widgets.Layout(width=\'130px\'))\nbtn_queue_start_subs = widgets.Button(description="📝 Download Subtitles", button_style=\'info\', layout=widgets.Layout(width=\'170px\', display=\'none\'))  # Hidden by default\nbtn_queue_cancel = widgets.Button(description="Cancel", button_style=\'warning\', layout=widgets.Layout(width=\'70px\'))\n\n# Subtitle language selector\nsubtitle_langs = widgets.SelectMultiple(\n    options=[(\'English\', \'en\'), (\'Vietnamese\', \'vi\'), (\'Chinese\', \'zh\'), (\'Japanese\', \'ja\'), \n             (\'Korean\', \'ko\'), (\'Thai\', \'th\'), (\'Indonesian\', \'id\'), (\'Spanish\', \'es\'), \n             (\'French\', \'fr\'), (\'German\', \'de\'), (\'Portuguese\', \'pt\'), (\'Russian\', \'ru\')],\n    value=[\'en\', \'vi\'],\n    description=\'Subtitles:\',\n    layout=widgets.Layout(width=\'200px\', height=\'80px\')\n)\n\nqueue_controls = widgets.HBox([\n    widgets.HTML("", layout=widgets.Layout(width=\'115px\')),  # spacer — buttons start where the row fields do\n    btn_queue_up, btn_queue_down, btn_queue_sort, btn_queue_select_all, btn_queue_select_none,\n    btn_queue_start, btn_queue_start_subs, btn_queue_cancel, btn_queue_remove])\nqueue_options = widgets.HBox([subtitle_langs])  # Uses description for alignment like queue_list\n\n# Manual identity correction: a TMDB match (automatic canonical name/year) or a\n# forced Name/Year (its manual counterpart, which wins over a match). The TMDB group\n# shows only when TMDB matching is enabled; the whole row only when auto-organise is\n# on — identity only matters when files are renamed and routed.\ntmdb_override_input = widgets.Text(placeholder=\'TMDB URL, tv:12345 / movie:12345, or a title to search\', layout=widgets.Layout(width=\'300px\'))\nbtn_tmdb_match = widgets.Button(description=\'🔍 Match Selected\', button_style=\'info\', tooltip=\'Apply a TMDB match to the selected queue item(s)\', layout=widgets.Layout(width=\'150px\'))\nbtn_tmdb_clear = widgets.Button(description=\'✖ Clear Match\', button_style=\'\', tooltip=\'Remove the TMDB match — use filename parsing instead\', layout=widgets.Layout(width=\'120px\'))\ntmdb_group = widgets.HBox([\n    widgets.HTML("<div style=\'text-align:right; padding-right:8px\'><small><b>🎬 Fix Match:</b></small></div>", layout=widgets.Layout(width=\'115px\')),\n    tmdb_override_input, btn_tmdb_match, btn_tmdb_clear\n])\nqueue_name_input = widgets.Text(placeholder=\'Name (forces folder/file name)\', tooltip=\'Manual show/movie name for the selected queue item(s) — wins over the TMDB match\', layout=widgets.Layout(width=\'220px\'))\nqueue_year_input = widgets.Text(placeholder=\'Year\', tooltip=\'Optional year for the folder name (e.g. 2025)\', layout=widgets.Layout(width=\'70px\'))\nbtn_name_apply = widgets.Button(description=\'Set Name\', button_style=\'info\', tooltip=\'Force this name/year for the selected queue item(s)\', layout=widgets.Layout(width=\'100px\'))\nbtn_name_clear = widgets.Button(description=\'✖ Clear Name\', button_style=\'\', tooltip=\'Remove the forced name — use TMDB/filename detection again\', layout=widgets.Layout(width=\'120px\'))\nname_group = widgets.HBox([\n    widgets.HTML("<small><b>&nbsp;&nbsp;✏️ Force Name:</b></small>"),\n    queue_name_input, queue_year_input, btn_name_apply, btn_name_clear\n])\nidentity_row = widgets.HBox([tmdb_group, name_group], layout=widgets.Layout(flex_flow=\'row wrap\', display=\'none\'))\n\n# Manual routing: send the selected rows to a specific library folder — so one batch\n# can mix anime and live-action, series and movies (\'Downloads (as-is)\' skips\n# organising for those rows). Auto = use filename/TMDB detection.\nroute_dropdown = widgets.Dropdown(\n    options=[(\'Auto\', \'\'), (\'TV Series\', \'tv\'), (\'Anime Series\', \'anime_series\'),\n             (\'Movie\', \'movie\'), (\'Anime Movie\', \'anime_movie\'), (\'Downloads (as-is)\', \'downloads\')],\n    value=\'\', layout=widgets.Layout(width=\'150px\'))\nbtn_route_apply = widgets.Button(description=\'Apply Route\', button_style=\'info\', tooltip=\'Route the selected queue item(s) to this folder (Auto = clear the override)\', layout=widgets.Layout(width=\'110px\'))\nbtn_route_clear = widgets.Button(description=\'✖ Clear Route\', button_style=\'\', tooltip=\'Remove the forced route — detect the category again\', layout=widgets.Layout(width=\'120px\'))\nroute_row = widgets.HBox([\n    widgets.HTML("<div style=\'text-align:right; padding-right:8px\'><small><b>🎯 Route as:</b></small></div>", layout=widgets.Layout(width=\'115px\')),\n    route_dropdown, btn_route_apply, btn_route_clear\n])\n\n\n# Manual season override (for batches whose filenames carry no season marker —\n# without this, season 2+ files parse as season 1 and get skipped as duplicates)\nseason_override_input = widgets.Text(placeholder=\'e.g. 2\', tooltip=\'Season number to force (0 = Specials)\', layout=widgets.Layout(width=\'80px\'))\nbtn_season_apply = widgets.Button(description=\'Set Season\', button_style=\'info\', tooltip=\'Force this season for the selected queue item(s) — overrides filename parsing and TMDB mapping\', layout=widgets.Layout(width=\'110px\'))\nbtn_season_clear = widgets.Button(description=\'✖ Clear Season\', button_style=\'\', tooltip=\'Remove the forced season — use filename/TMDB detection again\', layout=widgets.Layout(width=\'130px\'))\nrenumber_start_input = widgets.Text(placeholder=\'1\', tooltip=\'Episode number to start renumbering from (default 1). A range like 7-9 names the first file S01E07-E09 (multi-episode file); later files continue from E10\', layout=widgets.Layout(width=\'60px\'))\nbtn_renumber = widgets.Button(description=\'Renumber\', button_style=\'info\', tooltip=\'Renumber the selected queue item(s) sequentially from this episode, in queue order — a video and its subtitle share one number\', layout=widgets.Layout(width=\'100px\'))\nbtn_renumber_clear = widgets.Button(description=\'✖ Clear\', button_style=\'\', tooltip=\'Remove the forced episode numbers — use filename/TMDB detection again\', layout=widgets.Layout(width=\'90px\'))\n# Manual part suffix (multi-file episodes): append -ptN sequentially to the selected\n# rows, or strip a detected/forced suffix entirely — wins over Part X / 上篇 detection\npart_override_input = widgets.Text(placeholder=\'1\', tooltip=\'Part number to start from (default 1)\', layout=widgets.Layout(width=\'55px\'))\nbtn_part_apply = widgets.Button(description=\'Set Part\', button_style=\'info\', tooltip=\'Append -ptN sequentially to the selected item(s) in queue order (e.g. 1 → -pt1, -pt2, …) — a video and its subtitle share one number\', layout=widgets.Layout(width=\'90px\'))\nbtn_part_remove = widgets.Button(description=\'✖ No Part\', button_style=\'\', tooltip=\'Strip any -ptN suffix from the selected item(s), detected or forced\', layout=widgets.Layout(width=\'95px\'))\nseason_override_row = widgets.HBox([\n    widgets.HTML("<div style=\'text-align:right; padding-right:8px\'><small><b>🗂️ Force Season:</b></small></div>", layout=widgets.Layout(width=\'115px\')),\n    season_override_input, btn_season_apply, btn_season_clear,\n    widgets.HTML("<small><b>&nbsp;&nbsp;🔢 Renumber from:</b></small>"),\n    renumber_start_input, btn_renumber, btn_renumber_clear,\n    widgets.HTML("<small><b>&nbsp;&nbsp;📎 Part:</b></small>"),\n    part_override_input, btn_part_apply, btn_part_remove\n], layout=widgets.Layout(flex_flow=\'row wrap\'))\n\n# Playlist range selector (shown only for YouTube playlists)\nplaylist_options = widgets.HBox([\n    widgets.HTML("<div style=\'text-align:right; padding-right:8px\'><small><b>🎯 Playlist Range:</b></small></div>", layout=widgets.Layout(width=\'115px\')),\n    playlist_selection\n], layout=widgets.Layout(display=\'none\'))  # Hidden by default\nqueue_ui = widgets.VBox([\n    widgets.HTML("<b>📋 Queue Preview</b> <small>(Select items to manage)</small>"),\n    queue_list,\n    identity_row,\n    route_row,\n    season_override_row,\n    playlist_options,\n    queue_options,\n    queue_controls\n], layout=widgets.Layout(display=\'none\'))  # Hidden by default\n\n\ninput_ui = widgets.VBox([\n    widgets.HTML("<h3>🚀 Ultimate Downloader v6.8</h3>"),\n    widgets.HBox([auto_organize_checkbox, debrid_service_toggle]),\n    widgets.HBox([concurrent_slider, auto_retry_input]),\n    text_area,\n    widgets.HBox([btn, btn_quick, btn_retry, btn_resume, btn_restart, btn_history, btn_settings, btn_about]),\n    stop_hint,\n    settings_ui,\n    about_ui,\n    queue_ui,\n    progress_bar,\n    _per_task_accordion,\n    status_label,\n    widgets.HTML("<hr>")\n])\n\n# Ensure UI visibility is correct after all widgets are created\nupdate_main_ui_visibility()\n\n# --- SESSION MANAGEMENT ---\n# Cumulative YouTube download counters (persist across resume)\nyt_success_cumulative = 0\nyt_fail_cumulative = 0\n\ndef load_session() -> Optional[Dict[str, Any]]:\n    """Load previous session from Drive if it exists."""\n    try:\n        if os.path.exists(SESSION_FILE):\n            with open(SESSION_FILE, \'r\') as f:\n                return json.load(f)\n    except Exception as e:\n        print(f"⚠️ Could not load session: {e}")\n    return None\n\n_last_session_save = 0.0\nSESSION_SAVE_MIN_INTERVAL = 5.0  # seconds between throttled session writes (Drive FUSE is slow)\n\ndef save_session(\n    tasks: List[DownloadTask],\n    *,  # Force all following parameters to be keyword-only\n    playlist_range: str = "",\n    yt_success: int = 0,\n    yt_fail: int = 0,\n    subtitle_langs_value: list = None,\n    throttle: bool = False\n):\n    """Persist current download state to Drive.\n\n    API tokens and passwords are intentionally NOT saved — on resume they are\n    re-read from the widgets / Colab Secrets instead of plaintext on Drive.\n\n    With throttle=True, writes are skipped if the last write was less than\n    SESSION_SAVE_MIN_INTERVAL seconds ago (per-task saves during large batches).\n    """\n    global _last_session_save\n    if throttle and time.time() - _last_session_save < SESSION_SAVE_MIN_INTERVAL:\n        return\n    try:\n        session = {\n            "version": "6.8",\n            "started_at": datetime.now().isoformat(),\n            "playlist_range": playlist_range,\n            "yt_success": yt_success,\n            "yt_fail": yt_fail,\n            "subtitle_langs": list(subtitle_langs_value) if subtitle_langs_value else [\'en\', \'vi\'],\n            "tasks": [asdict(t) for t in tasks]\n        }\n        with _session_save_lock:\n            tmp = SESSION_FILE + \'.tmp\'\n            with open(tmp, \'w\') as f:\n                json.dump(session, f, indent=2)\n            os.replace(tmp, SESSION_FILE)  # readers never see a partial file\n        _last_session_save = time.time()\n    except Exception as e:\n        print(f"⚠️ Could not save session: {e}")\n\ndef clear_session():\n    """Delete session file after successful completion."""\n    try:\n        if os.path.exists(SESSION_FILE):\n            os.remove(SESSION_FILE)\n    except Exception:\n        pass\n\ndef check_resume_available():\n    """Show/hide resume button based on session file existence."""\n    if os.path.exists(SESSION_FILE):\n        btn_resume.layout.display = \'inline-flex\'\n    else:\n        btn_resume.layout.display = \'none\'\n\n# --- DOWNLOAD HISTORY ---\n_history_lock = Lock()  # parallel workers all log downloads — serialise the read-modify-write\n\ndef _load_history() -> list:\n    """Read history.json tolerantly. Before writes were locked and atomic, concurrent\n    workers could leave trailing garbage after the JSON array (\'Extra data\' errors) —\n    recover such files by taking the first complete JSON document and ignoring the\n    tail. Raises only when nothing parseable is left."""\n    if not os.path.exists(HISTORY_FILE):\n        return []\n    with open(HISTORY_FILE, \'r\') as f:\n        text = f.read()\n    try:\n        return json.loads(text)\n    except json.JSONDecodeError:\n        history, _ = json.JSONDecoder().raw_decode(text.lstrip())\n        if not isinstance(history, list):\n            raise ValueError("history is not a list")\n        return history\n\ndef log_download(filename: str, source: str, size_mb: float, destination: str, status: str = "success"):\n    """Append download to persistent history log for debugging. Locked and written\n    via temp-file + atomic replace: unsynchronised plain writes from parallel\n    workers are what used to corrupt the file with trailing garbage."""\n    try:\n        with _history_lock:\n            try:\n                history = _load_history()\n            except Exception:\n                history = []  # unrecoverable file — restart the log rather than stop logging\n            entry = {\n                "timestamp": datetime.now().isoformat(),\n                "filename": filename,\n                "source": source,\n                "size_mb": round(size_mb, 2),\n                "destination": destination,\n                "status": status\n            }\n            history.insert(0, entry)  # Newest first\n            history = history[:500]   # Keep last 500 entries\n            tmp = HISTORY_FILE + \'.tmp\'\n            with open(tmp, \'w\') as f:\n                json.dump(history, f, indent=2)\n            os.replace(tmp, HISTORY_FILE)  # readers never see a partial file\n    except Exception:\n        pass  # Silent fail for logging\n\ndef view_history(b=None):\n    """Open history file location in output."""\n    if os.path.exists(HISTORY_FILE):\n        print(f"📜 History file: {HISTORY_FILE}")\n        print(f"   (Open in Google Drive to view)")\n        try:\n            history = _load_history()\n            print(f"\\n📊 Last 10 downloads (times in UTC):")\n            for i, entry in enumerate(history[:10], 1):\n                ts = entry.get(\'timestamp\', \'\')[:16].replace(\'T\', \' \')\n                fn = entry.get(\'filename\', \'Unknown\')[:40]\n                src = entry.get(\'source\', \'?\')\n                size = entry.get(\'size_mb\', 0)\n                print(f"   {i}. [{ts}] {fn} ({src}, {size:.1f}MB)")\n        except Exception as e:\n            print(f"   ⚠️ Could not read history ({str(e)[:60]}) — clear it in ⚙️ Settings to start fresh")\n    else:\n        print("📜 No download history yet.")\n\n\ndef check_secrets_status():\n    """Check Colab secrets status and update display."""\n    gf_status = "✅" if token_gf.value.strip() else "❌"\n    rd_status = "✅" if token_rd.value.strip() else "❌"\n    tb_status = "✅" if token_tb.value.strip() else "❌"\n    tmdb_status = "✅" if token_tmdb.value.strip() else "❌"\n    fs_status = "✅" if token_fshare_email.value.strip() and token_fshare_password.value.strip() else "❌"\n    debrid = debrid_service_toggle.value\n    debrid_label = f"<b>[{debrid}]</b>"\n    secrets_status.value = f"<span style=\'font-size:12px\'>{gf_status} Gofile &nbsp; {rd_status} Real-Debrid &nbsp; {tb_status} TorBox &nbsp; {tmdb_status} TMDB &nbsp; {fs_status} FShare &nbsp; | Debrid: {debrid_label}</span>"\n\ndef check_cookie_status():\n    """Check if cookies.txt exists and update status display."""\n    if os.path.exists(COOKIE_PATH):\n        cookie_status.value = "<span style=\'color:green\'>✅ Loaded</span>"\n    else:\n        cookie_status.value = "<span style=\'color:gray\'>— None</span>"\n\ndef upload_cookies(b=None):\n    """Upload cookies.txt file for YouTube authentication (experimental)."""\n    try:\n        from google.colab import files\n        from IPython.display import clear_output\n        settings_status.value = "<span style=\'color:blue\'>📤 Select cookies.txt file...</span>"\n        uploaded = files.upload()\n        clear_output(wait=True)\n        display(input_ui)\n        if uploaded:\n            for filename in uploaded.keys():\n                shutil.move(filename, COOKIE_PATH)\n                settings_status.value = f"<span style=\'color:green\'>✅ Cookies uploaded from {filename}</span>"\n                break\n        else:\n            settings_status.value = "<span style=\'color:gray\'>Upload cancelled</span>"\n        check_cookie_status()\n    except ImportError:\n        settings_status.value = "<span style=\'color:red\'>❌ Cookie upload only works in Google Colab</span>"\n    except Exception as e:\n        settings_status.value = f"<span style=\'color:red\'>❌ Upload failed: {str(e)[:40]}</span>"\n\ndef clear_cookies(b=None):\n    """Delete cookies.txt file to fix authentication/format errors."""\n    try:\n        if os.path.exists(COOKIE_PATH):\n            os.remove(COOKIE_PATH)\n            settings_status.value = "<span style=\'color:green\'>✅ Cookies cleared! Downloads will use anonymous access.</span>"\n        else:\n            settings_status.value = "<span style=\'color:gray\'>ℹ️ No cookies file to clear.</span>"\n        check_cookie_status()\n    except Exception as e:\n        settings_status.value = f"<span style=\'color:red\'>❌ Error: {str(e)[:50]}</span>"\n\ndef toggle_settings(b=None):\n    """Toggle settings panel visibility."""\n    if settings_ui.layout.display == \'none\':\n        settings_ui.layout.display = \'block\'\n        settings_status.value = ""\n        confirm_box.layout.display = \'none\'\n        pending_action[\'type\'] = None\n        # Refresh status indicators\n        check_secrets_status()\n        check_cookie_status()\n    else:\n        settings_ui.layout.display = \'none\'\n\ndef close_settings(b=None):\n    """Close settings panel."""\n    settings_ui.layout.display = \'none\'\n    confirm_box.layout.display = \'none\'\n    pending_action[\'type\'] = None\n\ndef restart_runtime(b=None):\n    """Restart Colab runtime for fresh session."""\n    try:\n        from google.colab import runtime\n        print("🔄 Restarting runtime... Use \'Resume Previous\' after restart.")\n        runtime.unassign()\n    except ImportError:\n        print("❌ Runtime restart only available in Google Colab")\n    except Exception as e:\n        print(f"❌ Could not restart: {e}")\n\ndef show_confirmation(action_type: str, message: str):\n    """Show confirmation dialog for a pending action."""\n    pending_action[\'type\'] = action_type\n    confirm_message.value = f"<span style=\'color:#856404\'>⚠️ {message}</span>"\n    confirm_box.layout.display = \'flex\'\n    settings_status.value = ""\n\ndef cancel_confirmation(b=None):\n    """Cancel the pending confirmation."""\n    pending_action[\'type\'] = None\n    confirm_box.layout.display = \'none\'\n    settings_status.value = "<span style=\'color:gray\'>Cancelled.</span>"\n\ndef confirm_action(b=None):\n    """Execute the confirmed action."""\n    action = pending_action[\'type\']\n    pending_action[\'type\'] = None\n    confirm_box.layout.display = \'none\'\n    \n    if action == \'history\':\n        _do_clear_history()\n    elif action == \'ytarchive\':\n        _do_clear_ytarchive()\n    elif action == \'session\':\n        _do_clear_session()\n\ndef request_clear_history(b=None):\n    """Request confirmation to clear download history."""\n    show_confirmation(\'history\', "Delete download history? This action cannot be undone.")\n\ndef request_clear_ytarchive(b=None):\n    """Request confirmation to clear YT archive."""\n    show_confirmation(\'ytarchive\', "Delete YT archive? This allows re-downloading previously downloaded videos.")\n\ndef request_clear_session(b=None):\n    """Request confirmation to clear session."""\n    show_confirmation(\'session\', "Delete session file? This removes resume capability.")\n\ndef _do_clear_history():\n    """Actually clear the download history file."""\n    try:\n        if os.path.exists(HISTORY_FILE):\n            os.remove(HISTORY_FILE)\n            settings_status.value = "<span style=\'color:green\'>✅ Download history cleared!</span>"\n        else:\n            settings_status.value = "<span style=\'color:gray\'>ℹ️ No history file to clear.</span>"\n    except Exception as e:\n        settings_status.value = f"<span style=\'color:red\'>❌ Error: {str(e)[:50]}</span>"\n\ndef _do_clear_ytarchive():\n    """Actually clear the yt-dlp download archive."""\n    archive_path = f"{UD_CONFIG_PATH}yt_history.txt"\n    try:\n        if os.path.exists(archive_path):\n            os.remove(archive_path)\n            settings_status.value = "<span style=\'color:green\'>✅ YT archive cleared! You can now re-download previous videos.</span>"\n        else:\n            settings_status.value = "<span style=\'color:gray\'>ℹ️ No YT archive file to clear.</span>"\n    except Exception as e:\n        settings_status.value = f"<span style=\'color:red\'>❌ Error: {str(e)[:50]}</span>"\n\ndef _do_clear_session():\n    """Actually clear the session file."""\n    try:\n        if os.path.exists(SESSION_FILE):\n            os.remove(SESSION_FILE)\n            btn_resume.layout.display = \'none\'\n            btn_retry.layout.display = \'none\'\n            settings_status.value = "<span style=\'color:green\'>✅ Session cleared!</span>"\n        else:\n            settings_status.value = "<span style=\'color:gray\'>ℹ️ No session file to clear.</span>"\n    except Exception as e:\n        settings_status.value = f"<span style=\'color:red\'>❌ Error: {str(e)[:50]}</span>"\n\n\n\n# --- QUEUE MANAGEMENT ---\npending_queue: List[DownloadTask] = []  # Global queue state\n\ndef _queue_dest_preview(task) -> Optional[str]:\n    """Resolved destination for a queue row — the full Drive-relative path the file\n    will take, library folder included (e.g. \'TV Shows/Detective Conan (1996)/\n    Season 01/Detective Conan - S01E1206.mkv\'). Uses the same determine_destination_path\n    the download itself uses (dry_run), mirroring handle_file_processing\'s naming steps:\n    the queue size suffix is stripped and a subtitle\'s language tag is split off before\n    routing, then re-attached as a Plex ISO code. Returns None when no preview is\n    possible (no filename yet, or an archive whose contents are only known after\n    extraction) — the caller falls back to the plain row display."""\n    if not task.filename:\n        return None\n    clean = _strip_size_suffix(task.filename)\n    if os.path.splitext(clean)[1].lower() in (\'.rar\', \'.zip\', \'.7z\'):\n        return None\n    try:\n        processing_name, lang = _split_subtitle_lang(clean)\n        dest, _ = determine_destination_path(processing_name, task.source or "generic", dry_run=True)\n        if lang:  # same re-attachment as handle_file_processing\n            base, sub_ext = os.path.splitext(dest)\n            dest = f"{base}.{_normalize_sub_lang(lang)}{sub_ext}"\n        # Full Drive-relative path: with per-row routing the library folder is\n        # the part being controlled, so it belongs in the preview\n        return os.path.relpath(dest, DRIVE_BASE).replace(os.sep, \'/\')\n    except Exception:\n        return None  # a preview glitch must never break the queue display\n\n\ndef update_queue_display(preserve_selection: bool = False):\n    """Update the queue list widget with current pending_queue. Each row carries a\n    live destination preview (where the file will go and its final name), recomputed\n    from the current settings, TMDB matches, and forced seasons. preserve_selection\n    keeps the current row selection across the refresh (used by the live settings\n    observers); the default reselects everything, as at queue creation."""\n    selected = set(_selected_queue_indices()) if preserve_selection else None\n    options = []\n\n    for i, task in enumerate(pending_queue):\n        source_icon = {"gofile": "📁", "pixeldrain": "💾", "rd": "⚡", "tb": "📦", "tb_host": "📦", "direct": "🔗", \n                       "youtube": "▶️", "mega": "☁️", "mediafire": "🔥", "1fichier": "📦",\n                       "magnet": "🧲", "magnet_file": "🧲", "tb_magnet_file": "🧲", "archive": "📚",\n                       "fshare": "🇻🇳", "okru": "🟠"}.get(task.link_type, "📄")\n        name = task.filename[:50] if task.filename else task.url[:50]\n        sov = getattr(task, \'season_override\', None)\n        if sov is not None:\n            name += f"  [S{sov:02d}]"  # forced season marker\n        eov = getattr(task, \'episode_override\', None)\n        if eov is not None:\n            eev = getattr(task, \'episode_end_override\', None)\n            # forced episode marker; a range renumber shows its span (E07-E09)\n            name += f"  [E{eov:02d}-E{eev:02d}]" if eev else f"  [E{eov:02d}]"\n        pov = getattr(task, \'part_override\', None)\n        if pov is not None:\n            name += f"  [pt{pov}]" if pov else "  [no pt]"  # forced part suffix marker\n        ov = getattr(task, \'tmdb_override\', None)\n        m = get_tmdb_match(task.filename) if task.filename else None\n        dest = _queue_dest_preview(task)\n        if dest is not None:\n            # ✎ = manual TMDB correction, ✖ = match cleared — the path shows the outcome\n            mark = " ✖" if ov == TMDB_CLEARED else (" ✎" if (m and ov) else "")\n            options.append(f"{i+1}. {source_icon} {name}{mark} → {dest}")\n        elif ov == TMDB_CLEARED:\n            options.append(f"{i+1}. {source_icon} {name}  ✖ no TMDB")\n        elif m:\n            edited = " ✎" if ov else ""  # pencil marks a manual correction\n            tag = f" → {m[\'name\']}" + (f" ({m[\'year\']})" if m[\'year\'] else "") + edited\n            options.append(f"{i+1}. {source_icon} {name}{tag}")\n        else:\n            options.append(f"{i+1}. {source_icon} {name}")\n\n    queue_list.options = options\n    if selected is not None:\n        # Keep the user\'s selection across a live settings refresh (matched by row index)\n        queue_list.value = tuple(opt for i, opt in enumerate(options) if i in selected)\n    else:\n        queue_list.value = tuple(options)  # Select all by default\n\ndef _on_dest_setting_change(change):\n    """Live-refresh the queue\'s destination previews when a setting that affects\n    routing changes (Auto-organise, Force Name/Year, Type, Category, Numbering,\n    TMDB on/off). When TMDB matching becomes newly applicable (just enabled, or\n    Force Name cleared) with an empty match cache, the batch match runs first —\n    Start would do the same, so the preview keeps showing what will actually happen."""\n    if change.get(\'name\') != \'value\' or not pending_queue:\n        return\n    try:\n        if tmdb_is_enabled() and not _tmdb_match_cache:\n            analyze_batch_metadata([t.filename for t in pending_queue if t.filename])\n            _apply_tmdb_overrides(pending_queue)\n        # Keep the queue\'s conditional rows in sync with the toggles they depend on\n        tmdb_group.layout.display = \'flex\' if tmdb_is_enabled() else \'none\'\n        _org = \'flex\' if is_auto_organize_enabled() else \'none\'\n        identity_row.layout.display = _org\n        route_row.layout.display = _org\n        season_override_row.layout.display = _org\n        update_queue_display(preserve_selection=True)\n    except Exception:\n        pass  # never let a preview refresh break a settings change\n\nfor _w in (auto_organize_checkbox, episode_numbering_toggle, tmdb_enabled_checkbox):\n    _w.observe(_on_dest_setting_change, names=\'value\')\n\n\ndef show_queue_preview(tasks: List[DownloadTask], mode: str):\n    """Show queue UI with resolved tasks."""\n    global pending_queue\n    pending_queue = tasks.copy()\n    \n    # Run batch episode analysis to improve episode detection accuracy\n    # This analyzes all filenames together to find the varying number (episode) vs constants\n    filenames = [t.filename for t in tasks if t.filename]\n    if len(filenames) >= 2:\n        batch_results = analyze_batch_episodes(filenames)\n        if batch_results:\n            print(f"   🎯 Batch analysis detected episode numbers in {len(batch_results)} files")\n\n    # TMDB metadata matching (canonical names, years, season mapping)\n    if tmdb_is_enabled():\n        tmdb_matched = analyze_batch_metadata(filenames)\n        _apply_tmdb_overrides(pending_queue)  # reapply any prior manual corrections\n        if tmdb_matched:\n            print(f"   🎬 TMDB matched {tmdb_matched} of {len(filenames)} file(s)")\n    _apply_queue_overrides(pending_queue)  # independent of TMDB — works either way\n\n    update_queue_display()\n\n    # Hide subtitle and playlist options initially to prevent flash of old content\n    queue_options.layout.display = \'none\'\n    playlist_options.layout.display = \'none\'\n    # Identity / Route / Force Season rows: only meaningful when auto-organise will\n    # rename and route files; the TMDB group additionally needs TMDB matching active\n    tmdb_group.layout.display = \'flex\' if tmdb_is_enabled() else \'none\'\n    _org = \'flex\' if is_auto_organize_enabled() else \'none\'\n    identity_row.layout.display = _org\n    route_row.layout.display = _org\n    season_override_row.layout.display = _org\n    queue_ui.layout.display = \'block\'\n    \n    # Check for YouTube/streaming links\n    youtube_tasks = [t for t in tasks if t.link_type == \'youtube\']\n    has_streaming = len(youtube_tasks) > 0\n    \n    # Default full subtitle options\n    DEFAULT_SUBS = [(\'English\', \'en\'), (\'Vietnamese\', \'vi\'), (\'Chinese\', \'zh\'), (\'Japanese\', \'ja\'), \n                    (\'Korean\', \'ko\'), (\'Thai\', \'th\'), (\'Indonesian\', \'id\'), (\'Spanish\', \'es\'), \n                    (\'French\', \'fr\'), (\'German\', \'de\'), (\'Portuguese\', \'pt\'), (\'Russian\', \'ru\')]\n    \n    if has_streaming:\n        # Check if any URL is a playlist OR there are multiple YouTube videos\n        # After playlist expansion, check original_url for playlist detection\n        has_playlist = any(\n            (\'list=\' in (t.original_url or t.url) or \'/playlist\' in (t.original_url or t.url))\n            for t in youtube_tasks\n        )\n        has_multiple_videos = len(youtube_tasks) > 1\n        \n        if has_playlist or has_multiple_videos:\n            # For playlists or multiple videos, show full selector (can\'t efficiently check all)\n            subtitle_langs.options = DEFAULT_SUBS\n            subtitle_langs.value = [\'en\', \'vi\']\n            queue_options.layout.display = \'block\'\n            btn_queue_start_subs.layout.display = \'inline-block\'\n            if has_playlist:\n                # Show playlist range selector for actual playlists (non-expanded)\n                # Only show if there are unexpanded playlist URLs (original_url == url)\n                has_unexpanded = any(\n                    (t.original_url is None or t.original_url == t.url) and\n                    (\'list=\' in t.url or \'/playlist\' in t.url)\n                    for t in youtube_tasks\n                )\n                if has_unexpanded:\n                    playlist_options.layout.display = \'flex\'\n                    print("📋 Playlist detected - use Playlist Range to select specific videos (e.g. 1,3,5-10)")\n            else:\n                print("📋 Multiple videos detected - full subtitle languages available")\n        else:\n            # For a single video only, fetch actual available subtitles\n            print("🔍 Checking available subtitles...")\n            available_subs = get_youtube_subtitles(youtube_tasks[0].url) if youtube_tasks else {}\n            \n            if available_subs:\n                # Update subtitle selector with available languages\n                subtitle_langs.options = [(name, code) for code, name in sorted(available_subs.items(), key=lambda x: x[1])]\n                # Pre-select English and Vietnamese if available\n                preselect = [code for code in [\'en\', \'vi\', \'en-US\', \'en-GB\'] if code in available_subs]\n                subtitle_langs.value = preselect[:2] if preselect else []\n                queue_options.layout.display = \'block\'\n                btn_queue_start_subs.layout.display = \'inline-block\'\n                print(f"   ✓ Found {len(available_subs)} subtitle languages available")\n            else:\n                # No subtitles found - hide selector and button\n                queue_options.layout.display = \'none\'\n                btn_queue_start_subs.layout.display = \'none\'\n                print("   ℹ️ No manual subtitles available for this video")\n    else:\n        queue_options.layout.display = \'none\'\n        btn_queue_start_subs.layout.display = \'none\'\n    \n    btn.disabled = True\n    btn_quick.disabled = True\n    print(f"📋 Queue loaded with {len(tasks)} items. Review and click \'Start Download\' or \'Download Subtitles\' to begin.")\n\ndef hide_queue():\n    """Hide queue UI and reset state."""\n    global pending_queue\n    pending_queue = []\n    queue_ui.layout.display = \'none\'\n    queue_list.options = []\n    playlist_options.layout.display = \'none\'\n    identity_row.layout.display = \'none\'\n    route_row.layout.display = \'none\'\n    tmdb_override_input.value = \'\'\n    queue_name_input.value = \'\'\n    queue_year_input.value = \'\'\n    route_dropdown.value = \'\'\n    season_override_input.value = \'\'\n    renumber_start_input.value = \'\'\n    part_override_input.value = \'\'\n    btn_queue_start_subs.layout.display = \'none\'\n    btn.disabled = False\n    btn_quick.disabled = False\n\ndef queue_move_up(b=None):\n    """Move selected items up in the queue."""\n    global pending_queue\n    selected = list(queue_list.value)\n    if not selected:\n        return\n    indices = [int(s.split(\'.\')[0]) - 1 for s in selected]\n    indices.sort()\n    for idx in indices:\n        if idx > 0 and idx - 1 not in indices:\n            pending_queue[idx], pending_queue[idx-1] = pending_queue[idx-1], pending_queue[idx]\n    update_queue_display()\n    # Re-select moved items\n    new_selected = [queue_list.options[max(0, i-1)] for i in indices]\n    queue_list.value = tuple(new_selected)\n\ndef queue_move_down(b=None):\n    """Move selected items down in the queue."""\n    global pending_queue\n    selected = list(queue_list.value)\n    if not selected:\n        return\n    indices = [int(s.split(\'.\')[0]) - 1 for s in selected]\n    indices.sort(reverse=True)\n    for idx in indices:\n        if idx < len(pending_queue) - 1 and idx + 1 not in indices:\n            pending_queue[idx], pending_queue[idx+1] = pending_queue[idx+1], pending_queue[idx]\n    update_queue_display()\n    # Re-select moved items\n    new_selected = [queue_list.options[min(len(pending_queue)-1, i+1)] for i in indices]\n    queue_list.value = tuple(new_selected)\n\ndef queue_select_all(b=None):\n    """Select all items in queue."""\n    queue_list.value = tuple(queue_list.options)\n\ndef queue_select_none(b=None):\n    """Deselect all items in queue."""\n    queue_list.value = ()\n\ndef queue_remove_selected(b=None):\n    """Remove selected items from queue."""\n    global pending_queue\n    selected = list(queue_list.value)\n    if not selected:\n        return\n    indices_to_remove = {int(s.split(\'.\')[0]) - 1 for s in selected}\n    pending_queue = [t for i, t in enumerate(pending_queue) if i not in indices_to_remove]\n    update_queue_display()\n    if not pending_queue:\n        hide_queue()\n        print("📋 Queue is empty.")\n\ndef queue_sort_alpha(b=None):\n    """Sort queue items alphabetically by filename, toggling between A-Z and Z-A."""\n    global pending_queue, queue_sort_ascending\n    # Preserve selection: capture selected task IDs before sorting\n    selected_ids = set()\n    selected = list(queue_list.value)\n    if selected:\n        for s in selected:\n            idx = int(s.split(\'.\')[0]) - 1\n            if 0 <= idx < len(pending_queue):\n                selected_ids.add(pending_queue[idx].id)\n    pending_queue.sort(key=lambda t: (t.filename or t.url).lower(), reverse=not queue_sort_ascending)\n    update_queue_display()\n    # Restore selection by matching task IDs to new positions\n    if selected_ids:\n        new_selected = [opt for i, opt in enumerate(queue_list.options)\n                        if i < len(pending_queue) and pending_queue[i].id in selected_ids]\n        queue_list.value = tuple(new_selected)\n    # Toggle direction for next click\n    queue_sort_ascending = not queue_sort_ascending\n    if queue_sort_ascending:\n        btn_queue_sort.description = "Sort A-Z"\n        btn_queue_sort.icon = "sort-alpha-asc"\n    else:\n        btn_queue_sort.description = "Sort Z-A"\n        btn_queue_sort.icon = "sort-alpha-desc"\n\ndef queue_cancel(b=None):\n    """Cancel queue and return to link input."""\n    hide_queue()\n    print("❌ Queue cancelled.")\n\ndef start_from_queue(b=None, mode="video"):\n    """Start downloading selected items from queue."""\n    selected = list(queue_list.value)\n    if not selected:\n        print("⚠️ No items selected! Select items to download.")\n        return\n    \n    # Get selected indices\n    selected_indices = {int(s.split(\'.\')[0]) - 1 for s in selected}\n    selected_tasks = [t for i, t in enumerate(pending_queue) if i in selected_indices]\n    \n    if not selected_tasks:\n        print("⚠️ No valid items selected!")\n        return\n    \n    # Hide queue and start download\n    hide_queue()\n    if mode == "subs_only":\n        print(f"📝 Starting subtitle download of {len(selected_tasks)} selected items...")\n    else:\n        print(f"🚀 Starting download of {len(selected_tasks)} selected items...")\n    \n    # Process the selected tasks with the specified mode\n    execute_selected_tasks(selected_tasks, mode)\n\n# --- HELPER FUNCTIONS ---\ndef _clear_per_task_bars():\n    """Close and remove all per-download progress bars."""\n    for bar in _per_task_bars.values():\n        bar.close()\n    _per_task_bars.clear()\n    _per_task_done_at.clear()\n    _per_task_box.children = []\n    _per_task_accordion.layout.display = \'none\'\n\ndef reset_progress():\n    """Resets UI to idle state"""\n    progress_bar.value = 0\n    progress_bar.description = "Idle"\n    progress_bar.bar_style = \'info\'\n    status_label.value = ""\n    _clear_per_task_bars()\n\ndef update_status(message: str):\n    """Thread-safe status update."""\n    with progress_lock:\n        status_label.value = f"<small>{message}</small>"\n\ndef normalize_playlist_range(range_str):\n    """Normalize playlist range string for yt-dlp\'s playlist_items option."""\n    if not range_str or not range_str.strip():\n        return None\n    return range_str.replace(\' \', \'\')\n\ndef sanitize_filename(name: str) -> str:\n    name = unquote(name)\n    name = re.sub(r\'[<>:"/\\\\|?*]\', \'_\', name) \n    name = re.sub(r\'[\\s_]+\', \' \', name).strip()\n    return name\n\ndef clean_show_name(name: str) -> str:\n    """Strip leading bracketed content that looks like technical tags (not show names).\n    Technical tags: resolutions, codecs, release groups (usually single words or known patterns)\n    Show names: usually contain spaces (multiple words)\n    """\n    # Keep stripping technical-looking brackets from the start\n    while True:\n        # Check if next bracket is a technical tag (no spaces inside, or matches known patterns)\n        match = re.match(r\'^\\s*\\[([^\\]]*)\\]\', name)\n        if not match:\n            break\n        content = match.group(1)\n        # If bracket contains spaces, it\'s likely a show name - stop stripping\n        if \' \' in content and not re.match(r\'(?i)^(WEB-?DL|Dolby\\s*Vision|10\\s*bit)$\', content):\n            break\n        # Strip this bracket\n        name = name[match.end():]\n    \n    # Also strip leading parenthetical technical tags like (Hi10), (480p), (DragonFox)\n    while True:\n        match = re.match(r\'^\\s*\\(([^)]*)\\)\', name)\n        if not match:\n            break\n        content = match.group(1)\n        # If parentheses contain spaces, might be show name - stop stripping\n        if \' \' in content:\n            break\n        # Known technical patterns to strip\n        if re.match(r\'(?i)^(Hi10|10bit|x264|x265|HEVC|AVC|\\d{3,4}p|WEB-?DL|BluRay|[A-Za-z0-9_-]{1,15})$\', content):\n            name = name[match.end():]\n            continue\n        # Unknown single word - stop to be safe\n        break\n    \n    # Remove common YouTube prefixes (VIETSUB, ENGSUB, THUYẾT MINH, etc.)\n    name = re.sub(r\'(?i)^\\s*(?:VIETSUB|VietSub|ENGSUB|EngSub|ENG\\s*SUB|VIET\\s*SUB|THUYẾT\\s*MINH|RAW|FULL|HD)\\s*[|｜:：\\-–—]\\s*\', \'\', name)\n    # Remove technical tags in brackets or standalone\n    name = re.sub(r\'(?i)(?:\\[?\\s*(?:ENG\\s*SUB|ENGSUB|FULL|WEB-?DL|WEBRip|BluRay|HDR|10bit|Atmos|DV|Vision|DDP\\d\\.\\d|x265|HEVC|x264|H\\.\\d{3})\\s*\\]?)\', \'\', name)\n    name = re.sub(r\'(?i)\\b(2160p|1080p|720p|480p|4k|8k)\\b\', \'\', name)\n    name = re.sub(r\'[\\[\\]\\(\\)《》「」【】]\', \' \', name)\n    # Remove trailing pipe/separator sections (e.g., "Show Name | Episode Info |" -> "Show Name")\n    name = re.sub(r\'\\s*[|｜]\\s*$\', \'\', name)\n    name = re.sub(r\'[|｜._-]\', \' \', name)\n    name = re.sub(r\'(?i)\\s+\\b(END|FINALE|FINAL)\\b$\', \'\', name)\n    clean = re.sub(r\'\\s+\', \' \', name).strip()\n    return clean if clean else "Unknown Show"\n\ndef is_safe_path(base_dir: str, filename: str) -> bool:\n    """Prevent directory traversal attacks with strict prefix checking"""\n    try:\n        target_path = os.path.realpath(os.path.join(base_dir, filename))\n        base_path = os.path.realpath(base_dir)\n        return target_path.startswith(base_path + os.sep) or target_path == base_path\n    except Exception:\n        return False\n\n# --- BATCH EPISODE DETECTION ---\n# Global cache for batch analysis results\n_batch_episode_cache: Dict[str, int] = {}  # stripped filename -> detected episode number\n\ndef _strip_size_suffix(filename: str) -> str:\n    """Remove trailing file-size annotations like \' (126.7 MB)\' added for queue display.\n    Cache keys use the stripped form so lookups work both at queue time (with suffix)\n    and at download time (without)."""\n    return re.sub(r\'\\s*\\([^)]*[KMG]i?B\\s*\\)\\s*$\', \'\', filename)\n\ndef _split_subtitle_lang(filename: str) -> Tuple[str, str]:\n    """Split a subtitle\'s trailing 2-3 letter language tag from its name:\n    \'Show.EP01.Eng.srt\' -> (\'Show.EP01.srt\', \'Eng\'). Applies to every subtitle format\n    (KEEP_EXTENSIONS: .srt/.ass/.sub/.vtt). Non-tagged subs and non-subs return\n    (filename, \'\'). Single source of truth for the subtitle language convention, shared\n    by handle_file_processing (which names the saved file) and _match_cache_key (so a\n    subtitle keys the same show as its video)."""\n    _, ext = os.path.splitext(filename)\n    if ext.lower() in KEEP_EXTENSIONS:\n        parts = filename.split(\'.\')\n        if len(parts) >= 3 and len(parts[-2]) in (2, 3):\n            return ".".join(parts[:-2]) + ext, parts[-2]\n    return filename, \'\'\n\n# Release/fansub subtitle language tags → Plex-friendly ISO codes. 2-letter ISO 639-1,\n# with Traditional/Simplified Chinese kept distinct (zh-Hant/zh-Hans). Keys are\n# casefolded 2-3 char tags (all _split_subtitle_lang can yield); unmapped tags pass\n# through unchanged so an unexpected tag never regresses to a worse name.\n_SUB_LANG_MAP = {\n    \'en\': \'en\', \'eng\': \'en\',\n    \'vi\': \'vi\', \'vie\': \'vi\',\n    \'ja\': \'ja\', \'jp\': \'ja\', \'jpn\': \'ja\',\n    \'ko\': \'ko\', \'kor\': \'ko\',\n    \'th\': \'th\', \'tha\': \'th\',\n    \'id\': \'id\', \'ind\': \'id\',\n    \'es\': \'es\', \'spa\': \'es\',\n    \'fr\': \'fr\', \'fra\': \'fr\', \'fre\': \'fr\',\n    \'de\': \'de\', \'ger\': \'de\', \'deu\': \'de\',\n    \'pt\': \'pt\', \'por\': \'pt\',\n    \'ru\': \'ru\', \'rus\': \'ru\',\n    \'cht\': \'zh-Hant\', \'tc\': \'zh-Hant\',   # Traditional Chinese\n    \'chs\': \'zh-Hans\', \'sc\': \'zh-Hans\',   # Simplified Chinese\n    \'zh\': \'zh\', \'chi\': \'zh\', \'zho\': \'zh\',  # unspecified Chinese\n}\n\ndef _normalize_sub_lang(tag: str) -> str:\n    """Map a source subtitle language tag (\'Eng\', \'Cht\') to a Plex-friendly ISO code\n    (\'en\', \'zh-Hant\'). Unrecognised tags are returned unchanged."""\n    return _SUB_LANG_MAP.get(tag.casefold(), tag)\n\ndef _match_cache_key(filename: str) -> str:\n    """Canonical key for the TMDB-match and season-override caches. Strips the queue\n    size suffix AND any subtitle language tag, so \'Show.EP01.Eng.srt\' resolves to the\n    same entry as the language-stripped name determine_destination_path looks up for\n    subtitles (\'Show.EP01.srt\') — otherwise subtitles miss the cache and fall back to\n    messy filename parsing while their video gets the clean TMDB name."""\n    return _split_subtitle_lang(_strip_size_suffix(sanitize_filename(filename)))[0]\n\ndef analyze_batch_episodes(filenames: List[str]) -> Dict[str, int]:\n    """\n    Analyze a batch of filenames to detect episode numbers by finding varying patterns.\n    \n    Strategy: Find number patterns that vary sequentially across files (likely episodes)\n    vs patterns that are constant (resolutions, codecs, etc).\n    \n    Returns dict mapping filename -> detected episode number (or None if not found).\n    """\n    global _batch_episode_cache\n    _batch_episode_cache.clear()\n    \n    if len(filenames) < 2:\n        return {}  # Need at least 2 files for batch analysis\n    \n    # Extract all bracketed numbers from each file with their positions\n    def extract_bracket_numbers(filename: str) -> List[Tuple[int, int, str]]:\n        """Returns list of (position_index, number_value, matched_text) for bracketed numbers."""\n        results = []\n        for i, m in enumerate(re.finditer(r\'\\[(\\d{1,4})\\]\', filename)):\n            num = int(m.group(1))\n            results.append((i, num, m.group(0)))\n        return results\n    \n    # Also extract dash-separated numbers like "Show - 01" or "Show_-_01_"\n    def extract_dash_numbers(filename: str) -> List[Tuple[int, int, str]]:\n        """Returns list of (position_index, number_value, matched_text) for dash-separated numbers."""\n        results = []\n        # Pattern handles: "- 01 ", "- 01.", "_-_01_", "- 01(", "- 0724 " (4-digit)\n        # Negative lookahead (?![xX]\\d) skips NNxNN patterns (e.g., "- 01x05")\n        for i, m in enumerate(re.finditer(r\'[-–—]_?(\\d{1,4})(?![xX]\\d)(?:[_\\s\\.(\\[]|$)\', filename)):\n            num = int(m.group(1))\n            results.append((i + 100, num, m.group(0)))  # offset to distinguish from brackets\n        return results\n    \n    # Also extract space-separated numbers like "Show Name 01 Title" or "Slam Dunk 100"\n    def extract_space_numbers(filename: str) -> List[Tuple[int, int, str]]:\n        """Returns list of (position_index, number_value, matched_text) for space-separated episode numbers."""\n        results = []\n        # Match numbers that are surrounded by spaces (or start of string), followed by more text\n        # Avoid matching years (1900-2099) or resolutions (1080, 720, etc.)\n        # Accept [A-Za-z\\[] after space to handle fansub bracket tags like [NetflixJP]\n        for i, m in enumerate(re.finditer(r\'(?:^|\\s)(\\d{1,4})(?=\\s+[A-Za-z\\[])\', filename)):\n            num = int(m.group(1))\n            # Skip resolutions and other technical numbers\n            if num in (360, 480, 540, 720, 1080, 1440, 2160, 4320):\n                continue\n            if 1900 <= num <= 2099:  # Skip years\n                continue\n            results.append((i + 200, num, m.group(0)))  # offset to distinguish\n        return results\n    \n    # Extract NNxNN season-episode patterns like "01x05", "1x03", "02x15"\n    def extract_nxn_numbers(filename: str) -> List[Tuple[int, int, str]]:\n        """Returns list of (position_index, episode_value, matched_text) for NNxNN patterns."""\n        results = []\n        for i, m in enumerate(re.finditer(r\'(?i)\\b(\\d{1,2})x(\\d{1,4})\\b\', filename)):\n            ep_num = int(m.group(2))  # Extract episode number (after x)\n            results.append((i + 300, ep_num, m.group(0)))  # offset to distinguish\n        return results\n    \n    # Collect patterns from all files (pre-cleaned of file size info like "(126.7 MiB)")\n    all_patterns = []\n    for fname in filenames:\n        clean_name = _strip_size_suffix(fname)\n        patterns = extract_bracket_numbers(clean_name) + extract_dash_numbers(clean_name) + extract_space_numbers(clean_name) + extract_nxn_numbers(clean_name)\n        all_patterns.append((fname, patterns))  # Keep original filename as key\n    \n    if not all_patterns or not all_patterns[0][1]:\n        return {}  # No patterns found\n    \n    # Find which position index has varying values (likely episode numbers)\n    # Group by position index\n    position_values: Dict[int, List[Tuple[str, int]]] = {}\n    for fname, patterns in all_patterns:\n        for pos_idx, num_val, _ in patterns:\n            if pos_idx not in position_values:\n                position_values[pos_idx] = []\n            position_values[pos_idx].append((fname, num_val))\n    \n    # Find positions where values vary AND form a reasonable sequence\n    episode_position = None\n    best_score = 0\n    \n    for pos_idx, file_nums in position_values.items():\n        if len(file_nums) < len(filenames) * 0.8:\n            continue  # Skip if not present in most files\n        \n        values = [n for _, n in file_nums]\n        unique_values = set(values)\n        \n        # Skip if all same value (constant like 1080, 264, etc.)\n        if len(unique_values) == 1:\n            continue\n        \n        # Skip known non-episode numbers (resolutions, years, bit depths)\n        if any(v in (360, 480, 540, 720, 1080, 1440, 2160, 4320, 264, 265, 10) for v in unique_values):\n            if len(unique_values) == 1 or max(unique_values) > 500:\n                continue\n        \n        # Check if values form a reasonable episode sequence\n        sorted_vals = sorted(unique_values)\n        is_sequential = all(sorted_vals[i+1] - sorted_vals[i] <= 2 for i in range(len(sorted_vals)-1))\n        starts_low = min(unique_values) <= 10  # Episodes usually start from 1-10\n        reasonable_range = max(unique_values) <= 500  # Episodes rarely exceed 500\n        \n        # Score this position\n        score = 0\n        if is_sequential: score += 3\n        if starts_low: score += 2\n        if reasonable_range: score += 1\n        if len(unique_values) > 1: score += 1\n        \n        if score > best_score:\n            best_score = score\n            episode_position = pos_idx\n    \n    if episode_position is None:\n        return {}\n    \n    # Map filenames to their episode numbers at the detected position.\n    # Keys are stripped of size suffixes so download-time filenames still match.\n    result = {}\n    for fname, patterns in all_patterns:\n        for pos_idx, num_val, _ in patterns:\n            if pos_idx == episode_position:\n                result[_strip_size_suffix(fname)] = num_val\n                break\n\n    # Explicit SxxEyy markers outrank the varying-number heuristic: when a file\'s\n    # strict marker VARIES across the batch it is that file\'s real season/episode,\n    # so leave the file out of the batch result and let per-file regex parsing read\n    # it (e.g. \'... - 28 - S02E01v2\' is S02E01, not E28 — the dash number is only\n    # the absolute count). A strict marker constant across 2+ files is a pack label\n    # (\'Show.S01E01-E24.../03.mp4\'), and for those the heuristic result stands.\n    strict_pairs = {}\n    for fname in filenames:\n        key = _strip_size_suffix(fname)\n        m = re.search(r\'(?i)\\bS(\\d{1,2})EP?(\\d{1,4})(?:v\\d+)?(?:\\b|(?=EP?\\d))\', key)\n        if m:\n            strict_pairs[key] = (int(m.group(1)), int(m.group(2)))\n    if strict_pairs and (len(strict_pairs) == 1 or len(set(strict_pairs.values())) > 1):\n        for key in strict_pairs:\n            result.pop(key, None)\n\n\n    _batch_episode_cache = result\n    return result\n\ndef get_batch_episode(filename: str) -> Optional[int]:\n    """Get batch-detected episode number for a filename, if available."""\n    return _batch_episode_cache.get(_strip_size_suffix(filename))\n\n# --- TMDB METADATA MATCHING ---\n# Batch-time lookups against TMDB refine the regex-based filename detection:\n# canonical show names, years, and absolute-episode → season mapping. Everything\n# here degrades silently to the regex behaviour when disabled or unreachable.\n_tmdb_match_cache: Dict[str, dict] = {}  # stripped filename -> match dict (per batch)\n_tmdb_query_cache: Dict[str, Optional[dict]] = {}  # "kind|query|year" -> match/None (persistent)\n_tmdb_query_cache_loaded = False\n\ndef tmdb_is_enabled() -> bool:\n    """TMDB matching is active when the checkbox is on and a key is configured."""\n    return tmdb_enabled_checkbox.value and bool(token_tmdb.value.strip())\n\ndef _tmdb_get(path: str, params: dict) -> Optional[dict]:\n    """GET a TMDB v3 endpoint. Returns parsed JSON or None on any failure."""\n    try:\n        full_params = dict(params, api_key=token_tmdb.value.strip())\n        r = requests.get(f"{TMDB_API_BASE}{path}", params=full_params, timeout=REQUEST_TIMEOUT)\n        if r.status_code == 200:\n            return r.json()\n    except Exception:\n        pass\n    return None\n\ndef _tmdb_similarity(a: str, b: str) -> float:\n    return difflib.SequenceMatcher(None, a.casefold().strip(), b.casefold().strip()).ratio()\n\ndef _load_tmdb_query_cache():\n    global _tmdb_query_cache, _tmdb_query_cache_loaded\n    if _tmdb_query_cache_loaded:\n        return\n    _tmdb_query_cache_loaded = True\n    try:\n        if os.path.exists(TMDB_CACHE_FILE):\n            with open(TMDB_CACHE_FILE, \'r\') as f:\n                _tmdb_query_cache = json.load(f)\n    except Exception:\n        _tmdb_query_cache = {}  # Corrupt/unreadable cache — start fresh\n    if _tmdb_query_cache.get(\'__version__\') != TMDB_CACHE_VERSION:\n        # Search got smarter since this cache was written — cached misses (None)\n        # may match now, so drop them; successful matches stay valid.\n        _tmdb_query_cache = {k: v for k, v in _tmdb_query_cache.items()\n                             if v is not None and k != \'__version__\'}\n        _tmdb_query_cache[\'__version__\'] = TMDB_CACHE_VERSION\n\ndef _save_tmdb_query_cache():\n    try:\n        if len(_tmdb_query_cache) > TMDB_QUERY_CACHE_MAX:\n            for k in list(_tmdb_query_cache)[:len(_tmdb_query_cache) - TMDB_QUERY_CACHE_MAX]:\n                del _tmdb_query_cache[k]\n        with open(TMDB_CACHE_FILE, \'w\') as f:\n            json.dump(_tmdb_query_cache, f)\n    except Exception:\n        pass  # Drive not mounted — cache is best-effort\n\ndef _tmdb_alt_titles(kind: str, tmdb_id: int) -> List[str]:\n    """Alternative titles for a result (romaji anime titles live here)."""\n    data = _tmdb_get(f"/{kind}/{tmdb_id}/alternative_titles", {})\n    if not data:\n        return []\n    entries = data.get(\'results\') or data.get(\'titles\') or []  # tv uses \'results\', movie \'titles\'\n    return [e.get(\'title\', \'\') for e in entries if e.get(\'title\')]\n\ndef _tmdb_fetch_tv_seasons(tv_id: int) -> Dict[str, int]:\n    """{season_number(str): episode_count}, specials (season 0) excluded.\n    Keys are strings because the dict round-trips through the JSON cache."""\n    data = _tmdb_get(f"/tv/{tv_id}", {})\n    seasons = {}\n    for s in (data or {}).get(\'seasons\', []):\n        num = s.get(\'season_number\', 0)\n        count = s.get(\'episode_count\', 0)\n        if num > 0 and count > 0:\n            seasons[str(num)] = count\n    return seasons\n\ndef _tmdb_search(kind: str, query: str, year: Optional[str]) -> Optional[dict]:\n    """Search TMDB (\'tv\' or \'movie\') with a similarity gate. Uses the persistent\n    query cache (misses cached as None so they aren\'t re-queried)."""\n    _load_tmdb_query_cache()\n    cache_key = f"{kind}|{query.casefold()}|{year or \'\'}"\n    if cache_key in _tmdb_query_cache:\n        return _tmdb_query_cache[cache_key]\n\n    # Attempt tiers, most specific first. Filename years are often wrong (encode\n    # year, not release year), and a year left inside the query text ("True\n    # Detective 2019") makes TMDB\'s text search return nothing — so degrade\n    # gracefully: drop the year filter, then retry with the year stripped from\n    # the query itself. First match wins, so exact titles still take priority.\n    attempts = [(query, year)]\n    if year:\n        attempts.append((query, None))\n    stripped = re.sub(r"[\\s.]*\\(?(19|20)\\d{2}\\)?\\s*$", \'\', query).strip()\n    if stripped and stripped.casefold() != query.casefold():\n        attempts.append((stripped, year))\n        if year:\n            attempts.append((stripped, None))\n\n    match = None\n    for q, y in attempts:\n        params = {\'query\': q, \'include_adult\': \'false\'}\n        if y:\n            params[\'first_air_date_year\' if kind == \'tv\' else \'year\'] = y\n        data = _tmdb_get(f"/search/{kind}", params)\n        results = (data or {}).get(\'results\') or []\n        for r in results[:5]:\n            name = r.get(\'name\') or r.get(\'title\') or \'\'\n            orig = r.get(\'original_name\') or r.get(\'original_title\') or \'\'\n            score = max(_tmdb_similarity(q, name), _tmdb_similarity(q, orig))\n            if score >= TMDB_MATCH_THRESHOLD:\n                match = r\n                break\n        if match is None and results:\n            # Romaji queries score poorly against localized/original names but live in\n            # alternative titles — check them for the top result only\n            top = results[0]\n            if any(_tmdb_similarity(q, alt) >= TMDB_MATCH_THRESHOLD\n                   for alt in _tmdb_alt_titles(kind, top[\'id\'])):\n                match = top\n        if match is not None:\n            break\n\n    normalized = _tmdb_normalize(kind, match) if match is not None else None\n    _tmdb_query_cache[cache_key] = normalized\n    return normalized\n\ndef _tmdb_normalize(kind: str, result: dict) -> dict:\n    """Normalize a TMDB tv/movie result (from search or a /{kind}/{id} fetch) to the\n    match dict used everywhere: {type, id, name, year[, seasons]}. A full /tv/{id}\n    response already carries \'seasons\'; search results don\'t, so those are fetched."""\n    name = result.get(\'name\') or result.get(\'title\') or \'\'\n    date = result.get(\'first_air_date\') or result.get(\'release_date\') or \'\'\n    normalized = {\n        \'type\': kind,\n        \'id\': result[\'id\'],\n        \'name\': sanitize_filename(name) or \'Unknown\',\n        \'year\': date[:4] if date else \'\',\n    }\n    if kind == \'tv\':\n        if \'seasons\' in result:\n            seasons = {}\n            for s in result.get(\'seasons\', []):\n                num, count = s.get(\'season_number\', 0), s.get(\'episode_count\', 0)\n                if num > 0 and count > 0:\n                    seasons[str(num)] = count\n            normalized[\'seasons\'] = seasons\n        else:\n            normalized[\'seasons\'] = _tmdb_fetch_tv_seasons(result[\'id\'])\n    return normalized\n\ndef _tmdb_fetch_by_id(kind: str, tmdb_id) -> Optional[dict]:\n    """Fetch a specific tv/movie by TMDB id and normalize it (None if not found)."""\n    data = _tmdb_get(f"/{kind}/{tmdb_id}", {})\n    if not data or \'id\' not in data:\n        return None\n    return _tmdb_normalize(kind, data)\n\ndef _resolve_tmdb_override(text: str) -> Optional[dict]:\n    """Resolve a manual correction string to a match dict. Accepts a themoviedb.org\n    URL, a \'tv:12345\' / \'movie:12345\' id, or free text (searched via /search/multi).\n    Does NOT touch the auto-search query cache."""\n    text = text.strip()\n    if not text:\n        return None\n    url_m = re.search(r\'themoviedb\\.org/(tv|movie)/(\\d+)\', text)\n    if url_m:\n        return _tmdb_fetch_by_id(url_m.group(1), url_m.group(2))\n    id_m = re.match(r\'(?i)(tv|movie)\\s*[:=]\\s*(\\d+)$\', text)\n    if id_m:\n        return _tmdb_fetch_by_id(id_m.group(1).lower(), id_m.group(2))\n    # Free text — multi search returns tv/movie/person with a media_type tag\n    data = _tmdb_get("/search/multi", {\'query\': text, \'include_adult\': \'false\'})\n    for r in (data or {}).get(\'results\', []):\n        if r.get(\'media_type\') in (\'tv\', \'movie\'):\n            return _tmdb_normalize(r[\'media_type\'], r)\n    return None\n\ndef _apply_tmdb_overrides(tasks: List[DownloadTask]):\n    """Write each task\'s manual TMDB override into the match cache, overriding any\n    auto-match. Called after analyze_batch_metadata at every entry point so manual\n    corrections survive Quick Download and resume."""\n    for t in tasks:\n        ov = getattr(t, \'tmdb_override\', None)\n        if ov is None or not t.filename:\n            continue\n        key = _match_cache_key(t.filename)\n        if ov == TMDB_CLEARED:\n            _tmdb_match_cache.pop(key, None)  # force regex fallback\n        else:\n            _tmdb_match_cache[key] = ov\n\ndef _selected_queue_indices() -> List[int]:\n    """0-based indices of the queue rows the user has selected."""\n    return [int(s.split(\'.\')[0]) - 1 for s in queue_list.value]\n\ndef apply_tmdb_override(b=None):\n    """Match Selected: resolve the Fix-Match input and apply it to selected rows."""\n    if not tmdb_is_enabled():\n        print("⚠️ Enable TMDB matching (Settings) and set an API key first")\n        return\n    text = tmdb_override_input.value.strip()\n    if not text:\n        print("⚠️ Enter a TMDB URL, tv:12345 / movie:12345, or a title to search")\n        return\n    indices = [i for i in _selected_queue_indices() if 0 <= i < len(pending_queue)]\n    if not indices:\n        print("⚠️ Select the queue item(s) to correct first")\n        return\n    match = _resolve_tmdb_override(text)\n    if not match:\n        print(f"❌ No TMDB result for \'{text}\'")\n        return\n    for i in indices:\n        task = pending_queue[i]\n        task.tmdb_override = match\n        if task.filename:\n            _tmdb_match_cache[_match_cache_key(task.filename)] = match\n    tmdb_override_input.value = ""\n    update_queue_display()\n    label = match[\'name\'] + (f" ({match[\'year\']})" if match[\'year\'] else "")\n    print(f"✅ Matched {len(indices)} item(s) → {label}")\n\ndef clear_tmdb_override(b=None):\n    """Clear Match: drop the TMDB match for selected rows (use filename parsing)."""\n    indices = [i for i in _selected_queue_indices() if 0 <= i < len(pending_queue)]\n    if not indices:\n        print("⚠️ Select the queue item(s) to clear first")\n        return\n    for i in indices:\n        task = pending_queue[i]\n        task.tmdb_override = TMDB_CLEARED\n        if task.filename:\n            _tmdb_match_cache.pop(_match_cache_key(task.filename), None)\n    update_queue_display()\n    print(f"✖ Cleared TMDB match for {len(indices)} item(s) — will use filename parsing")\n\n# --- MANUAL QUEUE OVERRIDES (season / episode / name / route) ---\n# Mirror the TMDB Fix-Match pattern: overrides live on the task (so they persist\n# with the session across Stop/Resume) and are mirrored into filename-keyed caches\n# that determine_destination_path can read from any thread. Force Season solves\n# multi-season batches whose filenames carry no season marker; Renumber solves\n# absolute-numbered batches whose real season split TMDB can\'t provide (select the\n# season\'s files, Force Season, then Renumber from 1).\n_season_override_cache: Dict[str, int] = {}  # stripped filename -> forced season (per batch)\n_episode_override_cache: Dict[str, int] = {}  # stripped filename -> forced episode (per batch)\n_episode_end_override_cache: Dict[str, int] = {}  # stripped filename -> forced range end (per batch)\n_name_override_cache: Dict[str, Tuple[str, str]] = {}  # stripped filename -> (forced name, year)\n_route_override_cache: Dict[str, str] = {}  # stripped filename -> forced route category\n_part_override_cache: Dict[str, int] = {}  # stripped filename -> forced part (N = -ptN, 0 = none)\n\ndef get_season_override(filename: str) -> Optional[int]:\n    """Manual season for a filename (None when not overridden)."""\n    return _season_override_cache.get(_match_cache_key(filename))\n\ndef get_episode_override(filename: str) -> Optional[int]:\n    """Manual episode for a filename (None when not overridden)."""\n    return _episode_override_cache.get(_match_cache_key(filename))\n\ndef get_episode_end_override(filename: str) -> Optional[int]:\n    """Manual multi-episode range end for a filename (None = single episode)."""\n    return _episode_end_override_cache.get(_match_cache_key(filename))\n\ndef get_name_override(filename: str) -> Optional[Tuple[str, str]]:\n    """Manual (name, year) for a filename (None when not overridden)."""\n    return _name_override_cache.get(_match_cache_key(filename))\n\ndef get_route_override(filename: str) -> Optional[str]:\n    """Manual route category for a filename (None when not overridden):\n    \'tv\' | \'anime_series\' | \'movie\' | \'anime_movie\' | \'downloads\'."""\n    return _route_override_cache.get(_match_cache_key(filename))\n\ndef get_part_override(filename: str) -> Optional[int]:\n    """Manual part suffix for a filename: None = auto-detect, 0 = force no\n    suffix, N >= 1 = force -ptN."""\n    return _part_override_cache.get(_match_cache_key(filename))\n\ndef _apply_queue_overrides(tasks: List[DownloadTask]):\n    """Rebuild the season/episode/name/route override caches from the tasks\'\n    persisted overrides. Called at every entry point (queue preview, start, quick,\n    resume); clearing first means overrides never leak across batches with\n    colliding filenames."""\n    _season_override_cache.clear()\n    _episode_override_cache.clear()\n    _episode_end_override_cache.clear()\n    _name_override_cache.clear()\n    _route_override_cache.clear()\n    _part_override_cache.clear()\n    for t in tasks:\n        if not t.filename:\n            continue\n        key = _match_cache_key(t.filename)\n        sov = getattr(t, \'season_override\', None)\n        if sov is not None:\n            _season_override_cache[key] = int(sov)\n        eov = getattr(t, \'episode_override\', None)\n        if eov is not None:\n            _episode_override_cache[key] = int(eov)\n        eev = getattr(t, \'episode_end_override\', None)\n        if eev is not None:\n            _episode_end_override_cache[key] = int(eev)\n        nov = getattr(t, \'name_override\', None)\n        if nov:\n            _name_override_cache[key] = (nov, getattr(t, \'year_override\', None) or \'\')\n        rov = getattr(t, \'route_override\', None)\n        if rov:\n            _route_override_cache[key] = rov\n        pov = getattr(t, \'part_override\', None)\n        if pov is not None:\n            _part_override_cache[key] = int(pov)\n\n\ndef apply_season_override(b=None):\n    """Set Season: force a season number for the selected queue rows."""\n    text = season_override_input.value.strip()\n    if not text.isdigit():\n        print("⚠️ Enter a season number first (e.g. 2 — use 0 for Specials)")\n        return\n    indices = [i for i in _selected_queue_indices() if 0 <= i < len(pending_queue)]\n    if not indices:\n        print("⚠️ Select the queue item(s) to set the season for first")\n        return\n    season = int(text)\n    for i in indices:\n        task = pending_queue[i]\n        task.season_override = season\n        if task.filename:\n            _season_override_cache[_match_cache_key(task.filename)] = season\n    season_override_input.value = ""\n    update_queue_display()\n    print(f"✅ Season {season:02d} forced for {len(indices)} item(s)")\n\ndef clear_season_override(b=None):\n    """Clear Season: drop the forced season for the selected rows."""\n    indices = [i for i in _selected_queue_indices() if 0 <= i < len(pending_queue)]\n    if not indices:\n        print("⚠️ Select the queue item(s) to clear first")\n        return\n    for i in indices:\n        task = pending_queue[i]\n        task.season_override = None\n        if task.filename:\n            _season_override_cache.pop(_match_cache_key(task.filename), None)\n    update_queue_display()\n    print(f"✖ Cleared forced season for {len(indices)} item(s) — will use filename/TMDB detection")\n\ndef _number_rows_sequentially(rows: List[DownloadTask], start: int) -> Dict[str, int]:\n    """Assign sequential numbers to the rows in queue order, returned as\n    {task.id: number}. Subtitles never consume numbers: each inherits the number\n    of the video its name extends — matched by stem, so the pairing works no\n    matter how the release spells the language tag (Eng, big5, zh-Hant, …).\n    Subtitles with no video in the selection number themselves."""\n    def key_stem(task):\n        return os.path.splitext(_match_cache_key(task.filename))[0]\n\n    # Pass 1: number the videos (every non-subtitle row) by stem, in queue order\n    assigned: Dict[str, int] = {}  # stem -> number (deduped, so a re-run is stable)\n    numbers: Dict[str, int] = {}\n    subs, video_stems = [], []\n    for task in rows:\n        if os.path.splitext(_strip_size_suffix(task.filename))[1].lower() in KEEP_EXTENSIONS:\n            subs.append(task)\n            continue\n        stem = key_stem(task)\n        if stem not in assigned:\n            assigned[stem] = start + len(assigned)\n            video_stems.append(stem)\n        numbers[task.id] = assigned[stem]\n\n    # Pass 2: subtitles pair with the video whose stem equals theirs or is a\n    # dot-boundary prefix of it (\'Show - 28.big5.ass\' → \'Show - 28.mkv\'), longest\n    # match winning; unmatched subtitles get their own numbers like any other row\n    for task in subs:\n        stem = key_stem(task)\n        match = max((v for v in video_stems if stem == v or stem.startswith(v + \'.\')),\n                    key=len, default=None)\n        pair = match if match is not None else stem\n        if pair not in assigned:\n            assigned[pair] = start + len(assigned)\n        numbers[task.id] = assigned[pair]\n    return numbers\n\ndef apply_renumber(b=None):\n    """Renumber: rewrite the selected rows\' episode numbers sequentially, in queue\n    order, starting from the 🔢 field (default 1). A video and its subtitle share\n    one number (see _number_rows_sequentially). A range like 7-9 makes the FIRST\n    file a multi-episode span (S01E07-E09); the rest continue as single episodes\n    from the end of the range (E10, E11, …)."""\n    text = renumber_start_input.value.strip() or \'1\'\n    range_m = re.fullmatch(r\'(\\d{1,4})(?:\\s*[-~–—]\\s*(\\d{1,4}))?\', text)\n    if not range_m:\n        print("⚠️ Enter the episode number to start from (e.g. 7), or a range for a multi-episode file (e.g. 7-9)")\n        return\n    start = int(range_m.group(1))\n    span_end = int(range_m.group(2)) if range_m.group(2) else None\n    if span_end is not None and span_end <= start:\n        print("⚠️ The range end must be greater than the start (e.g. 7-9)")\n        return\n    indices = [i for i in _selected_queue_indices() if 0 <= i < len(pending_queue)]\n    if not indices:\n        print("⚠️ Select the queue item(s) to renumber first")\n        return\n    rows = [pending_queue[i] for i in sorted(indices) if pending_queue[i].filename]\n    skipped = len(indices) - len(rows)\n    numbers = _number_rows_sequentially(rows, start)\n    # Range mode: the first file (number == start) absorbs the whole span; later\n    # files shift past it so numbering continues from the end of the range\n    offset = (span_end - start) if span_end else 0\n    for task in rows:\n        key = _match_cache_key(task.filename)\n        if span_end and numbers[task.id] == start:\n            task.episode_override, task.episode_end_override = start, span_end\n            _episode_override_cache[key], _episode_end_override_cache[key] = start, span_end\n        else:\n            task.episode_override = numbers[task.id] + offset\n            task.episode_end_override = None\n            _episode_override_cache[key] = task.episode_override\n            _episode_end_override_cache.pop(key, None)\n\n    renumber_start_input.value = ""\n    update_queue_display()\n    if numbers:\n        last = max(numbers.values()) + offset\n        note = f" ({skipped} without filenames skipped)" if skipped else ""\n        first = f"E{start:02d}-E{span_end:02d}" if span_end else f"E{start:02d}"\n        tail = ""\n        if span_end and last > span_end:\n            tail = f", then E{span_end + 1:02d}" + (f"–E{last:02d}" if last > span_end + 1 else "")\n        elif not span_end and last > start:\n            tail = f"–E{last:02d}"\n        print(f"✅ Renumbered {len(rows)} item(s) → {first}{tail}{note}")\n    else:\n        print("⚠️ Selected item(s) have no filenames yet — resolve them first")\n\n\ndef clear_renumber(b=None):\n    """Clear Renumber: drop the forced episode numbers for the selected rows."""\n    indices = [i for i in _selected_queue_indices() if 0 <= i < len(pending_queue)]\n    if not indices:\n        print("⚠️ Select the queue item(s) to clear first")\n        return\n    for i in indices:\n        task = pending_queue[i]\n        task.episode_override = None\n        task.episode_end_override = None\n        if task.filename:\n            _episode_override_cache.pop(_match_cache_key(task.filename), None)\n            _episode_end_override_cache.pop(_match_cache_key(task.filename), None)\n    update_queue_display()\n    print(f"✖ Cleared forced episode numbers for {len(indices)} item(s) — will use filename/TMDB detection")\n\ndef apply_part_override(b=None):\n    """Set Part: append -ptN suffixes to the selected rows sequentially, in queue\n    order, starting from the 📎 field (default 1) — e.g. 1 with three files\n    selected → -pt1, -pt2, -pt3. A video and its subtitle share one part number.\n    Wins over Part X / 上篇 filename detection."""\n    text = part_override_input.value.strip() or \'1\'\n    if not text.isdigit() or int(text) < 1:\n        print("⚠️ Enter the part number to start from (e.g. 1 → -pt1)")\n        return\n    indices = [i for i in _selected_queue_indices() if 0 <= i < len(pending_queue)]\n    if not indices:\n        print("⚠️ Select the queue item(s) to set the part for first")\n        return\n    start = int(text)\n    rows = [pending_queue[i] for i in sorted(indices) if pending_queue[i].filename]\n    skipped = len(indices) - len(rows)\n    numbers = _number_rows_sequentially(rows, start)\n    for task in rows:\n        task.part_override = numbers[task.id]\n        _part_override_cache[_match_cache_key(task.filename)] = numbers[task.id]\n\n    part_override_input.value = ""\n    update_queue_display()\n    if numbers:\n        last = max(numbers.values())\n        note = f" ({skipped} without filenames skipped)" if skipped else ""\n        label = f"-pt{start}" if last == start else f"-pt{start}…-pt{last}"\n        print(f"✅ Part suffix {label} set for {len(rows)} item(s){note}")\n    else:\n        print("⚠️ Selected item(s) have no filenames yet — resolve them first")\n\ndef remove_part_override(b=None):\n    """No Part: strip any part suffix (detected or forced) from the selected rows."""\n    indices = [i for i in _selected_queue_indices() if 0 <= i < len(pending_queue)]\n    if not indices:\n        print("⚠️ Select the queue item(s) to strip the part suffix from first")\n        return\n    for i in indices:\n        task = pending_queue[i]\n        task.part_override = 0\n        if task.filename:\n            _part_override_cache[_match_cache_key(task.filename)] = 0\n    update_queue_display()\n    print(f"✖ Part suffix removed for {len(indices)} item(s)")\n\ndef apply_name_override(b=None):\n    """Set Name: force a show/movie name (and optional year) for the selected rows —\n    the manual counterpart of a TMDB match, and it wins over one."""\n    name = sanitize_filename(queue_name_input.value.strip())\n    year = queue_year_input.value.strip()\n    if not name:\n        print("⚠️ Enter the name to force first")\n        return\n    if year and not re.fullmatch(r\'(19|20)\\d{2}\', year):\n        print("⚠️ Year should look like 2025 (leave it empty to omit)")\n        return\n    indices = [i for i in _selected_queue_indices() if 0 <= i < len(pending_queue)]\n    if not indices:\n        print("⚠️ Select the queue item(s) to name first")\n        return\n    for i in indices:\n        task = pending_queue[i]\n        task.name_override = name\n        task.year_override = year or None\n        if task.filename:\n            _name_override_cache[_match_cache_key(task.filename)] = (name, year)\n    queue_name_input.value = ""\n    queue_year_input.value = ""\n    update_queue_display()\n    label = f"{name} ({year})" if year else name\n    print(f"✅ Name forced for {len(indices)} item(s) → {label}")\n\ndef clear_name_override(b=None):\n    """Clear Name: drop the forced name/year for the selected rows."""\n    indices = [i for i in _selected_queue_indices() if 0 <= i < len(pending_queue)]\n    if not indices:\n        print("⚠️ Select the queue item(s) to clear first")\n        return\n    for i in indices:\n        task = pending_queue[i]\n        task.name_override = None\n        task.year_override = None\n        if task.filename:\n            _name_override_cache.pop(_match_cache_key(task.filename), None)\n    update_queue_display()\n    print(f"✖ Cleared forced name for {len(indices)} item(s) — will use TMDB/filename detection")\n\ndef apply_route_override(b=None):\n    """Apply Route: send the selected rows to a specific library folder. Selecting\n    \'Auto\' clears the override (same as ✖ Clear Route)."""\n    route = route_dropdown.value\n    if not route:\n        clear_route_override()\n        return\n    indices = [i for i in _selected_queue_indices() if 0 <= i < len(pending_queue)]\n    if not indices:\n        print("⚠️ Select the queue item(s) to route first")\n        return\n    for i in indices:\n        task = pending_queue[i]\n        task.route_override = route\n        if task.filename:\n            _route_override_cache[_match_cache_key(task.filename)] = route\n    update_queue_display()\n    label = next(l for l, v in route_dropdown.options if v == route)\n    print(f"✅ Routed {len(indices)} item(s) → {label}")\n\ndef clear_route_override(b=None):\n    """Clear Route: drop the forced route for the selected rows."""\n    indices = [i for i in _selected_queue_indices() if 0 <= i < len(pending_queue)]\n    if not indices:\n        print("⚠️ Select the queue item(s) to clear first")\n        return\n    for i in indices:\n        task = pending_queue[i]\n        task.route_override = None\n        if task.filename:\n            _route_override_cache.pop(_match_cache_key(task.filename), None)\n    update_queue_display()\n    print(f"✖ Cleared forced route for {len(indices)} item(s) — will detect the category again")\n\n\n\ndef _map_absolute_episode(episode: int, seasons: Dict[str, int]) -> Tuple[int, int]:\n    """Convert an absolute episode number to (season, episode) using per-season\n    counts. Returns (1, episode) unchanged when mapping doesn\'t apply — including\n    episodes beyond TMDB\'s known total (a season TMDB hasn\'t listed yet)."""\n    if not seasons:\n        return (1, episode)\n    order = sorted(seasons, key=lambda k: int(k))\n    total = sum(seasons[k] for k in order)\n    if episode <= seasons.get(\'1\', 0) or episode > total:\n        return (1, episode)\n    remaining = episode\n    for k in order:\n        if remaining <= seasons[k]:\n            return (int(k), remaining)\n        remaining -= seasons[k]\n    return (1, episode)\n\ndef get_tmdb_match(filename: str) -> Optional[dict]:\n    """Batch-cached TMDB match for a filename (None when disabled or unmatched)."""\n    if not tmdb_is_enabled():\n        return None\n    return _tmdb_match_cache.get(_match_cache_key(filename))\n\ndef analyze_batch_metadata(filenames: List[str]) -> int:\n    """Match a batch of filenames against TMDB. Populates _tmdb_match_cache keyed\n    by _match_cache_key (size-suffix and subtitle-language stripped). One search per\n    distinct (kind, query, year). Returns the number of files matched."""\n    _tmdb_match_cache.clear()\n    if not tmdb_is_enabled() or not filenames:\n        return 0\n    queries: Dict[Tuple[str, str, Optional[str]], List[str]] = {}\n    for fname in filenames:\n        key = _match_cache_key(fname)  # subtitle-lang stripped so subs key like their video\n        info = detect_episode_info(key)\n        kind = \'tv\' if info[\'episode_detected\'] else \'movie\'\n        year_m = re.search(r\'\\b(19|20)\\d{2}\\b\', key)\n        year = year_m.group(0) if year_m else None\n        if kind == \'tv\':\n            query = info[\'show_name\']\n        else:\n            query = clean_show_name(key[:year_m.start()]) if year_m else clean_show_name(os.path.splitext(key)[0])\n        if not query or query == \'Unknown Show\':\n            continue\n        queries.setdefault((kind, query, year), []).append(key)\n\n    matched = 0\n    had_error = False\n    for (kind, query, year), keys in queries.items():\n        try:\n            match = _tmdb_search(kind, query, year)\n        except Exception:\n            had_error = True\n            match = None\n        if match:\n            for key in keys:\n                _tmdb_match_cache[key] = match\n                matched += 1\n    _save_tmdb_query_cache()\n    if had_error:\n        print("   ⚠️ TMDB: some lookups failed — falling back to filename parsing")\n    return matched\n\ndef check_duplicate_in_drive(filename: str, source: str = "generic", playlist_index: Optional[int] = None) -> bool:\n    """Check if file already exists in Drive to avoid re-downloading"""\n    dest_path, category = determine_destination_path(filename, source, dry_run=True, playlist_index=playlist_index)\n    if os.path.exists(dest_path):\n        file_size = os.path.getsize(dest_path) / (1024 * 1024)\n        print(f"   ⏭️  SKIPPED (Already exists): {os.path.basename(dest_path)} ({file_size:.1f} MB)")\n        return True\n    # Uploaded by the API earlier this session: really in Drive, but the mount has\n    # not polled for remote changes yet, so os.path.exists above cannot see it.\n    if _drive_path_exists(dest_path):\n        print(f"   ⏭️  SKIPPED (uploaded earlier this session): {os.path.basename(dest_path)}")\n        return True\n    return False\n\n# Multi-episode continuation glued directly to a strict/NxN match: \'-E03\',\n# \'-S01E03\', \'E02\' (S01E01E02), \'-03\', \'-1x03\'. Anchored — a space before the\n# tail means an episode title (\'S01E05 - 12 Angry Men\'), not a range.\n_MULTI_EP_TAIL = re.compile(r\'(?i)^(?:[-~–—]\\s*(?:S(\\d{1,2})\\s*)?(?:\\d{1,2}x)?EP?(\\d{1,4})|EP?(\\d{1,4})|[-~–—]\\s*(\\d{1,4}))\\b\')\n\ndef _multi_ep_end(filename: str, m, season_num: int) -> Optional[int]:\n    """End episode of a multi-episode file (\'S01E01-E03\', \'S01E01E02\',\n    \'S01E01-03\', \'1x01-02\') read from the text directly after the strict/NxN\n    match m. None when absent or implausible: resolution/year tails, a\n    different season on the end marker, or end <= start."""\n    tail_m = _MULTI_EP_TAIL.match(filename[m.end():])\n    if not tail_m:\n        return None\n    if tail_m.group(1) is not None and int(tail_m.group(1)) != season_num:\n        return None\n    end = int(next(g for g in tail_m.groups()[1:] if g))\n    if end <= int(m.group(2)):\n        return None\n    if end in (360, 480, 540, 720, 1080, 1440, 2160, 4320) or 1900 <= end <= 2099:\n        return None\n    return end\n\ndef detect_episode_info(filename: str) -> Dict[str, Any]:\n    """Parse a filename for episode/season/show-name markers.\n\n    Pure function (no widget/UI access) so the detection logic can be\n    unit-tested directly. Returns a dict with keys: episode_detected,\n    is_tv, season, episode, show_name, part_suffix (CJK multi-part),\n    english_part_suffix ("Part X"), has_sxe (strict SxxExx/NxN present),\n    episode_end (end of a multi-episode range, None for single episodes).\n    """\n    # CJK multi-part markers always apply (these genuinely split one episode into parts)\n    part_suffix = ""\n    if "上篇" in filename: part_suffix = "-pt1"\n    elif "下篇" in filename: part_suffix = "-pt2"\n    elif "中篇" in filename: part_suffix = "-pt2"\n    # English "Part X" detected separately — only applied when no SxxExx/NxN pattern exists\n    # (e.g. "S01E25 - The Real Folk Blues (Part 1)" → Part 2 is S01E26, not S01E25-pt2)\n    english_part_suffix = ""\n    if re.search(r\'(?i)(?:Part|Pt)\\.?\\s*1\\b\', filename): english_part_suffix = "-pt1"\n    elif re.search(r\'(?i)(?:Part|Pt)\\.?\\s*2\\b\', filename): english_part_suffix = "-pt2"\n\n    # Episode RANGE markers (EP01-70, E01-E24) name batch packs, not one episode —\n    # without this, \'Show.EP01-70.../01.mp4\' reads as episode 1 for every file.\n    # Mask the range with same-length spaces (indices stay aligned) so per-file\n    # numbers elsewhere drive detection, and keep its start as a show-name boundary.\n    range_m = re.search(r\'(?i)\\b(?:Ep?|Episode)[ ._-]?(\\d{1,4})\\s*[-~–—]\\s*(?:Ep?|Episode)?[ ._]?(\\d{1,4})\\b\', filename)\n    if range_m and int(range_m.group(2)) > int(range_m.group(1)):\n        filename = filename[:range_m.start()] + \' \' * (range_m.end() - range_m.start()) + filename[range_m.end():]\n    else:\n        range_m = None\n    def _name_cut(idx: int) -> int:\n        """Show name ends at the first marker — episode match or masked range."""\n        return min(idx, range_m.start()) if range_m else idx\n\n    f_ext = os.path.splitext(filename)[1]\n    def _name_after(marker_end: int) -> str:\n        """Show name read from AFTER the episode marker — for marker-first names\n        (\'1x02 - Chernobyl [x265].mkv\') where nothing usable precedes it."""\n        tail = filename[marker_end:]\n        if f_ext and tail.lower().endswith(f_ext.lower()):\n            tail = tail[:-len(f_ext)]\n        return clean_show_name(tail) if len(tail.strip(\' ._-\')) > 2 else "Unknown Show"\n\n    show_name = "Unknown Show"\n    \n    # Optional \'P\' after the E covers the SxxEPyy style (e.g. S02EP05 = S02E05); the\n    # glued \'EP\' otherwise breaks both this and the loose \\bEP pattern, so the episode\n    # goes undetected and every file in a season collapses to E01 (skipped as dupes).\n    # The (?=EP?\\d) alternative lets glued double-episode names (S01E01E02) match\n    # their first episode — \\b alone fails between the digits and the second E.\n    sxe_strict = re.search(r\'(?i)\\bS(\\d{1,2})EP?(\\d{1,4})(?:v\\d+)?(?:\\b|(?=EP?\\d))\', filename)\n    # NNxNN pattern: matches 01x05, 1x03, 02x15, etc. (common TV naming convention)\n    sxe_nxn = re.search(r\'(?i)\\b(\\d{1,2})x(\\d{1,4})\\b\', filename)\n    # Added Vietnamese "Tập", Korean "화", Portuguese "Episodio", and more flexible episode patterns\n    sxe_loose = re.search(r\'(?i)(?:\\b(?:Ep?|Episode|Episodio|Tập|Tập phim|Folge|Capitulo|Cap)[ .\\-_]?(\\d{1,4})\\b|[|\\-–—]\\s*(?:Ep?|Episode|Tập)?\\s*(\\d{1,4})\\s*[|\\]]?)\', filename)\n    sxe_asian = re.search(r\'(?:第(\\d+)[集話]|(\\d+)화)\', filename)\n    \n    # Bracketed episode pattern: matches [01], [02], [0724], etc. common in fansub releases\n    # Filters out resolution tags [1080P/720P/etc], codec tags [HEVC-10b/x265/etc], and source tags\n    sxe_bracket = None\n    bracket_matches = list(re.finditer(r\'\\[(\\d{1,4})\\]\', filename))\n    for bm in bracket_matches:\n        num = int(bm.group(1))\n        # Skip if it looks like a resolution (360, 480, 720, 1080, 2160, etc.) \n        # or a year (1900-2099) or a bit depth suffix like "10b" in [HEVC-10b]\n        if num in (360, 480, 540, 720, 1080, 1440, 2160, 4320):\n            continue\n        if 1900 <= num <= 2099:\n            continue\n        # Check if this bracket is part of a codec tag like [HEVC-10b] or [x264-10bit]\n        # Look for pattern where number is preceded by hyphen inside the bracket context\n        bracket_content_before = filename[max(0, bm.start()-20):bm.start()]\n        if re.search(r\'\\[[^\\]]*-$\', bracket_content_before):\n            continue  # Skip, it\'s likely a suffix like -10b\n        # This looks like a valid episode number\n        sxe_bracket = bm\n        break\n    \n    # Trailing number pattern: catches "HD 01", "Show Name 05", "filename - 03" before extension\n    # Uses negative lookbehind to avoid matching years (19xx, 20xx) and resolutions (1080, 720, etc.)\n    base_name = os.path.splitext(filename)[0]  # Remove extension for cleaner matching\n    # Years (19xx, 20xx) and resolutions are filtered below via an explicit range/set\n    # check — a leading lookbehind can\'t do this: \\d{1,4} greedily consumes all of a\n    # 4-digit year itself, so a lookbehind placed before the match never sees it.\n    sxe_trailing = re.search(r\'(?<!x)\\b(\\d{1,4})\\s*$\', base_name)\n    # Filter out likely years or resolutions captured by trailing pattern\n    if sxe_trailing:\n        num = int(sxe_trailing.group(1))\n        # Reject if it looks like a year (1900-2099) or resolution (360, 480, 720, 1080, 2160, etc.)\n        if 1900 <= num <= 2099 or num in (360, 480, 540, 720, 1080, 1440, 2160, 4320):\n            sxe_trailing = None\n    \n    # Underscore-dash pattern: handles _-_01_ or - 1042 format (common in high-episode anime)\n    # Negative lookahead (?![xX]\\d) prevents matching the season part of NNxNN patterns\n    sxe_underscore = re.search(r\'[-–—]_?(\\d{1,4})(?![xX]\\d)(?:_|\\(|\\s|$)\', filename)\n    \n    # Space-separated pattern: handles "Show Name 01 Title" or "Show Name 0724 [Tag]" format\n    # Exclude "Part X" which indicates movie sequels, not episodes\n    sxe_space = re.search(r\'(?:^|[\\s_])(\\d{1,4})(?=\\s+[A-Za-z\\[])\', filename)\n    if sxe_space:\n        num = int(sxe_space.group(1))\n        # Check if preceded by "Part" - indicates movie, not episode\n        pre_match = filename[:sxe_space.start() + 1]\n        if re.search(r\'(?i)\\bPart\\s*$\', pre_match):\n            sxe_space = None\n        elif num in (360, 480, 540, 720, 1080, 1440, 2160, 4320) or 1900 <= num <= 2099:\n            sxe_space = None\n\n    season_num, episode_num = 1, 1\n    is_tv = False\n    episode_detected = False\n\n    # PRIORITY 1: Use batch-detected episode if available (most reliable)\n    batch_ep = get_batch_episode(filename)\n    if batch_ep is not None:\n        episode_num = batch_ep\n        is_tv = True\n        episode_detected = True\n        # For show name, find the episode marker position\n        # Handle: [01], " 01 ", "0724 [Tag]", "- 01", "_-_01_", "01x05" (NNxNN)\n        # Try NNxNN pattern first (extracts both season and episode marker position)\n        nxn_marker = re.search(r\'(?i)\\b(\\d{1,2})x0*\' + str(batch_ep) + r\'\\b\', filename)\n        ep_marker = None\n        if nxn_marker:\n            season_num = int(nxn_marker.group(1))\n            show_name = clean_show_name(filename[:_name_cut(nxn_marker.start())])\n        else:\n            ep_marker = re.search(r\'\\[0*\' + str(batch_ep) + r\'\\]|(?:^|\\s)0*\' + str(batch_ep) + r\'(?=\\s+[A-Za-z\\[])|[-–—]_?0*\' + str(batch_ep) + r\'(?:[_\\s\\.\\(\\[]|$)\', filename)\n            if ep_marker:\n                show_name = clean_show_name(filename[:_name_cut(ep_marker.start())])\n            else:\n                show_name = clean_show_name(filename[:_name_cut(len(filename))])\n        # Marker-first names: nothing usable before the marker — look after it\n        marker = nxn_marker or ep_marker\n        if marker is not None and (show_name == \'Unknown Show\' or len(show_name) < 2):\n            show_name = _name_after(marker.end())\n    \n    # PRIORITY 2: Fall back to regex pattern matching if batch detection didn\'t find it\n    if not episode_detected:\n        # Collect all valid matches and find the earliest one to split correctly\n        matches = []\n        if sxe_strict: matches.append({\'m\': sxe_strict, \'type\': \'strict\', \'idx\': sxe_strict.start(), \'priority\': 1})\n        if sxe_nxn: matches.append({\'m\': sxe_nxn, \'type\': \'nxn\', \'idx\': sxe_nxn.start(), \'priority\': 1})\n        # Bracketed episode numbers have high priority (common in fansub releases)\n        if sxe_bracket: matches.append({\'m\': sxe_bracket, \'type\': \'bracket\', \'idx\': sxe_bracket.start(), \'priority\': 1})\n        if sxe_loose: matches.append({\'m\': sxe_loose, \'type\': \'loose\', \'idx\': sxe_loose.start(), \'priority\': 2})\n        if sxe_asian: matches.append({\'m\': sxe_asian, \'type\': \'asian\', \'idx\': sxe_asian.start(), \'priority\': 2})\n        # Underscore pattern: use if higher priority patterns not found\n        if sxe_underscore and not matches: \n            matches.append({\'m\': sxe_underscore, \'type\': \'underscore\', \'idx\': sxe_underscore.start(), \'priority\': 3})\n        # Space pattern: use if higher priority patterns not found\n        if sxe_space and not matches: \n            matches.append({\'m\': sxe_space, \'type\': \'space\', \'idx\': sxe_space.start(), \'priority\': 3})\n        # Trailing pattern is lowest priority - only use if no other patterns found\n        if sxe_trailing and not matches: \n            matches.append({\'m\': sxe_trailing, \'type\': \'trailing\', \'idx\': sxe_trailing.start(), \'priority\': 4})\n    \n        if matches:\n            # Sort by start index to find the FIRST occurrence (splitting show name from episode info)\n            best = min(matches, key=lambda x: x[\'idx\'])\n            match, m_type = best[\'m\'], best[\'type\']\n            \n            if m_type == \'strict\':\n                season_num, episode_num = int(match.group(1)), int(match.group(2))\n            elif m_type == \'nxn\':\n                season_num, episode_num = int(match.group(1)), int(match.group(2))\n            elif m_type == \'bracket\':\n                episode_num = int(match.group(1))\n            elif m_type == \'loose\':\n                ep_num = match.group(1) or match.group(2)\n                episode_num = int(ep_num) if ep_num else 1\n            elif m_type == \'asian\':\n                ep_num = match.group(1) or match.group(2)\n                episode_num = int(ep_num) if ep_num else 1\n            elif m_type == \'underscore\':\n                episode_num = int(match.group(1))\n            elif m_type == \'space\':\n                episode_num = int(match.group(1))\n            elif m_type == \'trailing\':\n                episode_num = int(match.group(1))\n                \n            # An explicit SxxEyy/NxN anywhere in the name outranks the value read\n            # from an earlier loose/absolute number (\'Show - 28 - S02E01v2\' is\n            # S02E01, not E28) — the earliest match still splits the show name.\n            if m_type not in (\'strict\', \'nxn\'):\n                if sxe_strict:\n                    season_num, episode_num = int(sxe_strict.group(1)), int(sxe_strict.group(2))\n                elif sxe_nxn:\n                    season_num, episode_num = int(sxe_nxn.group(1)), int(sxe_nxn.group(2))\n\n            # For trailing pattern, use base_name (without extension) for show name extraction\n            name_source = base_name if m_type == \'trailing\' else filename\n            show_name = clean_show_name(name_source[:_name_cut(match.start())])\n            # Marker-first names (\'1x02 - Chernobyl [x265].mkv\') have nothing usable\n            # before the marker — read the show name from the text after it instead\n            if show_name == \'Unknown Show\' or len(show_name) < 2:\n                show_name = _name_after(match.end())\n                \n            is_tv = True\n            episode_detected = True\n\n    # Multi-episode range (S01E01-E03 → 3): only when the strict/NxN marker is\n    # what actually set the episode — a batch-detected number means the marker\n    # was a pack label, so its glued range is not this file\'s span\n    episode_end = None\n    span_src = sxe_strict or sxe_nxn\n    if episode_detected and span_src is not None and episode_num == int(span_src.group(2)):\n        episode_end = _multi_ep_end(filename, span_src, season_num)\n\n    return {\n        \'episode_detected\': episode_detected,\n        \'is_tv\': is_tv,\n        \'season\': season_num,\n        \'episode\': episode_num,\n        \'show_name\': show_name,\n        \'part_suffix\': part_suffix,\n        \'english_part_suffix\': english_part_suffix,\n        \'has_sxe\': bool(sxe_strict or sxe_nxn),\n        \'episode_end\': episode_end,\n    }\n\ndef determine_destination_path(filename: str, source: str = "generic", dry_run: bool = False,\n                               playlist_index: Optional[int] = None, relative_path: Optional[str] = None,\n                               torrent_name: Optional[str] = None) -> Tuple[str, str]:\n    filename = sanitize_filename(filename)\n    \n    # If auto-organise is disabled, just return Downloads folder with original filename —\n    # or, for a file that came from a torrent (relative_path is not None), rebuild the\n    # torrent\'s own folder layout under Downloads/<TorrentName>/<subfolder>/<file> instead\n    # of flattening every file from every torrent into one folder.\n    if not is_auto_organize_enabled():\n        downloads_dir = os.path.join(DRIVE_BASE, get_downloads_path())\n        if relative_path is not None:\n            parts = [sanitize_filename(torrent_name)] if torrent_name else []\n            if relative_path:\n                parts.extend(sanitize_filename(p) for p in relative_path.split(\'/\') if p)\n            downloads_dir = os.path.join(downloads_dir, *parts) if parts else downloads_dir\n        if not dry_run: _ensure_dest_dir(downloads_dir)  # API path owns folder creation\n        return os.path.join(downloads_dir, filename), "Downloads"\n\n    # 🎯 Route as \'Downloads (as-is)\': keep the original name, skip organising —\n    # the per-file version of switching auto-organise off\n    route_ov = get_route_override(filename)\n    if route_ov == \'downloads\':\n        downloads_dir = os.path.join(DRIVE_BASE, get_downloads_path())\n        if not dry_run: _ensure_dest_dir(downloads_dir)  # API path owns folder creation\n        return os.path.join(downloads_dir, filename), "Downloads"\n    # Anime routes pick the anime library folders in the branches below\n    is_anime = route_ov in (\'anime_series\', \'anime_movie\')\n    \n    # Parse episode/show info from the filename (pure logic, unit-testable)\n    info = detect_episode_info(filename)\n    part_suffix = info[\'part_suffix\']\n    english_part_suffix = info[\'english_part_suffix\']\n    show_name = info[\'show_name\']\n    season_num, episode_num = info[\'season\'], info[\'episode\']\n    episode_end = info[\'episode_end\']  # multi-episode file (S01E01-E03), else None\n    is_tv = info[\'is_tv\']\n    episode_detected = info[\'episode_detected\']\n\n    # Manual name/year (queue ✏️ Force Name) — the manual counterpart of a TMDB match\n    name_ov = get_name_override(filename)\n    manual_show_name = name_ov[0] if name_ov else \'\'\n    manual_year = name_ov[1] if name_ov else \'\'\n\n    # 🎯 Route override decides movie-vs-series regardless of filename detection\n    if route_ov in (\'movie\', \'anime_movie\'):\n        is_tv = False\n        episode_detected = False  # Treat as movie, ignore detected episode\n    elif route_ov in (\'tv\', \'anime_series\'):\n        is_tv = True\n        if not episode_detected:\n            episode_num = 1  # Default to E01 if no episode detected\n\n    # Manual season override (queue 🗂️ Force Season) wins over filename parsing and\n    # TMDB mapping — the parsed number stays the episode, only the season is forced\n    season_ov = get_season_override(filename)\n    if season_ov is not None:\n        season_num = season_ov\n\n    # Manual renumbering (queue 🔢 Renumber) wins over every episode source, and\n    # marks the file as an episode even when detection found none\n    episode_ov = get_episode_override(filename)\n    if episode_ov is not None:\n        episode_num = episode_ov\n        # A renumber replaces the episode identity outright: a plain number makes\n        # the file a single episode, a range (🔢 7-9) makes it span E07-E09\n        episode_end = get_episode_end_override(filename)\n        is_tv = True\n        if not episode_detected:\n            episode_detected = True\n            # Nothing was parsed from the name, so \'Unknown Show\' would become the\n            # folder — use the cleaned filename stem instead (Force Name still wins)\n            if show_name == \'Unknown Show\':\n                show_name = clean_show_name(os.path.splitext(filename)[0]) or show_name\n\n    # TMDB metadata (populated by analyze_batch_metadata at queue/quick/resume time).\n    # Force Name always wins; a match refines names/years, never user input.\n    tmdb_match = None if manual_show_name else get_tmdb_match(filename)\n    tmdb_tv_year = \'\'\n    if tmdb_match and tmdb_match[\'type\'] == \'tv\' and (is_tv or episode_detected):\n        show_name = tmdb_match[\'name\']\n        tmdb_tv_year = tmdb_match[\'year\']\n        # Absolute-numbering conversion (e.g. "One Piece - 1085" → S19Exx) applies only\n        # when the filename carried no explicit SxxExx/NxN season marker, no season is\n        # manually forced, and Numbering is \'Season match\' (not \'Absolute\')\n        if (episode_detected and not info[\'has_sxe\'] and season_num == 1\n                and season_ov is None and episode_ov is None\n                and episode_numbering_toggle.value == \'Season match\'):\n            season_num, episode_num = _map_absolute_episode(episode_num, tmdb_match.get(\'seasons\') or {})\n\n    # Apply Force Name override - affects both TV shows and movies\n    if manual_show_name:\n        if is_tv or episode_detected:\n            # TV show: use forced name as show name\n            show_name = manual_show_name\n            if not episode_detected and playlist_index is not None:\n                episode_num = playlist_index\n        else:\n            # Movie: use forced name as folder/file name\n            folder_name = f"{manual_show_name} ({manual_year})" if manual_year else manual_show_name\n            _, ext = os.path.splitext(filename)\n            new_filename = f"{manual_show_name}{ext}"\n            if is_anime:\n                full_dir = os.path.join(f"{DRIVE_BASE}{get_anime_movies_path()}", folder_name)\n                if not dry_run: _ensure_dest_dir(full_dir)  # API path owns folder creation\n                return os.path.join(full_dir, new_filename), "Anime Movies"\n            else:\n                full_dir = os.path.join(f"{DRIVE_BASE}{get_movie_path()}", folder_name)\n                if not dry_run: _ensure_dest_dir(full_dir)  # API path owns folder creation\n                return os.path.join(full_dir, new_filename), "Movies"\n    elif is_tv:\n        pass  # Continue to TV show path generation below\n    else:\n        # No Force Name, not TV - detect movie (TMDB match wins over filename parsing)\n        if tmdb_match and tmdb_match[\'type\'] == \'movie\':\n            movie_name = tmdb_match[\'name\']\n            folder_name = f"{movie_name} ({tmdb_match[\'year\']})" if tmdb_match[\'year\'] else movie_name\n        else:\n            year_match = re.search(r\'\\b(19|20)\\d{2}\\b\', filename)\n            if year_match:\n                movie_name = clean_show_name(filename[:year_match.start()])\n                year = year_match.group(0)\n                folder_name = f"{movie_name} ({year})"\n            elif source == "youtube":\n                return os.path.join(f"{DRIVE_BASE}{get_youtube_path()}", filename), "YouTube"\n            else:\n                movie_name = clean_show_name(os.path.splitext(filename)[0])\n                folder_name = movie_name\n        _, ext = os.path.splitext(filename)\n        # Year goes on the folder only — the file keeps just the movie name\n        new_filename = f"{movie_name}{ext}"\n        # Use anime folder if anime mode is enabled\n        if is_anime:\n            full_dir = os.path.join(f"{DRIVE_BASE}{get_anime_movies_path()}", folder_name)\n            if not dry_run: _ensure_dest_dir(full_dir)  # API path owns folder creation\n            return os.path.join(full_dir, new_filename), "Anime Movies"\n        else:\n            full_dir = os.path.join(f"{DRIVE_BASE}{get_movie_path()}", folder_name)\n            if not dry_run: _ensure_dest_dir(full_dir)  # API path owns folder creation\n            return os.path.join(full_dir, new_filename), "Movies"\n\n    _, ext = os.path.splitext(filename)\n    # Apply English part suffix only when no SxxExx/NxN pattern was detected and the\n    # episode wasn\'t manually renumbered — either one already uniquely identifies the\n    # episode (Part 2 is typically the next episode, or the user assigned it a number)\n    if english_part_suffix and not info[\'has_sxe\'] and episode_ov is None:\n        part_suffix = part_suffix or english_part_suffix\n    # Manual part override (queue 📎 Set Part / ✖ No Part) wins over all detection\n    part_ov = get_part_override(filename)\n    if part_ov is not None:\n        part_suffix = f"-pt{part_ov}" if part_ov else ""\n    # Multi-episode files keep their span in the Plex convention (S01E01-E03)\n    ep_span = f"-E{episode_end:02d}" if episode_end and episode_end > episode_num else ""\n    new_filename = f"{show_name} - S{season_num:02d}E{episode_num:02d}{ep_span}{part_suffix}{ext}"\n    season_folder = "Specials" if season_num == 0 else f"Season {season_num:02d}"\n    # Append year to show folder name only (file name stays without year)\n    folder_year = manual_year or tmdb_tv_year\n    show_folder = f"{show_name} ({folder_year})" if folder_year else show_name\n    \n    # Use anime folder if anime mode is enabled\n    if is_anime:\n        base_path = f"{DRIVE_BASE}{get_anime_series_path()}"\n        full_dir = os.path.join(base_path, show_folder, season_folder)\n        if not dry_run: _ensure_dest_dir(full_dir)  # API path owns folder creation\n        return os.path.join(full_dir, new_filename), "Anime Series"\n    else:\n        base_path = f"{DRIVE_BASE}{get_tv_path()}"\n        full_dir = os.path.join(base_path, show_folder, season_folder)\n        if not dry_run: _ensure_dest_dir(full_dir)  # API path owns folder creation\n        return os.path.join(full_dir, new_filename), "TV"\n\n# --- CORE LOGIC ---\ndef setup_environment(needs_mega, needs_ytdlp, needs_aria):\n    drive_path = f"{COLAB_ROOT}drive"\n    if drive is not None and not os.path.exists(drive_path): drive.mount(drive_path)\n    \n    # Try to load secrets again (may not have been accessible on initial load)\n    check_and_load_secrets()\n    \n    # Restore saved settings now that Drive (and settings.json) is finally readable —\n    # the startup load ran before the mount and found nothing. Widgets the user already\n    # changed this session keep their values; then save once so choices made before the\n    # mount (unsaveable at the time — no Drive) are persisted too.\n    load_dir_settings()\n    save_dir_settings()\n\n    # Drive API auth happens here, on the main thread: Colab renders a consent\n    # prompt, and settings (which decide whether the API is wanted at all) have\n    # only just been restored above.\n    if drive_api_checkbox.value:\n        _init_drive_api()\n    \n    # Create media folders and config folder\n    for p in [get_tv_path(), get_movie_path(), get_youtube_path()]:\n        full_p = f"{DRIVE_BASE}{p}"\n        _ensure_dest_dir(full_p)\n    if not os.path.exists(UD_CONFIG_PATH): os.makedirs(UD_CONFIG_PATH)\n    \n    if needs_ytdlp:\n        # Always upgrade yt-dlp to latest version (YouTube changes frequently)\n        print("🛠️ Installing/updating yt-dlp...")\n        subprocess.run(["pip", "install", "-U", "yt-dlp"], check=True, stdout=subprocess.DEVNULL)\n    else:\n        print("⭐️ Skipping yt-dlp (Not needed)")\n\n    pkg_map = {\n        "unrar": "unrar", \n        "p7zip-full": "7z", \n        "megatools": "megadl", \n        "aria2": "aria2c", \n        "ffmpeg": "ffmpeg"\n    }\n    \n    needed_pkgs = ["unrar", "p7zip-full"]\n    if needs_mega: needed_pkgs.append("megatools")\n    if needs_aria: needed_pkgs.append("aria2")\n    if needs_ytdlp: needed_pkgs.append("ffmpeg")\n    \n    to_install = [pkg for pkg in needed_pkgs if not shutil.which(pkg_map[pkg])]\n\n    if to_install:\n        print(f"🛠️ Installing tools: {\', \'.join(to_install)}...")\n        subprocess.run(["apt-get", "update", "-qq"], check=False)\n        subprocess.run(["apt-get", "install", "-y"] + to_install, \n                       check=True, stdout=subprocess.DEVNULL)\n    else:\n        print("✅ Required tools already present.")\n    \n    check_resume_available()\n\ndef ytdl_hook(d):\n    if d[\'status\'] == \'downloading\':\n        try:\n            p = d.get(\'_percent_str\', \'0%\').replace(\'%\',\'\')\n            speed = d.get(\'_speed_str\', \'N/A\')\n            with progress_lock:\n                progress_bar.value = float(p)\n                progress_bar.description = f"YT: {p}% ({speed})"\n        except Exception: pass\n    elif d[\'status\'] == \'finished\':\n        with progress_lock:\n            progress_bar.value = 100\n            progress_bar.description = "Done!"\n\ndef get_youtube_title(url: str) -> str:\n    """Quickly fetch video/playlist title for queue display."""\n    try:\n        import yt_dlp\n        ydl_opts = {\n            \'quiet\': True,\n            \'no_warnings\': True,\n            \'extract_flat\': True,  # Don\'t download, just get metadata\n            \'skip_download\': True,\n        }\n        with yt_dlp.YoutubeDL(ydl_opts) as ydl:\n            info = ydl.extract_info(url, download=False)\n            if info:\n                title = info.get(\'title\', \'\')\n                # For playlists, show playlist name + count\n                if info.get(\'_type\') == \'playlist\':\n                    count = len(info.get(\'entries\', []))\n                    return f"📋 {title} ({count} videos)"\n                return title[:60] + "..." if len(title) > 60 else title\n    except Exception:\n        pass\n    return ""  # Fall back to showing URL\n\ndef resolve_youtube_playlist(url: str) -> List[DownloadTask]:\n    """Expand a YouTube playlist URL into individual video DownloadTasks for queue display.\n    For single videos, returns a single-element list.\n    Uses extract_flat for fast metadata-only extraction.\n    """\n    try:\n        import yt_dlp\n        ydl_opts = {\n            \'quiet\': True,\n            \'no_warnings\': True,\n            \'extract_flat\': True,  # Fast: get metadata without full extraction\n            \'skip_download\': True,\n        }\n        with yt_dlp.YoutubeDL(ydl_opts) as ydl:\n            info = ydl.extract_info(url, download=False)\n            if info and info.get(\'_type\') == \'playlist\':\n                entries = [e for e in info.get(\'entries\', []) if e is not None]\n                playlist_title = info.get(\'title\', \'Playlist\')\n                tasks = []\n                for i, entry in enumerate(entries, 1):\n                    video_title = entry.get(\'title\', f\'Video {i}\')\n                    video_url = entry.get(\'url\') or entry.get(\'webpage_url\') or entry.get(\'id\', \'\')\n                    if video_url and not video_url.startswith(\'http\'):\n                        video_url = f"https://www.youtube.com/watch?v={video_url}"\n                    display = f"[{i}/{len(entries)}] {video_title}"\n                    if len(display) > 60:\n                        display = display[:57] + "..."\n                    tasks.append(DownloadTask(\n                        url=video_url, filename=display,\n                        source="youtube", link_type="youtube",\n                        original_url=url  # Store original playlist URL\n                    ))\n                if tasks:\n                    print(f"   📋 Expanded playlist \'{playlist_title}\': {len(tasks)} videos")\n                return tasks\n            elif info:\n                # Single video — return as-is\n                title = info.get(\'title\', \'\')\n                display = title[:60] + "..." if len(title) > 60 else title\n                return [DownloadTask(\n                    url=url, filename=display or url[:50] + "...",\n                    source="youtube", link_type="youtube"\n                )]\n    except Exception as e:\n        print(f"   ⚠️ Could not expand YouTube link: {e}")\n    # Fallback: return single task with URL\n    return [DownloadTask(url=url, filename=url[:50] + "...", source="youtube", link_type="youtube")]\n\n\ndef get_youtube_subtitles(url: str) -> dict:\n    """Fetch available manual subtitles (not auto-generated) from YouTube video.\n    Returns dict of {lang_code: lang_name} or empty dict if none available.\n    """\n    try:\n        import yt_dlp\n        ydl_opts = {\n            \'quiet\': True,\n            \'no_warnings\': True,\n            \'skip_download\': True,\n            \'writesubtitles\': True,\n            \'listsubtitles\': False,  # We\'ll get them from info dict\n        }\n        with yt_dlp.YoutubeDL(ydl_opts) as ydl:\n            info = ydl.extract_info(url, download=False)\n            if info:\n                # For playlists, get subs from first available entry\n                if info.get(\'_type\') == \'playlist\':\n                    entries = info.get(\'entries\', [])\n                    for entry in entries:\n                        if entry and entry.get(\'subtitles\'):\n                            info = entry\n                            break\n                    else:\n                        return {}\n                \n                # Get manual subtitles only (not automatic_captions)\n                subtitles = info.get(\'subtitles\', {})\n                if not subtitles:\n                    return {}\n                \n                # Build {code: name} dict\n                result = {}\n                for lang_code, formats in subtitles.items():\n                    # Skip if it\'s a weird format or live_chat\n                    if lang_code.startswith(\'live_chat\'):\n                        continue\n                    # Get language name from first format or use code\n                    lang_name = formats[0].get(\'name\', lang_code) if formats else lang_code\n                    # Clean up: "English" not "English - en"\n                    if \' - \' in lang_name:\n                        lang_name = lang_name.split(\' - \')[0]\n                    result[lang_code] = lang_name\n                return result\n    except Exception:\n        pass\n    return {}\n\ndef process_youtube_link(url, mode="video", apply_playlist_range=True) -> Tuple[int, int, int]:\n    """Process YouTube link. Returns (success_count, fail_count, total_count).\n    \n    Args:\n        url: YouTube URL to process\n        mode: \'video\' or \'subtitles\'\n        apply_playlist_range: If False, download all items (ignore playlist_selection)\n    """\n    import yt_dlp\n    print(f"   ▶️ Processing Video: {url}")\n    if not _wait_for_disk_space(DISK_START_GB, label="YouTube download"):\n        return (0, 1, 1)\n    with progress_lock:\n        progress_bar.value = 0\n        progress_bar.description = "Starting..."\n    \n    success_count = 0\n    fail_count = 0\n    skip_count = 0\n    \n    archive_path = f"{UD_CONFIG_PATH}yt_history.txt"\n    # Only apply playlist range if flag is True and user provided a range\n    playlist_items = normalize_playlist_range(playlist_selection.value) if apply_playlist_range else None\n    \n    ydl_opts = {\n        \'outtmpl\': {\n            \'default\': f\'{COLAB_ROOT}%(title)s.%(ext)s\',\n            \'subtitle\': f\'{COLAB_ROOT}%(title)s.%(ext)s\',  # Match video naming for subtitles\n        },\n        \'quiet\': True, \'no_warnings\': True, \n        \'restrictfilenames\': False, \n        \'ignoreerrors\': True, \n        \'writesubtitles\': True, \n        \'subtitleslangs\': [f\'{lang}.*\' if lang == \'en\' else lang for lang in subtitle_langs.value] or [\'en\'],  # Use selected languages \n        \'subtitlesformat\': \'srt\', \n        \'progress_hooks\': [ytdl_hook], \n        \'noprogress\': True,\n        \'download_archive\': archive_path,\n    }\n    \n    if playlist_items:\n        ydl_opts[\'playlist_items\'] = playlist_items\n        print(f"   🎯 Playlist filter: {playlist_items}")\n    \n    # Experimental: Use cookies if available (may cause issues - use Clear Cookies if errors occur)\n    if os.path.exists(COOKIE_PATH): \n        print(f"      🍪 Cookies detected (experimental)")\n        ydl_opts[\'cookiefile\'] = COOKIE_PATH\n    \n    if mode == "video":\n        # Best quality available (including 4K), with fallback to combined formats\n        ydl_opts[\'format\'] = \'bestvideo+bestaudio/best\'\n        ydl_opts[\'merge_output_format\'] = \'mkv\'\n    else:\n        ydl_opts[\'skip_download\'] = True\n    \n    try:\n        with yt_dlp.YoutubeDL(ydl_opts) as ydl:\n            try: \n                info = ydl.extract_info(url, download=False)\n            except Exception as e:\n                print(f"   ❌ YouTube Error: {str(e)[:100]}")\n                return (0, 1, 1)\n            if not info: \n                return (0, 1, 1)\n            \n            # Get entries, filtering out None values (unavailable videos)\n            if \'entries\' in info:\n                raw_entries = list(info[\'entries\'])\n                entries = [e for e in raw_entries if e is not None]\n                none_count = len(raw_entries) - len(entries)\n                if none_count > 0:\n                    print(f"   ⚠️ {none_count} videos unavailable in playlist")\n                    fail_count += none_count\n            else:\n                entries = [info]\n            \n            total_items = len(entries)\n            print(f"   📜 Processing {total_items} item(s)...")\n            \n            for i, entry in enumerate(entries, 1):\n                if _cancel_requested:\n                    print(f"      🛑 Stopped — skipping remaining {total_items - i + 1} item(s)")\n                    break\n                if not entry:\n                    fail_count += 1\n                    continue\n                \n                # For playlists, entries may have shallow metadata - extract full info per video\n                video_url = entry.get(\'webpage_url\') or entry.get(\'url\') or entry.get(\'id\')\n                if not video_url:\n                    print(f"      [{i}/{total_items}] ⚠️ Skipped: No valid URL found")\n                    fail_count += 1\n                    continue\n                \n                # If entry looks like shallow metadata (no formats), fetch full info\n                if \'formats\' not in entry and \'id\' in entry:\n                    try:\n                        entry = ydl.extract_info(video_url, download=False) or entry\n                    except Exception:\n                        pass  # Fall back to shallow entry if extraction fails\n                \n                title = entry.get(\'title\', \'Unknown\')\n                ext = \'mkv\' if mode == "video" else \'srt\'\n                temp_filename = f"{title}.{ext}"\n                if check_duplicate_in_drive(temp_filename, source="youtube", playlist_index=i):\n                    skip_count += 1\n                    continue\n                \n                print(f"      [{i}/{total_items}] Downloading: {title}")\n                \n                try:\n                    before = set(os.listdir(COLAB_ROOT))\n                    ydl.download([entry.get(\'webpage_url\', entry.get(\'url\'))])\n                    after = set(os.listdir(COLAB_ROOT))\n                    new_files = list(after - before)\n                    \n                    if not new_files:\n                        fail_count += 1\n                        continue\n                    for f in new_files:\n                        if f.endswith((\'.part\', \'.ytdl\')): continue\n                        handle_file_processing(os.path.join(COLAB_ROOT, f), source="youtube")\n                    success_count += 1\n                except Exception as e:\n                    print(f"      ❌ Failed to download {title}: {str(e)[:80]}")\n                    fail_count += 1\n    except Exception as e:\n        print(f"   ❌ YouTube processing failed: {str(e)[:100]}")\n        return (success_count, fail_count + 1, success_count + fail_count + skip_count + 1)\n    \n    with progress_lock:\n        progress_bar.description = "Idle"\n    \n    total = success_count + fail_count + skip_count\n    return (success_count, fail_count, total)\n\ndef process_mega_link(url) -> bool:\n    """Process Mega.nz download. Returns True on success, False on failure."""\n    print(f"   ☁️ Processing Mega: {url}")\n    if not _wait_for_disk_space(DISK_START_GB, label="Mega download"):\n        return False\n    with progress_lock:\n        progress_bar.description = "Mega DL..."\n        progress_bar.value = 0\n        progress_bar.bar_style = \'info\'\n    # megadl can\'t handle /folder/.../file/... URLs — strip /file/ID to download entire folder\n    dl_url = url\n    folder_file_match = re.match(r\'(https?://mega\\.nz/folder/[^/]+#[^/]+)/file/.+\', url)\n    if folder_file_match:\n        dl_url = folder_file_match.group(1)\n        print(f"   📦 Folder/file link detected — downloading entire folder")\n    \n    cmd = [\'megadl\', \'--path\', COLAB_ROOT, dl_url]\n    success = False\n    # Snapshot files before download to detect silent failures\n    skip_names = {\'sample_data\', \'.config\', \'drive\', \'temp_extract\', \'cookies.txt\'}\n    try:\n        files_before = set(os.listdir(COLAB_ROOT))\n    except Exception:\n        files_before = set()\n    try:\n        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, universal_newlines=True)\n        mega_proc_key = f"mega_{str(uuid4())[:8]}"\n        _register_proc(mega_proc_key, process)\n        last_speed = ""\n        for line in process.stdout:\n            match = re.search(r\'(\\d+\\.\\d+)%\', line)\n            speed_match = re.search(r\'(\\d+\\.?\\d*\\s*[KMG]B/s)\', line)\n            if match:\n                try:\n                    val = float(match.group(1))\n                    speed_str = speed_match.group(1) if speed_match else last_speed\n                    if speed_match: last_speed = speed_str\n                    with progress_lock:\n                        progress_bar.value = val\n                        progress_bar.description = f"Mega: {int(val)}% ({speed_str})"\n                except Exception: pass\n        process.wait()\n        _unregister_proc(mega_proc_key)\n        if _cancel_requested:\n            return False  # Terminated by Stop\n        if process.returncode == 0:\n            # Verify files were actually downloaded (megadl can exit 0 for unsupported folder/file URLs)\n            try:\n                files_after = set(os.listdir(COLAB_ROOT))\n            except Exception:\n                files_after = set()\n            new_files = [f for f in (files_after - files_before) if f not in skip_names]\n            if new_files:\n                print("   ✅ Mega Download Complete")\n                with progress_lock:\n                    progress_bar.value = 100\n                for f in new_files:\n                    handle_file_processing(os.path.join(COLAB_ROOT, f), source="mega")\n                success = True\n            else:\n                print("   ❌ Mega: megadl exited OK but no files were downloaded")\n                print("   💡 Tip: Folder/file links may not be supported by megadl. Try using Real-Debrid.")\n        else: \n            print(f"   ❌ Mega Error (Code {process.returncode}) - Possible causes: Invalid link, auth required, or file not found")\n    except Exception as e: \n        print(f"   ❌ Mega Execution Error: {e}")\n    with progress_lock:\n        progress_bar.bar_style = \'info\'\n    return success\n\ndef _speed_to_mbs(value: float, unit: str) -> float:\n    """Convert a speed value with a K/M/G unit prefix to MB/s."""\n    if unit == \'K\': return value / 1024\n    if unit == \'G\': return value * 1024\n    return value\n\ndef download_with_aria2(url: str, filename: str, dest_folder: str, cookie: Optional[str] = None,\n                        task_id: Optional[str] = None, update_bar: bool = True,\n                        url_refresh: Optional[Callable[[], str]] = None) -> Optional[str]:\n    """Thread-safe aria2 download with progress tracking.\n\n    update_bar=False leaves the shared progress bar to the batch monitor thread\n    (parallel downloads); sequential callers keep direct bar updates.\n\n    url_refresh re-mints the download URL between attempts. Debrid direct links\n    are time-limited, and a batch pre-requests every link before the first\n    download starts — so on a long batch the later files\' URLs have expired by\n    the time a worker reaches them, and aria2 gets an error page instead of a\n    file (exit 22). Retrying the same dead URL can never succeed, so callers\n    that can mint a fresh one pass this and each retry gets a live link.\n    """\n    filename = sanitize_filename(filename)\n\n    # Safeguard: if filename is empty, extract from URL\n    if not filename or not filename.strip():\n        filename = os.path.basename(unquote(urlparse(url).path)) or "download"\n        filename = sanitize_filename(filename)\n        if not filename or not filename.strip():\n            filename = f"download_{int(time.time())}"\n\n    if check_duplicate_in_drive(filename):\n        return DUPLICATE_SKIP\n\n    final_path = os.path.join(dest_folder, filename)\n    # A file already sitting at final_path is only genuinely complete if aria2 left\n    # NO .aria2 control file beside it. A large partial WITH a control file — the disk\n    # guard terminating aria2, a give-up, or a runtime restart mid-download — must not\n    # be treated as done, or a truncated video moves to Drive as if complete. Fall\n    # through so the aria2 command below resumes it via -c and finishes the pieces.\n    if (os.path.exists(final_path) and os.path.getsize(final_path) > 1024*1024\n            and not os.path.exists(final_path + \'.aria2\')):\n        return final_path\n    with print_lock:\n        print(f"   ⬇️ Downloading: {filename}")\n\n    with progress_lock:\n        if task_id:\n            download_stats[task_id] = {\'pct\': 0.0, \'speed_mbs\': 0.0}\n        if update_bar:\n            progress_bar.value = 0\n            progress_bar.bar_style = \'info\'\n            progress_bar.description = "Starting..."\n\n    # Disk-space guard: don\'t start writing into a nearly-full disk — wait for\n    # queued Drive moves to drain first (Colab terminates the session when the\n    # local disk fills). Bails out if nothing is freeing space.\n    if not _wait_for_disk_space(DISK_START_GB, label=filename[:40], task_id=task_id):\n        _mark_disk_gave_up(task_id)\n        return None\n\n    # Request-metered hosts get fewer connections and aria2\'s stock 20M split\n    # floor, so a near-complete file can\'t spray Range requests at the limiter.\n    rate_limited = url_matches_host(url, RATE_LIMITED_HOSTS)\n    conn_args = [\'-x\', \'2\', \'-s\', \'2\'] if rate_limited else [\'-x\', \'16\', \'-s\', \'16\']\n    retry_wait = \'30\' if rate_limited else \'2\'\n    cmd = [\'aria2c\', url, \'-d\', dest_folder, \'-o\', filename, *conn_args, \'-k\', \'20M\',\n           \'-c\', \'--file-allocation=none\', \'--user-agent\', \'Mozilla/5.0\',\n           \'--connect-timeout=30\', \'--timeout=60\', \'--max-tries=3\', f\'--retry-wait={retry_wait}\',\n           \'--summary-interval=1\', \'--show-console-readout=true\']\n    if cookie: cmd.extend([\'--header\', f\'Cookie: accountToken={cookie}\'])\n\n    proc_key = task_id or str(uuid4())\n    attempt = 0\n    throttle_waits = 0  # 429 cool-offs taken so far (they don\'t consume an attempt)\n    while attempt < 3:\n        attempt += 1\n        if _cancel_requested:\n            return None\n        try:\n            dl_start = time.time()\n            completed_path = None  # Path reported by aria2\'s "Download complete:" line\n            last_error_line = ""  # aria2\'s own error text, surfaced when the attempt fails\n            disk_paused = False  # Set by the watchdog when it terminates aria2 on low disk\n            last_disk_check = time.time()\n            process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, universal_newlines=True)\n            _register_proc(proc_key, process)\n            last_speed = ""\n            last_speed_mbs = 0.0\n            for line in process.stdout:\n                complete_match = re.search(r\'Download complete:\\s*(\\S.*)\', line)\n                if complete_match:\n                    completed_path = complete_match.group(1).strip()\n                # Keep aria2\'s own diagnosis (status codes, "Timeout", "errorCode")\n                # — without it a failure only ever surfaced as a generic exit code\n                # and the real cause (403/429/expired link) was invisible.\n                err_match = re.search(r\'(errorCode=\\d+.*|status=\\d{3}.*|Timeout\\b.*|\'\n                                      r\'Authorization failed.*|Resource not found.*)\', line)\n                if err_match:\n                    last_error_line = err_match.group(1).strip()[:120]\n                match = re.search(r\'\\((\\d+)%\\)\', line)\n                # aria2 outputs speed as "DL:5.2MiB" or "DL: 5.2MiB/s" - capture various formats\n                speed_match = re.search(r\'DL:\\s*(\\d+\\.?\\d*)\\s*([KMG])i?B\', line)\n                if match:\n                    try:\n                        val = float(match.group(1))\n                        if speed_match:\n                            last_speed = f"{speed_match.group(1)}{speed_match.group(2)}iB/s"\n                            last_speed_mbs = _speed_to_mbs(float(speed_match.group(1)), speed_match.group(2))\n                        with progress_lock:\n                            if task_id:\n                                download_stats[task_id] = {\'pct\': val, \'speed_mbs\': last_speed_mbs}\n                            if update_bar:\n                                progress_bar.value = val\n                                progress_bar.description = f"DL: {int(val)}% ({last_speed})" if last_speed else f"DL: {int(val)}%"\n                    except Exception: pass\n                # Disk-space watchdog: pause before the disk actually fills —\n                # Colab kills the whole session at 0 bytes free. aria2 resumes\n                # the partial via its .aria2 control file when space returns.\n                if not disk_paused and time.time() - last_disk_check >= DISK_CHECK_SECS:\n                    last_disk_check = time.time()\n                    if _disk_free_gb() < DISK_FLOOR_GB and not _cancel_requested:\n                        disk_paused = True\n                        with print_lock:\n                            print(f"      ⏸️ Local disk nearly full — pausing {filename[:40]} until Drive moves free space")\n                        try:\n                            process.terminate()\n                        except Exception:\n                            pass\n            process.wait()\n            _unregister_proc(proc_key)\n            if _cancel_requested:\n                return None  # Terminated by Stop — no fallback scan, no retries\n            if disk_paused:\n                # A disk pause is not a failure — wait for space, then resume the\n                # partial (aria2 -c picks up the .aria2 control file) without\n                # consuming a retry attempt.\n                attempt -= 1\n                if _wait_for_disk_space(DISK_START_GB, label=filename[:40], task_id=task_id):\n                    continue\n                _mark_disk_gave_up(task_id)\n                return None\n            if process.returncode == 0 and os.path.exists(final_path):\n                return final_path\n            if process.returncode == 0:\n                # aria2 succeeded but saved under a different name (common with URL-encoded names).\n                # Most reliable: the path aria2 itself reported as complete.\n                if completed_path and os.path.isfile(completed_path):\n                    with print_lock:\n                        print(f"      ✓ Found downloaded file: {os.path.basename(completed_path)}")\n                    return completed_path\n                # Fallback: a matching file created by THIS attempt. Files with an .aria2\n                # control file belong to another still-running worker - never take those.\n                _, ext = os.path.splitext(filename)\n                for f in os.listdir(dest_folder):\n                    candidate = os.path.join(dest_folder, f)\n                    if not (f.endswith(ext) and os.path.isfile(candidate)):\n                        continue\n                    if os.path.exists(candidate + \'.aria2\'):\n                        continue  # in-progress download owned by another task\n                    if os.path.getmtime(candidate) < dl_start:\n                        continue  # existed before this attempt started\n                    with print_lock:\n                        print(f"      ✓ Found downloaded file: {f}")\n                    return candidate\n            detail = f" — {last_error_line}" if last_error_line else ""\n            with print_lock:\n                if not os.path.exists(final_path):\n                    print(f"      ⚠️ Retry {attempt}/3 - File not found at expected path{detail}")\n                    print(f"         Expected: {final_path}")\n                else:\n                    print(f"      ⚠️ Retry {attempt}/3 - aria2 returned code {process.returncode}{detail}")\n            # A 429 says the account is being throttled, not that the link died:\n            # the URL is still good and aria2 -c has a resumable partial on disk.\n            # Re-minting would only spend more of the rate budget on requestdl\n            # (TorBox\'s most rate-limited endpoint) and make the throttle worse.\n            throttled = \'status=429\' in last_error_line\n            if url_refresh and attempt < 3 and not throttled:\n                # The prefetched debrid link may simply have expired while this file\n                # waited its turn — re-mint it so the retry has a live URL to use.\n                # aria2 -c still resumes whatever partial is already on disk.\n                try:\n                    fresh_url = url_refresh()\n                except Exception:\n                    fresh_url = \'\'\n                if fresh_url and fresh_url != cmd[1]:\n                    cmd[1] = fresh_url\n                    with print_lock:\n                        print(f"      🔄 Re-requested a fresh download link")\n            if throttled and throttle_waits < MAX_THROTTLE_WAITS:\n                # Same treatment as a disk pause: the file is fine and -c resumes it,\n                # so a throttle must not burn a retry — otherwise a 99%-complete file\n                # exhausts its 3 attempts on the last few MB and can never land.\n                throttle_waits += 1\n                attempt -= 1\n                cool_off = THROTTLE_BACKOFF_SECS * throttle_waits\n                with print_lock:\n                    print(f"      ⏳ Rate limited (429) — cooling off {cool_off}s, then resuming from where it stopped")\n                for _ in range(cool_off):\n                    if _cancel_requested:\n                        return None\n                    time.sleep(1)\n                continue\n            time.sleep(2**attempt)\n        except Exception as e:\n            _unregister_proc(proc_key)\n            with print_lock:\n                print(f"      ❌ Download error (attempt {attempt}/3): {str(e)[:80]}")\n            break\n\n    if _cancel_requested:\n        return None\n    with print_lock:\n        print(f"   ❌ Download failed after 3 attempts - Check URL validity or network connection")\n    return None\n\n# --- EMBEDDED SUBTITLE EXTRACTION ---\n# Merged from the standalone Smart Subtitle Extractor. Containers worth probing\n# for embedded subtitle tracks (text subs live almost exclusively in mkv/mp4,\n# but probing the others is one cheap ffprobe call).\nSUB_EXTRACT_VIDEO_EXTS = {\'.mkv\', \'.mp4\', \'.m4v\', \'.webm\', \'.avi\', \'.ts\'}\n# Codecs ffmpeg can convert to .srt. Image-based tracks (PGS/VobSub/DVB) would\n# need OCR, so they are reported and skipped rather than failing the file.\n_TEXT_SUB_CODECS = {\'subrip\', \'srt\', \'ass\', \'ssa\', \'mov_text\', \'webvtt\', \'text\', \'sami\', \'subviewer\', \'mpl2\', \'realtext\', \'stl\', \'vplayer\'}\n\ndef _ensure_ffmpeg() -> bool:\n    """ffmpeg/ffprobe come preinstalled on Colab images; install only when missing\n    (setup_environment installs ffmpeg only when yt-dlp is in play)."""\n    if shutil.which(\'ffmpeg\') and shutil.which(\'ffprobe\'):\n        return True\n    print("🛠️ Installing ffmpeg...")\n    subprocess.run(["apt-get", "install", "-y", "ffmpeg"], check=False,\n                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)\n    return bool(shutil.which(\'ffmpeg\') and shutil.which(\'ffprobe\'))\n\ndef _probe_subtitle_streams(video_path: str) -> List[dict]:\n    """List a video\'s subtitle streams as [{\'rel\', \'lang\', \'codec\'}]. \'rel\' is the\n    index among subtitle streams (what ffmpeg\'s 0:s:N specifier counts); \'lang\'\n    is the tag mapped through _normalize_sub_lang (\'\' when untagged/und)."""\n    try:\n        res = subprocess.run(\n            [\'ffprobe\', \'-v\', \'error\', \'-select_streams\', \'s\',\n             \'-show_entries\', \'stream=index,codec_name:stream_tags=language\',\n             \'-of\', \'json\', video_path],\n            capture_output=True, text=True, timeout=300)\n        streams = json.loads(res.stdout or \'{}\').get(\'streams\', []) if res.returncode == 0 else []\n    except Exception:\n        return []\n    out = []\n    for rel, s in enumerate(streams):\n        tag = ((s.get(\'tags\') or {}).get(\'language\') or \'\').strip()\n        lang = \'\' if tag.casefold() in (\'\', \'und\') else _normalize_sub_lang(tag)\n        out.append({\'rel\': rel, \'lang\': lang, \'codec\': (s.get(\'codec_name\') or \'\').lower()})\n    return out\n\ndef extract_embedded_subs(video_path: str, langs: List[str], any_fallback: bool = False,\n                          quiet: bool = False) -> List[str]:\n    """Extract embedded subtitle tracks to .srt sidecars beside the video.\n\n    For each requested language (Plex-friendly codes, the _normalize_sub_lang\n    convention) the first matching text track becomes \'{video base}.{lang}.srt\'\n    — the exact naming handle_file_processing gives downloaded subtitles. A track\n    tagged more precisely than the request keeps its precision (\'zh\' request +\n    \'cht\' track → \'.zh-Hant.srt\'). With any_fallback, a video whose tracks match\n    no requested language at all yields its first text track as \'{base}.srt\'.\n    Existing sidecars are never overwritten; results under 100 bytes (track\n    exists but is empty) are deleted. Returns the sidecar paths written."""\n    if not (langs or any_fallback) or not _ensure_ffmpeg():\n        return []\n    streams = _probe_subtitle_streams(video_path)\n    if not streams:\n        return []\n    base = os.path.splitext(video_path)[0]\n    targets = []          # (subtitle-relative stream index, sidecar path)\n    lang_matched = False  # a request matched SOME track, even an unconvertible one\n    for want in langs:\n        pick = next((s for s in streams if s[\'lang\'] and\n                     (s[\'lang\'] == want or s[\'lang\'].split(\'-\')[0] == want)), None)\n        if pick is None:\n            continue\n        lang_matched = True\n        if pick[\'codec\'] not in _TEXT_SUB_CODECS:\n            if not quiet:\n                print(f"      ⚠️ \'{want}\' track is image-based ({pick[\'codec\']}) — can\'t convert to .srt: {os.path.basename(video_path)}")\n            continue\n        targets.append((pick[\'rel\'], f"{base}.{pick[\'lang\']}.srt"))\n    if not targets and any_fallback and not lang_matched:\n        pick = next((s for s in streams if s[\'codec\'] in _TEXT_SUB_CODECS), None)\n        if pick is not None:\n            targets.append((pick[\'rel\'], f"{base}.srt"))\n    written = []\n    for rel, srt_path in targets:\n        if os.path.exists(srt_path):\n            continue\n        if cancel_requested():\n            break  # batch cancelled mid-file — stop starting new track extractions\n        try:\n            res = subprocess.run(\n                [\'ffmpeg\', \'-nostdin\', \'-v\', \'error\', \'-i\', video_path,\n                 \'-map\', f\'0:s:{rel}\', \'-c:s\', \'srt\', srt_path],\n                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=1800)\n            ok = res.returncode == 0 and os.path.exists(srt_path) and os.path.getsize(srt_path) > 100\n        except KeyboardInterrupt:\n            # Runtime → Interrupt mid-track: subprocess.run has already killed the\n            # ffmpeg child — drop the partial sidecar so a rescan doesn\'t mistake\n            # it for a finished subtitle, then let the caller handle the stop.\n            try:\n                if os.path.exists(srt_path): os.remove(srt_path)\n            except Exception:\n                pass\n            raise\n        except Exception:\n            ok = False\n        if ok:\n            written.append(srt_path)\n            if not quiet:\n                print(f"      ✅ Extracted: {os.path.basename(srt_path)}")\n        elif os.path.exists(srt_path):\n            try: os.remove(srt_path)\n            except Exception: pass\n    return written\n\ndef _sidecar_exists(f_base: str, lang: str, fset: set) -> bool:\n    """True when \'{f_base}\' already has a sidecar satisfying a request for \'lang\'.\n\n    extract_embedded_subs names sidecars after the TRACK\'s tag, which can be more\n    precise than the request: a \'cht\' track under a \'zh\' request is written as\n    \'.zh-Hant.srt\'. Matching only \'.zh.srt\' here would leave those files looking\n    permanently unextracted, so every rescan would re-probe them over the Drive\n    FUSE mount — the whole cost this listing check exists to avoid."""\n    if f"{f_base}.{lang}.srt" in fset:\n        return True\n    prefix = f"{f_base}.{lang}-"\n    return any(n.startswith(prefix) and n.endswith(\'.srt\') for n in fset)\n\ndef run_library_extract(b=None):\n    """Retroactive counterpart of the auto-extract checkbox: walk a Drive folder\n    and write missing .srt sidecars from embedded tracks. Synchronous like every\n    download (Runtime → Interrupt execution to stop a long scan). Probing reads\n    each video over the Drive FUSE mount, so files whose requested sidecars all\n    exist are skipped straight from the directory listing without touching the\n    video at all."""\n    folder = extract_dir_input.value.strip().strip(\'/\')\n    langs = list(extract_sub_langs.value)\n    any_fb = extract_any_track_checkbox.value\n    if not langs and not any_fb:\n        print("❌ Select at least one subtitle language (or the first-track fallback).")\n        return\n    if drive is not None and not os.path.exists(DRIVE_BASE):\n        print("📂 Mounting Google Drive...")\n        drive.mount(f"{COLAB_ROOT}drive")\n    target = os.path.join(DRIVE_BASE, folder) if folder else DRIVE_BASE\n    if not os.path.isdir(target):\n        print(f"❌ Folder not found: {target}")\n        return\n    if not _ensure_ffmpeg():\n        print("❌ ffmpeg unavailable — cannot extract subtitles.")\n        return\n    btn_extract_library.disabled = True\n    _reset_cancel_state()  # a stale cancel flag from an interrupted batch must not mute extraction\n    probed = extracted = 0\n    try:\n        label = \', \'.join(langs) if langs else \'first text track\'\n        print(f"🚀 Scanning \'{folder or \'My Drive\'}\' for embedded subtitle tracks ({label})...")\n        for root, dirs, files in os.walk(target):\n            fset = set(files)\n            for f in sorted(files):\n                f_base, f_ext = os.path.splitext(f)\n                if f_ext.lower() not in SUB_EXTRACT_VIDEO_EXTS:\n                    continue\n                if langs:\n                    needed = [l for l in langs if not _sidecar_exists(f_base, l, fset)]\n                    if not needed:\n                        continue  # every requested sidecar already exists\n                else:\n                    needed = []\n                    if f"{f_base}.srt" in fset:\n                        continue\n                probed += 1\n                print(f"   🔎 {f[:70]}...", end=\'\\r\')\n                # Fallback only when NOTHING pre-existed for this file — if some\n                # languages already have sidecars, the missing ones are genuinely\n                # absent from the file, and a first-track grab would just\n                # duplicate an existing language under no name.\n                extracted += len(extract_embedded_subs(\n                    os.path.join(root, f), needed,\n                    any_fallback=any_fb and len(needed) == len(langs)))\n        print(f"\\n✨ Finished: {extracted} subtitle file(s) extracted ({probed} video(s) probed).")\n    except KeyboardInterrupt:\n        # Same contract as a download batch: kernel interrupt stops cleanly.\n        # Every sidecar written so far is complete (the interrupted track\'s\n        # partial file was already removed), so running the scan again simply\n        # continues from where this one stopped.\n        print(f"\\n🛑 Stopped by user (kernel interrupt) — {extracted} subtitle file(s) extracted ({probed} video(s) probed). Run again to continue.")\n    finally:\n        btn_extract_library.disabled = False\n\n# --- DRIVE API UPLOAD ---\n# Writing through the drivefs FUSE mount is a write-back cache: the copy returns as\n# soon as the bytes are staged locally and drivefs\'s own background uploader pushes\n# them to Google afterwards. Two consequences, both bad. "Transfer complete" is a\n# lie — files keep uploading after a batch ends, and a terminated runtime loses\n# whatever is still queued even though the local copy was already deleted and the\n# history says it landed. And throughput is capped by that one background uploader,\n# so no amount of writer concurrency changes it (measured: 6 concurrent writers gave\n# no improvement at all). Uploading through the Drive REST API instead makes\n# completion mean completion and puts the transfer under our own control.\nDRIVE_API_CHUNK_MB = 64      # Resumable upload chunk; must be a multiple of 256 KB\nDRIVE_API_MAX_ATTEMPTS = 4   # Per-file retries on transient (429/5xx) errors\nDRIVE_FOLDER_RECHECK_SECS = 3  # Re-look-up before creating a folder (search index lag)\n\n_drive_api_ready = False               # Auth succeeded and the client imports cleanly\n_drive_api_warned = False              # One-shot notice when a file falls back to the mount\n_drive_api_local = local()             # googleapiclient is NOT thread-safe; one client per thread\n_drive_folder_ids = {}                 # Drive-relative dir -> folder ID (resolved once)\n_drive_folder_lock = RLock()           # Reentrant: _drive_folder_id recurses under it\n_drive_uploaded_paths = set()          # Landed via API, not yet visible through the mount\n_drive_uploaded_lock = Lock()\n\ndef _init_drive_api() -> bool:\n    """Authenticate for the Drive REST API. Main thread only — Colab renders a\n    consent prompt, which does not work reliably from a worker thread. Never fatal:\n    on failure uploads keep using the mount exactly as before."""\n    global _drive_api_ready\n    if _drive_api_ready:\n        return True\n    try:\n        from google.colab import auth\n        from googleapiclient.discovery import build\n        import logging\n        # google_auth_httplib2 warns that httplib2 has no per-request timeout on every\n        # single request — several lines per uploaded file, which buries the real output.\n        logging.getLogger(\'google_auth_httplib2\').setLevel(logging.ERROR)\n        auth.authenticate_user()\n        build(\'drive\', \'v3\', cache_discovery=False)  # Fail here, not mid-batch\n        _drive_api_ready = True\n        print("🔐 Drive API ready — uploads bypass the Drive mount")\n    except Exception as e:\n        _drive_api_ready = False\n        print(f"⚠️ Drive API unavailable ({str(e)[:70]}) — uploads will use the Drive mount")\n    return _drive_api_ready\n\ndef _use_drive_api() -> bool:\n    return bool(drive_api_checkbox.value) and _drive_api_ready\n\ndef _ensure_dest_dir(path: str):\n    """Create a Drive destination directory — unless the API owns folder creation.\n\n    Two creators is one too many. A mkdir through the mount makes a real Drive\n    folder, but Drive\'s search index lags behind it, so _drive_folder_id\'s own\n    lookup can miss it and create a second folder with the same name. That is what\n    produced the duplicate \'Rome (2005)\' and \'Deadwood (2004)\' folders, and it needs\n    no thread race at all — one file on one thread is enough.\n\n    With the API on, _drive_folder_id is the single creator and builds the whole path\n    itself. The mount fallback in _move_with_progress_impl creates what it needs at\n    copy time instead."""\n    if _use_drive_api():\n        return\n    os.makedirs(path, exist_ok=True)\n\ndef _warn_api_fallback(err: Exception):\n    """Announce the first API failure of the session, then stay quiet."""\n    global _drive_api_warned\n    with print_lock:\n        if not _drive_api_warned:\n            _drive_api_warned = True\n            print(f"   ⚠️ Drive API upload failed ({str(err)[:80]}) — falling back to the Drive mount")\n\ndef _drive_service():\n    """Per-thread Drive client. googleapiclient sits on httplib2, which is not\n    thread-safe — one shared service across parallel workers corrupts responses."""\n    svc = getattr(_drive_api_local, \'svc\', None)\n    if svc is None:\n        from googleapiclient.discovery import build\n        svc = build(\'drive\', \'v3\', cache_discovery=False)\n        _drive_api_local.svc = svc\n    return svc\n\ndef _drive_escape(name: str) -> str:\n    """Escape a name for use inside a Drive API query string literal."""\n    return name.replace(\'\\\\\', \'\\\\\\\\\').replace("\'", "\\\\\'")\n\ndef _drive_folder_id(rel_dir: str) -> str:\n    """Resolve a Drive-relative directory to a folder ID, creating what is missing.\n\n    Memoised, so a folder costs one lookup per session.\n\n    The whole lookup-then-create runs under the lock, and that is load-bearing:\n    Drive allows duplicate sibling names, so when several uploads raced to resolve a\n    brand-new show folder they each saw "not found" and each created it — three\n    identical \'Rome (2005)\' folders with the episodes scattered across them. Checking\n    and creating has to be one atomic step. The lock is an RLock because this\n    recurses up the path, and resolution is memoised, so serialising it costs a\n    couple of API calls per new folder and nothing per file."""\n    rel_dir = rel_dir.strip(\'/\')\n    if not rel_dir:\n        return \'root\'\n    with _drive_folder_lock:\n        cached = _drive_folder_ids.get(rel_dir)\n        if cached:\n            return cached\n        parent_rel, _, leaf = rel_dir.rpartition(\'/\')\n        parent_id = _drive_folder_id(parent_rel) if parent_rel else \'root\'\n        svc = _drive_service()\n        q = ("\'%s\' in parents and name=\'%s\' and mimeType=\'application/vnd.google-apps.folder\'"\n             " and trashed=false" % (parent_id, _drive_escape(leaf)))\n        found = svc.files().list(q=q, fields=\'files(id)\', pageSize=1,\n                                 supportsAllDrives=True, includeItemsFromAllDrives=True\n                                 ).execute().get(\'files\', [])\n        if not found:\n            # Drive\'s search index lags creates by a few seconds. Re-check once before\n            # adding a folder, so one that exists but is not yet indexed (created by an\n            # earlier session, or through the mount) is found instead of duplicated.\n            time.sleep(DRIVE_FOLDER_RECHECK_SECS)\n            found = svc.files().list(q=q, fields=\'files(id)\', pageSize=1,\n                                     supportsAllDrives=True, includeItemsFromAllDrives=True\n                                     ).execute().get(\'files\', [])\n        if found:\n            fid = found[0][\'id\']\n        else:\n            meta = {\'name\': leaf, \'mimeType\': \'application/vnd.google-apps.folder\', \'parents\': [parent_id]}\n            fid = svc.files().create(body=meta, fields=\'id\', supportsAllDrives=True).execute()[\'id\']\n        _drive_folder_ids[rel_dir] = fid\n        return fid\n\ndef _drive_clear_existing(folder_id: str, name: str):\n    """Trash any same-named file already in the target folder.\n\n    The caller\'s duplicate check reads the mount, which lags behind API uploads, so\n    it can miss a file that really is there — and Drive would happily keep both\n    copies under one name."""\n    svc = _drive_service()\n    q = "\'%s\' in parents and name=\'%s\' and trashed=false" % (folder_id, _drive_escape(name))\n    for f in svc.files().list(q=q, fields=\'files(id)\', pageSize=10,\n                              supportsAllDrives=True, includeItemsFromAllDrives=True\n                              ).execute().get(\'files\', []):\n        try:\n            svc.files().delete(fileId=f[\'id\'], supportsAllDrives=True).execute()\n        except Exception:\n            pass  # A file we cannot replace is not worth failing the download over\n\ndef _mark_drive_uploaded(dest: str):\n    with _drive_uploaded_lock:\n        _drive_uploaded_paths.add(os.path.normpath(dest))\n\ndef _drive_path_exists(dest: str) -> bool:\n    """True if the path is in Drive — through the mount, or uploaded by the API this\n    session. API uploads are invisible to the mount until drivefs next polls for\n    remote changes, so the mount alone would report a file we just wrote as absent."""\n    if os.path.exists(dest):\n        return True\n    with _drive_uploaded_lock:\n        return os.path.normpath(dest) in _drive_uploaded_paths\n\ndef _drive_api_upload(src: str, dest: str):\n    """Upload src to the Drive path dest over the REST API.\n\n    Returns only once Drive holds the whole file, so the caller can safely delete\n    the local copy and log the download as complete."""\n    from googleapiclient.http import MediaFileUpload\n    rel = os.path.relpath(dest, DRIVE_BASE).replace(os.sep, \'/\')\n    rel_dir, _, name = rel.rpartition(\'/\')\n    folder_id = _drive_folder_id(rel_dir)\n    file_size = os.path.getsize(src)\n    size_mb = file_size / (1024 * 1024)\n    _drive_clear_existing(folder_id, name)\n\n    chunk = DRIVE_API_CHUNK_MB * 1024 * 1024\n    report_interval = 500 * 1024 * 1024\n    last_error = None\n    for attempt in range(1, DRIVE_API_MAX_ATTEMPTS + 1):\n        media = MediaFileUpload(src, chunksize=chunk, resumable=True)\n        request = _drive_service().files().create(\n            body={\'name\': name, \'parents\': [folder_id]},\n            media_body=media, fields=\'id,size\', supportsAllDrives=True)\n        start_time = time.time()\n        last_report = 0\n        if size_mb >= 100:\n            with print_lock:\n                print(f"      📤 Uploading to Drive: {size_mb:.0f} MB..."\n                      + (f" (attempt {attempt})" if attempt > 1 else ""))\n        try:\n            response = None\n            while response is None:\n                if cancel_requested():\n                    raise KeyboardInterrupt("cancelled during Drive upload")\n                status, response = request.next_chunk()\n                if status is None or size_mb < 100:\n                    continue\n                sent = status.resumable_progress\n                if sent - last_report >= report_interval:\n                    elapsed = time.time() - start_time\n                    speed = (sent / (1024 * 1024)) / elapsed if elapsed > 0 else 0\n                    with print_lock:\n                        print(f"         {(sent / file_size) * 100:.0f}% "\n                              f"({sent / (1024*1024):.0f}/{size_mb:.0f} MB) @ {speed:.1f} MB/s")\n                    last_report = sent\n            uploaded = int(response.get(\'size\') or 0)\n            if uploaded and uploaded != file_size:\n                # Drive disagreeing on size means a truncated upload — never accept it,\n                # the caller is about to delete the only complete copy.\n                raise IOError(f"Drive reported {uploaded} bytes, expected {file_size}")\n            if size_mb >= 100:\n                elapsed = time.time() - start_time\n                speed = size_mb / elapsed if elapsed > 0 else 0\n                with print_lock:\n                    print(f"      ✅ Upload complete: {size_mb:.0f} MB in {elapsed:.0f}s ({speed:.1f} MB/s)")\n            return\n        except KeyboardInterrupt:\n            raise\n        except Exception as e:\n            last_error = e\n            if attempt >= DRIVE_API_MAX_ATTEMPTS:\n                break\n            # Clear the partial before retrying so a failed attempt cannot leave a\n            # half-uploaded file sitting under the final name.\n            try:\n                _drive_clear_existing(folder_id, name)\n            except Exception:\n                pass\n            wait = min(60, 5 * (2 ** (attempt - 1)))\n            for _ in range(wait):\n                if cancel_requested():\n                    raise KeyboardInterrupt("cancelled during Drive upload backoff")\n                time.sleep(1)\n        finally:\n            try:\n                media.stream().close()\n            except Exception:\n                pass\n    raise last_error if last_error else IOError("Drive upload failed")\n\ndef move_with_progress(src: str, dest: str):\n    """Move a file to Drive, tracking the transfer for the disk-space guard —\n    paused downloads keep waiting while any move is in flight, since each\n    completed move frees that file\'s local space all at once.\n\n    Completed transfers also feed the end-of-batch throughput summary, which is\n    what says whether the current mover count is buying anything. Both are only\n    truthful on the API path: a write to the mount returns before Drive has the\n    bytes, so it neither frees the space nor measures the upload."""\n    global _moves_in_flight\n    with _disk_guard_lock:\n        _moves_in_flight += 1\n    started = time.time()\n    try:\n        copied = _move_with_progress_impl(src, dest)\n    finally:\n        with _disk_guard_lock:\n            _moves_in_flight -= 1\n    # Only reached on success — a raising move records nothing.\n    if copied:\n        _record_drive_transfer(copied, started, time.time())\n\ndef _move_with_progress_impl(src: str, dest: str):\n    """Move a file, with progress output for large cross-filesystem transfers.\n    \n    When moving across filesystems (e.g. local disk → Google Drive FUSE),\n    shutil.move does a full copy+delete. This wrapper uses buffered copy\n    with periodic progress prints so the user can see the transfer happening.\n    For same-filesystem moves, falls back to os.rename (instant).\n\n    Returns the bytes actually pushed to Drive — 0 for a rename or a small-file\n    shutil.move, so the batch throughput summary only counts real transfers.\n    """\n    # Prefer the Drive API. A write through the mount only reaches a local\n    # write-back cache — see the DRIVE API UPLOAD notes above — so it returns long\n    # before Drive has the bytes. The API upload returns when the file is really\n    # there, which is what makes the os.remove(src) below safe.\n    if _use_drive_api() and dest.startswith(DRIVE_BASE):\n        try:\n            api_bytes = os.path.getsize(src)\n            _drive_api_upload(src, dest)\n            _mark_drive_uploaded(dest)\n            os.remove(src)\n            return api_bytes\n        except KeyboardInterrupt:\n            raise\n        except Exception as e:\n            _warn_api_fallback(e)  # Fall through to the mount rather than fail the file\n    # Reached only when the API is off or its upload failed, so the destination\n    # directory may not exist yet — _ensure_dest_dir deliberately skips creating it\n    # while the API owns folder creation.\n    os.makedirs(os.path.dirname(dest), exist_ok=True)\n    try:\n        # Try rename first (instant for same filesystem)\n        os.rename(src, dest)\n        return 0\n    except OSError:\n        pass  # Different filesystems — need to copy+delete\n    \n    file_size = os.path.getsize(src)\n    size_mb = file_size / (1024 * 1024)\n    \n    # For small files (<100MB), just use shutil.move silently\n    if size_mb < 100:\n        shutil.move(src, dest)\n        return 0\n    \n    # Large file: buffered copy with progress\n    chunk_size = 8 * 1024 * 1024  # 8MB chunks\n    copied = 0\n    last_report = 0\n    report_interval = 500 * 1024 * 1024  # Print every 500MB\n    start_time = time.time()\n    \n    with print_lock:\n        print(f"      📤 Transferring to Drive: {size_mb:.0f} MB...")\n    \n    try:\n        with open(src, \'rb\') as fsrc, open(dest, \'wb\') as fdst:\n            while True:\n                buf = fsrc.read(chunk_size)\n                if not buf:\n                    break\n                fdst.write(buf)\n                copied += len(buf)\n                \n                if copied - last_report >= report_interval:\n                    elapsed = time.time() - start_time\n                    speed = (copied / (1024 * 1024)) / elapsed if elapsed > 0 else 0\n                    pct = (copied / file_size) * 100\n                    with print_lock:\n                        print(f"         {pct:.0f}% ({copied / (1024*1024):.0f}/{size_mb:.0f} MB) @ {speed:.1f} MB/s")\n                    last_report = copied\n        \n        elapsed = time.time() - start_time\n        speed = size_mb / elapsed if elapsed > 0 else 0\n        with print_lock:\n            print(f"      ✅ Transfer complete: {size_mb:.0f} MB in {elapsed:.0f}s ({speed:.1f} MB/s)")\n        \n        os.remove(src)\n        return file_size\n    except Exception as e:\n        # If copy failed, clean up partial destination and re-raise\n        if os.path.exists(dest):\n            try:\n                os.remove(dest)\n            except Exception:\n                pass\n        raise\n\ndef handle_file_processing(file_path, source="generic", relative_path=None, torrent_name=None):\n    if not file_path or not os.path.exists(file_path): return\n    filename = os.path.basename(file_path)\n    _, ext = os.path.splitext(filename)\n\n    if ext not in [\'.rar\', \'.zip\', \'.7z\']:\n        # Strip a subtitle\'s language tag before routing (so \'Show.EP01.Eng.srt\' matches\n        # its video\'s TMDB/season entry) and re-attach it to the final name afterwards.\n        processing_name, lang = _split_subtitle_lang(filename)\n\n        final_dest, cat = determine_destination_path(processing_name, source,\n                                                      relative_path=relative_path, torrent_name=torrent_name)\n\n        if lang:  # re-attach a Plex-friendly language code before the subtitle extension\n            base, sub_ext = os.path.splitext(final_dest)\n            final_dest = f"{base}.{_normalize_sub_lang(lang)}{sub_ext}"\n\n        if _drive_path_exists(final_dest):\n            # Subtitles are cheap and re-downloading them is an explicit user action - refresh.\n            # Anything else: keep the existing Drive copy, consistent with duplicate skipping.\n            if ext in KEEP_EXTENSIONS:\n                # Guarded: an API-uploaded file counts as present before the mount can see\n                # it, and the uploader clears same-named files server-side anyway.\n                if os.path.exists(final_dest): os.remove(final_dest)\n            else:\n                size_mb = os.path.getsize(file_path) / (1024 * 1024)\n                print(f"   ⏭️  Already in Drive (kept existing): {os.path.basename(final_dest)}")\n                os.remove(file_path)\n                log_download(os.path.basename(final_dest), source, size_mb, final_dest, status="skipped")\n                return\n\n        _ensure_dest_dir(os.path.dirname(final_dest))\n        size_mb = os.path.getsize(file_path) / (1024 * 1024)\n\n        # Auto-extract embedded subs while the video is still on fast local disk —\n        # the library scan tool has to read videos back over the Drive FUSE mount,\n        # which costs a full download\'s worth of I/O per file. Sidecars are named\n        # off final_dest directly (no re-routing: they belong to this video).\n        extracted_subs = []\n        if auto_extract_subs_checkbox.value and not cancel_requested() and ext.lower() in SUB_EXTRACT_VIDEO_EXTS:\n            extracted_subs = extract_embedded_subs(file_path, list(extract_sub_langs.value),\n                                                   any_fallback=extract_any_track_checkbox.value, quiet=True)\n        local_base = os.path.splitext(file_path)[0]\n\n        move_with_progress(file_path, final_dest)\n        print(f"   ✨ Moved to {cat}: {os.path.basename(final_dest)}")\n        log_download(os.path.basename(final_dest), source, size_mb, final_dest)\n\n        dest_base = os.path.splitext(final_dest)[0]\n        for sub_path in extracted_subs:\n            sub_dest = dest_base + sub_path[len(local_base):]  # carries the \'.en.srt\' tail\n            try:\n                sub_size_mb = os.path.getsize(sub_path) / (1024 * 1024)\n                if os.path.exists(sub_dest): os.remove(sub_dest)  # refresh, like downloaded subs\n                move_with_progress(sub_path, sub_dest)\n                print(f"   📑 Extracted sub: {os.path.basename(sub_dest)}")\n                log_download(os.path.basename(sub_dest), source, sub_size_mb, sub_dest)\n            except Exception as e:\n                print(f"   ⚠️ Could not move extracted sub {os.path.basename(sub_path)}: {str(e)[:80]}")\n        return\n\n    print(f"   📦 Archive Detected: {filename}")\n    # Extraction needs roughly the archive\'s size again on local disk\n    archive_gb = os.path.getsize(file_path) / (1024 ** 3)\n    if not _wait_for_disk_space(archive_gb + DISK_FLOOR_GB, label=f"extracting {filename[:40]}"):\n        raise OSError(f"Not enough local disk space to extract {filename} (archive kept for Retry)")\n    # Per-archive temp dir. This used to be one shared path that was rmtree\'d on entry,\n    # so two archives extracting at once (the download pool runs several workers) deleted\n    # each other\'s files mid-extraction and silently lost one of them.\n    extract_temp = os.path.join(COLAB_ROOT, f"temp_extract_{uuid4().hex[:8]}")\n    if os.path.exists(extract_temp): shutil.rmtree(extract_temp)\n    os.makedirs(extract_temp)\n\n    archive_files = []\n    try:\n        if \'.rar\' in ext:\n            res = subprocess.run([\'unrar\', \'lb\', file_path], capture_output=True, text=True)\n            if res.returncode == 0: archive_files = res.stdout.strip().splitlines()\n        else:\n            res = subprocess.run([\'7z\', \'l\', \'-ba\', \'-slt\', file_path], capture_output=True, text=True)\n            if res.returncode == 0:\n                for line in res.stdout.splitlines():\n                    if line.strip().startswith(\'Path = \'): archive_files.append(line.split(\' = \', 1)[1])\n    except Exception as e:\n        print(f"   ❌ Failed to read archive: {str(e)[:80]}")\n        return\n\n    # Filter the listing to extractable, safe entries\n    wanted = []\n    excluded = False\n    for f_path in archive_files:\n        if f_path.endswith((\'/\', \'\\\\\')) or \'__MACOSX\' in f_path:\n            excluded = True\n            continue\n        if not is_safe_path(extract_temp, f_path):\n            print(f"      ⚠️ SKIPPING UNSAFE PATH: {f_path}")\n            excluded = True\n            continue\n        wanted.append(f_path)\n\n    if not wanted:\n        print("   ⚠️ No extractable files found in archive")\n        shutil.rmtree(extract_temp, ignore_errors=True)\n        return\n\n    # Extract in a single pass - per-file extraction re-decompresses solid RAR archives\n    # from the start each time (O(N^2)). Pass an explicit file list only when some\n    # entries were excluded above; otherwise extract everything.\n    total_files = len(wanted)\n    print(f"   📄 Extracting {total_files} files...")\n    with progress_lock:\n        progress_bar.description = f"Extracting {total_files} files..."\n        progress_bar.value = 0\n    if \'.rar\' in ext:\n        cmd = [\'unrar\', \'x\', \'-o+\', file_path] + (wanted if excluded else []) + [extract_temp + \'/\']\n    else:\n        cmd = [\'7z\', \'x\', \'-y\', file_path, f\'-o{extract_temp}\'] + (wanted if excluded else [])\n    res = subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)\n    if res.returncode != 0:\n        print(f"      ⚠️ Extractor exited with code {res.returncode} - some files may be missing")\n\n    processed_count = 0\n    for idx, f_path in enumerate(wanted, 1):\n        with progress_lock:\n            progress_bar.description = f"Organise: {idx}/{total_files}"\n            progress_bar.value = (idx / total_files) * 100\n\n        extracted_full = os.path.join(extract_temp, f_path)\n        if not os.path.exists(extracted_full) or os.path.isdir(extracted_full):\n            continue\n        if os.path.getsize(extracted_full) < MIN_FILE_SIZE_MB * 1024 * 1024 and not f_path.endswith(tuple(KEEP_EXTENSIONS)):\n            os.remove(extracted_full)\n            continue\n        # Same subtitle-language handling as handle_file_processing: route on the\n        # language-stripped name, then re-attach a Plex-friendly code before the ext.\n        processing_name, lang = _split_subtitle_lang(f_path)\n        final_dest, cat = determine_destination_path(processing_name, source)\n        if lang:\n            base, sub_ext = os.path.splitext(final_dest)\n            final_dest = f"{base}.{_normalize_sub_lang(lang)}{sub_ext}"\n\n        if _drive_path_exists(final_dest):\n            print(f"      -> ⚠️ Duplicate in Drive (kept existing): {os.path.basename(final_dest)}")\n            os.remove(extracted_full)\n            continue\n\n        _ensure_dest_dir(os.path.dirname(final_dest))\n        size_mb = os.path.getsize(extracted_full) / (1024 * 1024)\n        move_with_progress(extracted_full, final_dest)\n        print(f"      [{idx}/{total_files}] -> {os.path.basename(final_dest)}")\n        log_download(os.path.basename(final_dest), source, size_mb, final_dest)\n        processed_count += 1\n\n    shutil.rmtree(extract_temp, ignore_errors=True)\n    if processed_count == 0 and res.returncode != 0:\n        print(f"   ❌ Extraction failed - archive kept for manual retry: {file_path}")\n    else:\n        os.remove(file_path)\n        print(f"   ✅ Extraction complete: {processed_count} files processed")\n    with progress_lock:\n        progress_bar.description = "Idle"\n\ndef get_gofile_session(token: Optional[str]) -> Tuple[requests.Session, dict]:\n    """Create authenticated Gofile session."""\n    s = requests.Session()\n    s.headers.update({\'User-Agent\': \'Mozilla/5.0\'})\n    t = {\'token\': token, \'wt\': GOFILE_WEBSITE_TOKEN}\n    if not token:\n        try: \n            r = s.post("https://api.gofile.io/accounts", json={}, timeout=REQUEST_TIMEOUT)\n            t[\'token\'] = r.json()[\'data\'][\'token\'] if r.status_code == 200 else None\n        except Exception: pass\n    return s, t\n\ndef resolve_gofile(url, s, t) -> List[Tuple[str, str]]:\n    try:\n        match = re.search(r\'gofile\\.io/d/([a-zA-Z0-9]+)\', url)\n        if not match: return []\n        r = s.get(f"https://api.gofile.io/contents/{match.group(1)}", \n                  params={\'wt\': t[\'wt\']}, headers={\'Authorization\': f"Bearer {t[\'token\']}"}, timeout=30)\n        data = r.json()\n        if data[\'status\'] == \'ok\': return [(c[\'link\'], c[\'name\']) for c in data[\'data\'][\'children\'].values() if c.get(\'link\')]\n        else:\n            print(f"   ❌ Gofile Error: {data.get(\'status\', \'unknown\')} - Check if link is valid or requires authentication")\n    except Exception as e:\n        print(f"   ❌ Gofile API Error: {str(e)[:80]}")\n    return [] \n\ndef resolve_pixeldrain(url, s) -> List[Tuple[str, str]]:\n    """Resolve Pixeldrain URL to direct download link."""\n    try:\n        match = re.search(r\'pixeldrain\\.com/u/([a-zA-Z0-9]+)\', url)\n        if not match:\n            print(f"   ⚠️ Pixeldrain: Could not extract file ID from URL")\n            return []\n        fid = match.group(1)\n        name = s.get(f"https://pixeldrain.com/api/file/{fid}/info", timeout=REQUEST_TIMEOUT).json().get(\'name\', f"pixeldrain_{fid}")\n        return [(f"https://pixeldrain.com/api/file/{fid}?download", sanitize_filename(name))]\n    except Exception as e:\n        print(f"   ❌ Pixeldrain Error: {str(e)[:80]} - File may not exist or be private")\n    return []\n\ndef process_rd_link(link, key) -> bool:\n    """Download an RD link or magnet. Returns True on success (or duplicate skip),\n    False on any error/timeout/failed download so the caller can mark it for retry."""\n    h = {"Authorization": f"Bearer {key}"}\n    if "magnet:?" in link:\n        print("   🧲 Resolving Magnet...")\n        try:\n            r = requests.post("https://api.real-debrid.com/rest/1.0/torrents/addMagnet", data={"magnet": link}, headers=h, timeout=30).json()\n            if \'error\' in r:\n                print(f"   ❌ RD Magnet Error: {r.get(\'error\', \'Unknown\')} - Check token or magnet validity")\n                return False\n            requests.post(f"https://api.real-debrid.com/rest/1.0/torrents/selectFiles/{r[\'id\']}", data={"files": "all"}, headers=h, timeout=30)\n\n            # Poll for torrent status with progress updates\n            with progress_lock:\n                progress_bar.value = 0\n                progress_bar.bar_style = \'info\'\n                progress_bar.description = "RD: Caching..."\n\n            for poll_count in range(60):  # 2 minutes max (60 * 2s)\n                i = requests.get(f"https://api.real-debrid.com/rest/1.0/torrents/info/{r[\'id\']}", headers=h, timeout=30).json()\n\n                # Update progress bar with RD caching progress\n                progress_pct = i.get(\'progress\', 0)\n                status = i.get(\'status\', \'unknown\')\n                _update_torrent_progress("RD", status, progress_pct,\n                                         state_labels={\'waiting_files_selection\': \'Selecting files...\'})\n\n                if status == \'downloaded\':\n                    with progress_lock:\n                        progress_bar.value = 100\n                        progress_bar.description = "RD: Cached ✓"\n                    print(f"   ✅ Torrent cached - {len(i.get(\'links\', []))} file(s)")\n                    all_ok = True\n                    for idx, l in enumerate(i[\'links\'], 1):\n                        print(f"   📥 Downloading file {idx}/{len(i[\'links\'])}...")\n                        if not process_rd_link(l, key):\n                            all_ok = False  # One failed file fails the magnet; retry re-skips the done ones\n                    return all_ok\n                time.sleep(2)\n\n            print("   ❌ RD Timeout - Torrent took too long to cache (2 min limit)")\n        except Exception as e:\n            print(f"   ❌ RD Magnet Error: {str(e)[:80]}")\n        finally:\n            _reset_progress_bar()\n        return False\n\n    # Regular RD link (unrestrict and download)\n    try:\n        d = requests.post("https://api.real-debrid.com/rest/1.0/unrestrict/link", data={"link": link}, headers=h, timeout=30).json()\n        if \'error\' in d:\n            print(f"   ❌ RD Unrestrict Error: {d.get(\'error\', \'Unknown\')} - Check if link is supported")\n            return False\n\n        # Generate a task_id for progress tracking\n        task_id = f"rd_{str(uuid4())[:8]}"\n        with progress_lock:\n            progress_bar.value = 0\n            progress_bar.bar_style = \'info\'\n\n        f = download_with_aria2(d[\'download\'], d[\'filename\'], COLAB_ROOT, task_id=task_id)\n        if f is DUPLICATE_SKIP:\n            return True  # Already in Drive\n        if f:\n            handle_file_processing(f)\n            return True\n        return False\n    except Exception as e:\n        print(f"   ❌ RD Error: {str(e)[:80]}")\n        return False\n\ndef resolve_rd_link(url: str, rd_key: str) -> List[Tuple[str, str]]:\n    """Unrestrict a Real-Debrid link and return (download_url, filename) tuple."""\n    if not rd_key:\n        print(f"   ❌ RD Token required for: {url}")\n        return []\n    try:\n        h = {"Authorization": f"Bearer {rd_key}"}\n        d = requests.post("https://api.real-debrid.com/rest/1.0/unrestrict/link", \n                         data={"link": url}, headers=h, timeout=30).json()\n        if \'error\' in d:\n            print(f"   ❌ RD Unrestrict Error: {d.get(\'error\', \'Unknown\')}")\n            return []\n        return [(d[\'download\'], d[\'filename\'])]\n    except Exception as e:\n        print(f"   ❌ RD Resolve Error: {str(e)[:80]}")\n        return []\n\ndef resolve_magnet_files(magnet_url: str, rd_key: str) -> List[DownloadTask]:\n    """\n    Add magnet to RD and wait for file list to be available.\n    Returns list of DownloadTask objects for each file in the torrent.\n    """\n    if not rd_key:\n        print(f"   ❌ RD Token required for magnets")\n        return []\n    \n    h = {"Authorization": f"Bearer {rd_key}"}\n    \n    try:\n        print("   🧲 Adding magnet to Real-Debrid...")\n        \n        # Add magnet to RD (with retry on rate-limit)\n        global _rd_magnet_delay\n        r = None\n        for attempt in range(4):  # Up to 4 attempts (0, 1, 2, 3)\n            r = requests.post("https://api.real-debrid.com/rest/1.0/torrents/addMagnet", \n                             data={"magnet": magnet_url}, headers=h, timeout=30).json()\n            if \'error\' not in r:\n                break  # Success\n            # Check if rate-limited — retry with exponential backoff\n            err_msg = r.get(\'error\', \'\').lower().replace(\'_\', \' \')\n            if attempt < 3 and any(kw in err_msg for kw in [\'too many\', \'limit\', \'flood\', \'action already done\']):\n                _rd_magnet_delay = 2  # Enable pacing for all subsequent magnets\n                wait = 5 * (2 ** attempt)  # 5s, 10s, 20s\n                print(f"   ⏳ RD rate limit hit, retrying in {wait}s (attempt {attempt + 2}/4)...")\n                time.sleep(wait)\n            else:\n                print(f"   ❌ RD Magnet Error: {r.get(\'error\', \'Unknown\')}")\n                return []\n        \n        torrent_id = r[\'id\']\n        \n        # Wait for file list to become available (magnet_conversion -> waiting_files_selection)\n        print("   ⏳ Waiting for torrent metadata...")\n        for _ in range(30):  # 60 seconds max\n            info = requests.get(f"https://api.real-debrid.com/rest/1.0/torrents/info/{torrent_id}", \n                               headers=h, timeout=30).json()\n            status = info.get(\'status\', \'\')\n            \n            if status == \'waiting_files_selection\':\n                # Files are available for selection\n                files = info.get(\'files\', [])\n                if not files:\n                    print("   ⚠️ No files found in torrent")\n                    return []\n                \n                torrent_name = info.get(\'filename\', \'Unknown Torrent\')\n                print(f"   ✅ Found {len(files)} file(s) in: {torrent_name[:50]}")\n                \n                # Create DownloadTask for each file\n                tasks = []\n                for f in files:\n                    file_id = f.get(\'id\')\n                    file_path = f.get(\'path\', \'\').lstrip(\'/\')\n                    file_name = os.path.basename(file_path) if file_path else f"file_{file_id}"\n                    task = _make_torrent_file_task(magnet_url, "magnet_file", torrent_id,\n                                                   file_id, file_name, f.get(\'bytes\', 0))\n                    if task:\n                        tasks.append(task)\n\n                return tasks\n            \n            elif status == \'downloaded\':\n                # Already cached - get links directly\n                links = info.get(\'links\', [])\n                torrent_name = info.get(\'filename\', \'Unknown Torrent\')\n                print(f"   ✅ Already cached: {torrent_name[:50]} ({len(links)} files)")\n                \n                tasks = []\n                for link_idx, link in enumerate(links):\n                    # Rate limit: delay between unrestrict calls to avoid RD fair-use blocks\n                    if link_idx > 0:\n                        time.sleep(2)\n                    # Unrestrict to get filename\n                    try:\n                        d = requests.post("https://api.real-debrid.com/rest/1.0/unrestrict/link",\n                                         data={"link": link}, headers=h, timeout=30).json()\n                        if \'download\' in d:\n                            tasks.append(DownloadTask(\n                                url=d[\'download\'],\n                                filename=d.get(\'filename\', \'unknown\'),\n                                source="magnet",\n                                link_type="rd",  # Already unrestricted, can download directly\n                                original_url=link,\n                            ))\n                    except Exception:\n                        pass\n                \n                return tasks\n            \n            elif status == \'magnet_error\':\n                print(f"   ❌ Magnet error - invalid or dead torrent")\n                # Clean up the failed torrent\n                try:\n                    requests.delete(f"https://api.real-debrid.com/rest/1.0/torrents/delete/{torrent_id}", \n                                   headers=h, timeout=30)\n                except Exception:\n                    pass\n                return []\n            \n            time.sleep(2)\n        \n        print("   ❌ Timeout waiting for torrent metadata (60s)")\n        return []\n        \n    except Exception as e:\n        print(f"   ❌ RD Magnet Resolve Error: {str(e)[:80]}")\n        return []\n\ndef process_magnet_file_tasks(tasks: List[DownloadTask], rd_key: str) -> int:\n    """\n    Process magnet_file tasks - select files in RD, wait for cache, download.\n    Returns number of successfully downloaded files.\n    """\n    if not tasks or not rd_key:\n        return 0\n    \n    h = {"Authorization": f"Bearer {rd_key}"}\n    success_count = 0\n\n    # Process each torrent (tasks grouped by torrent_id from original_url)\n    for torrent_id, file_list in _group_tasks_by_torrent(tasks).items():\n        # Default to failed; upgraded to done/skipped per file below. This way a\n        # torrent error or cache timeout leaves the tasks failed (retryable on resume).\n        for _fid, task in file_list:\n            task.status = "failed"\n            task.error = "Not downloaded"\n        try:\n            file_ids = [fid for fid, _ in file_list]\n            print(f"\\n   🧲 Processing torrent with {len(file_ids)} selected file(s)...")\n            \n            # Select only the chosen files\n            file_selection = \',\'.join(file_ids)\n            select_resp = requests.post(\n                f"https://api.real-debrid.com/rest/1.0/torrents/selectFiles/{torrent_id}",\n                data={"files": file_selection}, headers=h, timeout=30\n            )\n            \n            if select_resp.status_code != 204:\n                print(f"   ⚠️ File selection may have failed (status {select_resp.status_code})")\n            \n            # Wait for caching with progress\n            with progress_lock:\n                progress_bar.value = 0\n                progress_bar.bar_style = \'info\'\n                progress_bar.description = "RD: Caching..."\n            \n            for poll_count in range(120):  # 4 minutes max\n                if _cancel_requested:\n                    break\n                info = requests.get(\n                    f"https://api.real-debrid.com/rest/1.0/torrents/info/{torrent_id}",\n                    headers=h, timeout=30\n                ).json()\n                \n                status = info.get(\'status\', \'\')\n                progress_pct = info.get(\'progress\', 0)\n\n                _update_torrent_progress("RD", status, progress_pct)\n\n                if status == \'downloaded\':\n                    links = info.get(\'links\', [])\n                    print(f"   ✅ Cached! Downloading {len(links)} file(s)...")\n                    \n                    with progress_lock:\n                        progress_bar.value = 100\n                        progress_bar.description = "RD: Cached ✓"\n                    \n                    # Download each link. RD returns links in selected-file order,\n                    # so links[k] maps to file_list[k].\n                    for idx, link in enumerate(links, 1):\n                        if _cancel_requested:\n                            break\n                        # Rate limit: delay between unrestrict calls to avoid RD fair-use blocks\n                        if idx > 1:\n                            time.sleep(2)\n                        task = file_list[idx-1][1] if idx-1 < len(file_list) else None\n                        try:\n                            d = requests.post(\n                                "https://api.real-debrid.com/rest/1.0/unrestrict/link",\n                                data={"link": link}, headers=h, timeout=30\n                            ).json()\n\n                            if \'download\' in d:\n                                print(f"   📥 [{idx}/{len(links)}] {d.get(\'filename\', \'file\')[:50]}")\n                                task_id = task.id if task else f"rd_{str(uuid4())[:8]}"\n                                f = download_with_aria2(d[\'download\'], d[\'filename\'], COLAB_ROOT, task_id=task_id)\n                                if f is DUPLICATE_SKIP:\n                                    if task: task.status, task.error = "skipped", None\n                                    success_count += 1\n                                elif f:\n                                    handle_file_processing(f, source="magnet")\n                                    if task: task.status, task.error = "done", None\n                                    success_count += 1\n                                else:\n                                    if task: task.error = "Download failed"\n                            else:\n                                if task: task.error = f"Unrestrict error: {d.get(\'error\', \'unknown\')}"\n                        except Exception as e:\n                            print(f"   ❌ Failed to download: {str(e)[:60]}")\n                            if task: task.error = str(e)[:100]\n\n                    break\n                \n                elif status in [\'magnet_error\', \'error\', \'dead\']:\n                    print(f"   ❌ Torrent error: {status}")\n                    break\n                \n                time.sleep(2)\n            else:\n                print("   ❌ Timeout waiting for torrent to cache (4 min)")\n                \n        except Exception as e:\n            print(f"   ❌ Error processing torrent: {str(e)[:80]}")\n        finally:\n            _reset_progress_bar()\n\n    return success_count\n\n# --- DEBRID SERVICE HELPERS ---\n# Shared building blocks for the Real-Debrid and TorBox flows. Both services follow\n# the same shape (add magnet → poll until cached → request per-file links), so the\n# structural pieces live here and only the API specifics stay per-service.\n\ndef _reset_progress_bar():\n    """Reset the shared progress bar to idle (used in debrid finally blocks)."""\n    with progress_lock:\n        progress_bar.description = "Idle"\n        progress_bar.bar_style = \'info\'\n\ndef _update_torrent_progress(prefix: str, status: str, progress_pct: float,\n                             downloading_states=(\'downloading\',), state_labels=None):\n    """Update the shared progress bar while a debrid service caches a torrent."""\n    with progress_lock:\n        progress_bar.value = progress_pct\n        if status in downloading_states:\n            progress_bar.description = f"{prefix}: {int(progress_pct)}% cached"\n        elif state_labels and status in state_labels:\n            progress_bar.description = f"{prefix}: {state_labels[status]}"\n        elif status == \'queued\':\n            progress_bar.description = f"{prefix}: Queued..."\n        else:\n            progress_bar.description = f"{prefix}: {status}"\n\ndef _make_torrent_file_task(magnet_url: str, link_type: str, torrent_id, file_id,\n                            file_name: str, size_bytes: int,\n                            relative_dir: str = \'\', torrent_name: str = \'\') -> Optional[DownloadTask]:\n    """Build a queue task for one file inside a debrid torrent.\n    Returns None for tiny files (samples/NFOs) unless they are subtitles.\n    relative_dir is the file\'s subfolder *within the torrent* (e.g. "Season 1"),\n    \'\' if the file sits at the torrent root — kept so folder structure can be\n    reconstructed on save when auto-organise is off (see determine_destination_path)."""\n    size_mb = size_bytes / (1024 * 1024)\n    if size_mb < 1 and not file_name.endswith(tuple(KEEP_EXTENSIONS)):\n        return None\n    return DownloadTask(\n        url=magnet_url,\n        filename=f"{file_name} ({size_mb:.1f} MB)" if size_mb > 0 else file_name,\n        source="magnet",\n        link_type=link_type,\n        original_url=f"{torrent_id}:{file_id}",  # torrent_id:file_id for later selection\n        relative_path=relative_dir,\n        torrent_name=torrent_name,\n    )\n\ndef _group_tasks_by_torrent(tasks: List[DownloadTask]) -> Dict[str, List[Tuple[str, DownloadTask]]]:\n    """Group magnet-file tasks by torrent id (original_url stores \'torrent_id:file_id\')."""\n    grouped: Dict[str, List[Tuple[str, DownloadTask]]] = {}\n    for task in tasks:\n        if task.original_url and \':\' in task.original_url:\n            torrent_id, file_id = task.original_url.split(\':\', 1)\n            grouped.setdefault(torrent_id, []).append((file_id, task))\n    return grouped\n\ndef get_active_debrid() -> Tuple[str, str, str]:\n    """Return (service_name, rd_key, tb_key) based on debrid toggle selection.\n    service_name: \'rd\', \'tb\', or \'none\'\n    rd_key: Real-Debrid API key (empty if TorBox selected or None)\n    tb_key: TorBox API key (empty if Real-Debrid selected or None)\n    """\n    service = debrid_service_toggle.value\n    if service == \'TorBox\':\n        return (\'tb\', \'\', token_tb.value.strip())\n    elif service == \'Real-Debrid\':\n        return (\'rd\', token_rd.value.strip(), \'\')\n    return (\'none\', \'\', \'\')\n\n# --- TORBOX API FUNCTIONS ---\ndef _get_tb_headers(tb_key: str) -> dict:\n    """Return authorization headers for TorBox API."""\n    return {"Authorization": f"Bearer {tb_key}"}\n\ndef _tb_fetch_item(endpoint: str, item_id, tb_key: str) -> Optional[dict]:\n    """Fetch one entry from a TorBox mylist endpoint (\'torrents\' or \'webdl\').\n    Returns the item dict, or None if not available yet (caller keeps polling)."""\n    try:\n        r = requests.get(f"{TORBOX_API_BASE}/{endpoint}/mylist",\n                         params={"id": item_id}, headers=_get_tb_headers(tb_key), timeout=30)\n        data = r.json()\n        if not data.get(\'success\'):\n            return None\n        item = data.get(\'data\')\n        if isinstance(item, list):\n            item = next((x for x in item if x.get(\'id\') == item_id), None)\n        return item\n    except Exception:\n        return None  # Transient poll error — caller retries until its poll limit\n\ndef _tb_progress_pct(item: dict) -> float:\n    """TorBox reports progress as 0-1 or 0-100 depending on endpoint — normalise to 0-100."""\n    progress = item.get(\'progress\', 0)\n    return progress * 100 if progress <= 1 else progress\n\ndef _tb_extract_download_url(dl_data: dict) -> str:\n    """Extract the download URL from a TorBox requestdl response (\'\' if missing)."""\n    if not (dl_data.get(\'success\') and dl_data.get(\'data\')):\n        return \'\'\n    data = dl_data[\'data\']\n    if isinstance(data, str):\n        return data\n    return data.get(\'download_url\', data.get(\'url\', \'\'))\n\n# torbox.app/download share links carry ?type=<kind> — map to the API path segment\n_TB_ENDPOINT_BY_TYPE = {\'torrents\': \'torrents\', \'torrent\': \'torrents\',\n                        \'usenet\': \'usenet\', \'webdl\': \'webdl\', \'web\': \'webdl\'}\n# requestdl names its item-id parameter differently per endpoint\n_TB_REQUESTDL_ID_PARAM = {\'torrents\': \'torrent_id\', \'usenet\': \'usenet_id\', \'webdl\': \'web_id\'}\n\ndef _tb_err_is_throttle(err: str) -> bool:\n    """True when a _tb_requestdl error smells like rate limiting or an overloaded\n    edge rather than a per-file refusal — used to stop batch link prefetching\n    early instead of burning more of the rate budget."""\n    e = err.lower()\n    return \'http\' in e or \'rate\' in e or \'limit\' in e or \'many requests\' in e\n\ndef _tb_requestdl(endpoint: str, id_param: str, item_id, file_id, tb_key: str,\n                  attempts: int = 4) -> Tuple[str, str]:\n    """Request one file\'s download URL from TorBox: (url, \'\') or (\'\', error).\n\n    requestdl is TorBox\'s most rate-limited endpoint, and throttled or overloaded\n    calls come back as 429/5xx with empty or HTML bodies — parsing those with a\n    bare .json() raised \'Expecting value: line 1 column 1 (char 0)\' and failed\n    the file outright. Treat them (and rate-limit refusals inside valid JSON) as\n    transient: back off — honouring Retry-After when TorBox sends it — and retry;\n    only a clean JSON refusal (bad id, expired token) fails immediately."""\n    try:\n        item_id, file_id = int(item_id), int(file_id)\n    except (TypeError, ValueError):\n        pass  # non-numeric ids go through as-is\n    dl_params = {"token": tb_key, id_param: item_id, "file_id": file_id}\n    if id_param == \'web_id\':\n        dl_params["web_download_id"] = item_id  # older param name, harmless\n    err = \'Could not get download URL\'\n    retry_after = 0.0\n    for attempt in range(attempts):\n        if attempt:\n            delay = max(retry_after, 5 * (2 ** (attempt - 1)))  # 5s, 10s, 20s\n            print(f"      ⏳ TorBox throttled — retrying in {delay:.0f}s ({err[:60]})")\n            for _ in range(int(delay)):\n                if cancel_requested():\n                    return \'\', \'Cancelled by user\'\n                time.sleep(1)\n        try:\n            dl_r = requests.get(f"{TORBOX_API_BASE}/{endpoint}/requestdl",\n                                params=dl_params, headers=_get_tb_headers(tb_key), timeout=30)\n        except Exception as e:\n            err = str(e)[:100]\n            retry_after = 0.0\n            continue\n        try:\n            retry_after = min(float(dl_r.headers.get(\'Retry-After\', 0)), 60.0)\n        except (TypeError, ValueError):\n            retry_after = 0.0\n        try:\n            dl_data = dl_r.json()\n        except ValueError:\n            body = (dl_r.text or \'\').strip()\n            err = f"TorBox HTTP {dl_r.status_code}: {\'non-JSON response\' if body else \'empty response\'}"\n            continue  # throttle or edge hiccup — worth retrying\n        download_url = _tb_extract_download_url(dl_data)\n        if download_url:\n            return download_url, \'\'\n        err = str(dl_data.get(\'detail\') or dl_data.get(\'error\') or err)[:120]\n        if dl_r.status_code < 429 and not _tb_err_is_throttle(err):\n            return \'\', err  # genuine refusal — retrying won\'t change the answer\n    return \'\', err\n\ndef resolve_tb_folder_files(url: str, tb_key: str) -> List[DownloadTask]:\n    """Resolve a torbox.app/download?id=X&type=Y share link (the site\'s\n    \'Copy JDownloader Folder Links\' button) into per-file queue tasks.\n    The id references an item already in the TorBox account, so this only\n    reads mylist — nothing new is queued on TorBox. Tasks carry the real\n    filenames (so episode/TMDB matching works) and reuse the tb_magnet_file\n    download pipeline.\n    """\n    if not tb_key:\n        print(f"   ❌ TorBox Token required for: {url[:60]}")\n        return []\n    params = parse_qs(urlparse(url).query)\n    item_id = (params.get(\'id\') or [\'\'])[0]\n    type_param = (params.get(\'type\') or [\'torrents\'])[0].lower()\n    endpoint = _TB_ENDPOINT_BY_TYPE.get(type_param)\n    if not item_id.isdigit() or not endpoint:\n        print(f"   ❌ Unrecognised TorBox link: {url[:70]}")\n        return []\n\n    item = None\n    for attempt in range(3):  # mylist can hiccup — brief retry\n        item = _tb_fetch_item(endpoint, int(item_id), tb_key)\n        if item:\n            break\n        time.sleep(2)\n    if not item:\n        print(f"   ❌ TorBox: item {item_id} not found in your {endpoint} list (check account/API key)")\n        return []\n\n    name = item.get(\'name\', f\'torbox_{item_id}\')\n    files = item.get(\'files\') or []\n    if not files:\n        print(f"   ⚠️ No files listed for: {name[:50]} (still caching?)")\n        return []\n    print(f"   ✅ Found {len(files)} file(s) in: {name[:50]}")\n\n    # Torrents keep the bare id so original_url matches the magnet flow;\n    # usenet/webdl prefix the endpoint so requestdl later hits the right API.\n    group_id = item_id if endpoint == \'torrents\' else f"{endpoint}/{item_id}"\n    tasks = []\n    for f in files:\n        file_id = f.get(\'id\', 0)\n        # TorBox \'name\' is the file\'s path within the torrent; strip the leading folder\n        # (often a release/quality string) so it can\'t pollute show-name detection.\n        raw_name = f.get(\'name\') or f.get(\'short_name\') or f\'file_{file_id}\'\n        file_name = os.path.basename(raw_name.replace(\'\\\\\', \'/\')) or raw_name\n        task = _make_torrent_file_task(url, "tb_magnet_file", group_id,\n                                       file_id, file_name, f.get(\'size\', 0))\n        if task:\n            tasks.append(task)\n    return tasks\n\ndef _resolve_tb_cdn(url: str, session: requests.Session) -> List[Tuple[str, str]]:\n    """TorBox CDN links (store-*.tb-cdn.io/dld/<uuid>) have opaque UUID paths.\n    Read the real filename from Content-Disposition so episode matching works;\n    fall back to the URL basename. The link itself downloads directly."""\n    filename = \'\'\n    try:\n        r = session.head(url, allow_redirects=True, timeout=20)\n        cd = r.headers.get(\'content-disposition\', \'\')\n        if not cd:  # some CDNs only send headers on GET\n            r = session.get(url, stream=True, allow_redirects=True, timeout=20)\n            cd = r.headers.get(\'content-disposition\', \'\')\n            r.close()\n        m = re.search(r"filename\\*\\s*=\\s*(?:UTF-8\'\')?([^;]+)", cd, re.I) or \\\n            re.search(r\'filename\\s*=\\s*"?([^";]+)\', cd, re.I)\n        if m:\n            filename = sanitize_filename(unquote(m.group(1).strip().strip(\'"\')))\n    except Exception:\n        pass  # naming is best-effort — the download itself still works\n    if not filename:\n        filename = os.path.basename(unquote(urlparse(url).path)) or "download"\n    return [(url, filename)]\n\ndef resolve_tb_link(url: str, tb_key: str) -> List[Tuple[str, str]]:\n    """Unrestrict a link via TorBox Web Downloads.\n    Creates a web download, polls until completed, then gets download URL.\n    Returns [(download_url, filename)] list matching RD\'s return format.\n    """\n    if not tb_key:\n        print(f"   ❌ TorBox Token required for: {url}")\n        return []\n    \n    h = _get_tb_headers(tb_key)\n    \n    try:\n        # Step 1: Create web download\n        r = requests.post(f"{TORBOX_API_BASE}/webdl/createwebdownload",\n                         json={"url": url}, headers=h, timeout=30)\n        data = r.json()\n        \n        if not data.get(\'success\'):\n            err = data.get(\'detail\', data.get(\'error\', \'Unknown error\'))\n            print(f"   ❌ TorBox Web DL Error: {err}")\n            return []\n        \n        webdl_id = data[\'data\'].get(\'webdownload_id\') or data[\'data\'].get(\'id\')\n        if not webdl_id:\n            print(f"   ❌ TorBox: No download ID returned")\n            return []\n        \n        # Step 2: Poll for completion\n        print(f"   ⏳ TorBox: Processing download...")\n        with progress_lock:\n            progress_bar.value = 0\n            progress_bar.bar_style = \'info\'\n            progress_bar.description = "TB: Processing..."\n        \n        for poll_count in range(90):  # 3 minutes max (90 * 2s)\n            item = _tb_fetch_item(\'webdl\', webdl_id, tb_key)\n            if not item:\n                time.sleep(2)\n                continue\n\n            status = item.get(\'download_state\', item.get(\'status\', \'\'))\n            progress_pct = _tb_progress_pct(item)\n\n            with progress_lock:\n                progress_bar.value = progress_pct\n                progress_bar.description = f"TB: {int(progress_pct)}%"\n\n            if status in [\'completed\', \'cached\', \'done\']:\n                filename = item.get(\'name\', item.get(\'filename\', \'download\'))\n\n                # Step 3: Get download URL (first/main file)\n                files = item.get(\'files\', [])\n                file_id = files[0].get(\'id\', 0) if files else 0\n\n                download_url, dl_err = _tb_requestdl(\'webdl\', \'web_id\', webdl_id, file_id, tb_key)\n                if download_url:\n                    with progress_lock:\n                        progress_bar.value = 100\n                        progress_bar.description = "TB: Ready ✓"\n                    return [(download_url, sanitize_filename(filename))]\n\n                print(f"   ❌ TorBox: {dl_err}")\n                return []\n            \n            elif status in [\'error\', \'failed\']:\n                err = item.get(\'error\', \'Unknown error\')\n                print(f"   ❌ TorBox download error: {err}")\n                return []\n            \n            time.sleep(2)\n        \n        print("   ❌ TorBox: Timeout waiting for download (3 min)")\n        return []\n        \n    except Exception as e:\n        print(f"   ❌ TorBox Resolve Error: {str(e)[:80]}")\n        return []\n    finally:\n        _reset_progress_bar()\n\ndef resolve_tb_magnet_files(magnet_url: str, tb_key: str) -> List[DownloadTask]:\n    """Add magnet to TorBox and wait for file list.\n    Returns list of DownloadTask objects for each file in the torrent.\n    """\n    if not tb_key:\n        print(f"   ❌ TorBox Token required for magnets")\n        return []\n    \n    h = _get_tb_headers(tb_key)\n    \n    try:\n        print("   🧲 Adding magnet to TorBox...")\n        \n        # Add magnet via createtorrent\n        r = requests.post(f"{TORBOX_API_BASE}/torrents/createtorrent",\n                         data={"magnet": magnet_url}, headers=h, timeout=30)\n        data = r.json()\n        \n        if not data.get(\'success\'):\n            err = data.get(\'detail\', data.get(\'error\', \'Unknown error\'))\n            print(f"   ❌ TorBox Magnet Error: {err}")\n            return []\n        \n        torrent_id = data[\'data\'].get(\'torrent_id\') or data[\'data\'].get(\'id\')\n        torrent_name = data[\'data\'].get(\'name\', \'Unknown Torrent\')\n        \n        if not torrent_id:\n            print(f"   ❌ TorBox: No torrent ID returned")\n            return []\n        \n        # Poll for torrent to be ready\n        print(f"   ⏳ Waiting for torrent metadata...")\n        for _ in range(60):  # 2 minutes max\n            item = _tb_fetch_item(\'torrents\', torrent_id, tb_key)\n            if not item:\n                time.sleep(2)\n                continue\n\n            status = item.get(\'download_state\', item.get(\'status\', \'\'))\n\n            if status in [\'completed\', \'cached\', \'done\', \'downloading\', \'uploading\', \'stalled\']:\n                files = item.get(\'files\', [])\n                torrent_name = item.get(\'name\', torrent_name)\n\n                if not files:\n                    print(f"   ⚠️ No files found in torrent")\n                    return []\n\n                print(f"   ✅ Found {len(files)} file(s) in: {torrent_name[:50]}")\n\n                tasks = []\n                for f in files:\n                    file_id = f.get(\'id\', 0)\n                    # TorBox returns the file\'s path relative to the torrent root, e.g.\n                    # "Season 1/S01E01.mkv" or "S01E01.mkv" for a single-file torrent.\n                    # Keep the subfolder (relative_dir) so it can be reconstructed on save;\n                    # file_name itself stays just the leaf name.\n                    raw_name = (f.get(\'name\') or f.get(\'short_name\') or f\'file_{file_id}\').replace(\'\\\\\', \'/\')\n                    relative_dir = os.path.dirname(raw_name)\n                    file_name = os.path.basename(raw_name) or raw_name\n                    task = _make_torrent_file_task(magnet_url, "tb_magnet_file", torrent_id,\n                                                   file_id, file_name, f.get(\'size\', 0),\n                                                   relative_dir=relative_dir, torrent_name=torrent_name)\n                    if task:\n                        tasks.append(task)\n\n                return tasks\n            \n            elif status in [\'error\', \'failed\', \'magnet_error\']:\n                print(f"   ❌ Torrent error: {status}")\n                # Try to clean up\n                try:\n                    requests.post(f"{TORBOX_API_BASE}/torrents/controltorrent",\n                                 json={"torrent_id": torrent_id, "operation": "delete"},\n                                 headers=h, timeout=30)\n                except Exception:\n                    pass\n                return []\n            \n            time.sleep(2)\n        \n        print("   ❌ Timeout waiting for torrent metadata (2 min)")\n        return []\n        \n    except Exception as e:\n        print(f"   ❌ TorBox Magnet Resolve Error: {str(e)[:80]}")\n        return []\n\ndef convert_tb_tasks_to_parallel(tasks: List[DownloadTask], tb_key: str) -> Tuple[List[DownloadTask], List[DownloadTask]]:\n    """Pre-request direct URLs for TorBox files whose item is already cached, so\n    they join the parallel aria2 pool instead of downloading one-by-one.\n    Returns (parallel_ready, still_sequential). Uncached items (fresh magnets)\n    stay sequential — process_tb_magnet_file_tasks polls the caching there.\n    link_type stays \'tb_magnet_file\' so a resumed session re-requests fresh URLs.\n    """\n    if not tasks or not tb_key:\n        return [], tasks or []\n    ready: List[DownloadTask] = []\n    sequential: List[DownloadTask] = []\n    for group_id, file_list in _group_tasks_by_torrent(tasks).items():\n        endpoint, _, item_id = group_id.rpartition(\'/\')\n        endpoint = endpoint or \'torrents\'\n        id_param = _TB_REQUESTDL_ID_PARAM.get(endpoint, \'torrent_id\')\n        item = _tb_fetch_item(endpoint, int(item_id), tb_key)\n        status = (item or {}).get(\'download_state\', (item or {}).get(\'status\', \'\'))\n        if not item or status not in (\'completed\', \'cached\', \'done\'):\n            sequential.extend(t for _fid, t in file_list)\n            continue\n        print(f"   🔗 Fetching {len(file_list)} TorBox direct link(s) for parallel download...")\n        for idx, (file_id, task) in enumerate(file_list):\n            if _cancel_requested:\n                sequential.extend(t for _fid, t in file_list[idx:])\n                break\n            if idx:\n                time.sleep(0.3)  # stay under the TorBox API rate limit\n            try:\n                # One attempt only: on a throttle, stop prefetching instead of\n                # burning more of the rate budget — whatever lands in the\n                # sequential flow gets _tb_requestdl\'s backoff retries there.\n                download_url, dl_err = _tb_requestdl(endpoint, id_param, item_id, file_id, tb_key, attempts=1)\n            except Exception as e:\n                download_url, dl_err = \'\', str(e)[:80]\n            if not download_url and _tb_err_is_throttle(dl_err):\n                print(f"   ⏳ TorBox is throttling link requests — {len(file_list) - idx} file(s) moved to the sequential flow ({dl_err[:60]})")\n                sequential.extend(t for _fid, t in file_list[idx:])\n                break\n            if download_url:\n                task.url = download_url\n                task.filename = _strip_size_suffix(task.filename)\n                ready.append(task)\n            else:\n                sequential.append(task)  # falls back to the sequential flow\'s retry\n    return ready, sequential\n\ndef process_tb_magnet_file_tasks(tasks: List[DownloadTask], tb_key: str) -> int:\n    """Process tb_magnet_file tasks - get download links from TorBox and download.\n    Returns number of successfully downloaded files.\n    """\n    if not tasks or not tb_key:\n        return 0\n    \n    success_count = 0\n\n    # Process each torrent (tasks grouped by torrent_id from original_url)\n    for torrent_id, file_list in _group_tasks_by_torrent(tasks).items():\n        # Folder links may reference usenet/webdl items (\'endpoint/id\');\n        # bare ids are torrents (the magnet flow\'s original format).\n        endpoint, _, item_id = torrent_id.rpartition(\'/\')\n        endpoint = endpoint or \'torrents\'\n        id_param = _TB_REQUESTDL_ID_PARAM.get(endpoint, \'torrent_id\')\n        # Default to failed; upgraded to done/skipped per file below so a torrent\n        # error or cache timeout leaves the tasks failed (retryable on resume).\n        for _fid, task in file_list:\n            task.status = "failed"\n            task.error = "Not downloaded"\n        try:\n            print(f"\\n   🧲 Downloading {len(file_list)} file(s) from TorBox torrent...")\n            \n            # Wait for torrent to be fully cached\n            with progress_lock:\n                progress_bar.value = 0\n                progress_bar.bar_style = \'info\'\n                progress_bar.description = "TB: Caching..."\n            \n            for poll_count in range(120):  # 4 minutes max\n                if _cancel_requested:\n                    break\n                item = _tb_fetch_item(endpoint, int(item_id), tb_key)\n                if not item:\n                    time.sleep(2)\n                    continue\n\n                status = item.get(\'download_state\', item.get(\'status\', \'\'))\n                progress_pct = _tb_progress_pct(item)\n\n                _update_torrent_progress("TB", status, progress_pct,\n                                         downloading_states=(\'downloading\', \'uploading\'))\n\n                if status in [\'completed\', \'cached\', \'done\']:\n                    print(f"   ✅ Torrent ready! Downloading files...")\n                    \n                    with progress_lock:\n                        progress_bar.value = 100\n                        progress_bar.description = "TB: Cached ✓"\n                    \n                    # Download each selected file\n                    for idx, (file_id, task) in enumerate(file_list, 1):\n                        if _cancel_requested:\n                            break\n                        if idx > 1:\n                            time.sleep(1)  # Rate limiting\n                        try:\n                            # Request the file\'s download link — _tb_requestdl retries\n                            # with backoff when TorBox throttles the endpoint\n                            download_url, dl_err = _tb_requestdl(endpoint, id_param, item_id, file_id, tb_key)\n                            if download_url:\n                                # Clean filename (remove size suffix for actual download)\n                                clean_name = _strip_size_suffix(task.filename)\n                                print(f"   📥 [{idx}/{len(file_list)}] {clean_name[:50]}")\n                                f = download_with_aria2(download_url, clean_name, COLAB_ROOT, task_id=task.id)\n                                if f is DUPLICATE_SKIP:\n                                    task.status, task.error = "skipped", None\n                                    success_count += 1\n                                elif f:\n                                    handle_file_processing(f, source="magnet",\n                                                          relative_path=task.relative_path,\n                                                          torrent_name=task.torrent_name)\n                                    task.status, task.error = "done", None\n                                    success_count += 1\n                                else:\n                                    task.error = "Download failed"\n                            else:\n                                print(f"   ❌ TorBox DL error: {dl_err}")\n                                task.error = dl_err[:100]\n                        except Exception as e:\n                            print(f"   ❌ Failed to download: {str(e)[:60]}")\n                            task.error = str(e)[:100]\n                    \n                    break\n                \n                elif status in [\'error\', \'failed\', \'dead\']:\n                    print(f"   ❌ Torrent error: {status}")\n                    break\n                \n                time.sleep(2)\n            else:\n                print("   ❌ Timeout waiting for torrent to cache (4 min)")\n                \n        except Exception as e:\n            print(f"   ❌ Error processing torrent: {str(e)[:80]}")\n        finally:\n            _reset_progress_bar()\n\n    return success_count\n\ndef process_tb_link(link: str, tb_key: str) -> bool:\n    """Process a link through TorBox (both magnets and regular links).\n    Returns True on success (or duplicate skip), False on any failure."""\n    if "magnet:?" in link:\n        # For magnets, add to TorBox and download all files\n        print("   🧲 Processing magnet via TorBox...")\n        tasks = resolve_tb_magnet_files(link, tb_key)\n        if not tasks:\n            return False\n        success = process_tb_magnet_file_tasks(tasks, tb_key)\n        return success > 0  # At least one file downloaded\n\n    # Regular link - unrestrict via web download\n    resolved = resolve_tb_link(link, tb_key)\n    if not resolved:\n        return False\n    all_ok = True\n    for dl_url, filename in resolved:\n        task_id = f"tb_{str(uuid4())[:8]}"\n        with progress_lock:\n            progress_bar.value = 0\n            progress_bar.bar_style = \'info\'\n        f = download_with_aria2(dl_url, filename, COLAB_ROOT, task_id=task_id)\n        if f is DUPLICATE_SKIP:\n            continue  # Already in Drive\n        if f:\n            handle_file_processing(f)\n        else:\n            all_ok = False\n    return all_ok\n\ndef resolve_mediafire(url: str, session: requests.Session) -> List[Tuple[str, str]]:\n    """Resolve MediaFire link to direct download URL by parsing HTML."""\n    try:\n        resp = session.get(url, timeout=30)\n        # Look for the download button href\n        match = re.search(r\'href="(https://download\\d*\\.mediafire\\.com/[^"]+)"\', resp.text)\n        if match:\n            download_url = match.group(1)\n            # Extract filename from URL or page title\n            filename_match = re.search(r\'/([^/]+)$\', download_url)\n            if filename_match:\n                filename = unquote(filename_match.group(1))\n                print(f"   📁 MediaFire: {filename}")\n                return [(download_url, sanitize_filename(filename))]\n        # Try alternate pattern for older MediaFire pages\n        match2 = re.search(r\'aria-label="Download file"\\s+href="([^"]+)"\', resp.text)\n        if match2:\n            download_url = match2.group(1)\n            filename = re.search(r\'/([^/]+)$\', download_url).group(1)\n            return [(download_url, sanitize_filename(unquote(filename)))]\n        print(f"   ⚠️ MediaFire: Could not find download link")\n    except Exception as e:\n        print(f"   ❌ MediaFire Error: {str(e)[:80]}")\n    return []\n\ndef resolve_1fichier(url: str, session: requests.Session) -> List[Tuple[str, str]]:\n    """Resolve 1fichier link to direct download URL."""\n    try:\n        # Get the page first to extract any needed info\n        resp = session.get(url, timeout=30)\n        \n        # Extract filename from page\n        filename_match = re.search(r\'<title>([^<]+)</title>\', resp.text)\n        filename = "1fichier_download"\n        if filename_match:\n            title = filename_match.group(1)\n            # Clean up title (remove "1fichier.com:" prefix if present)\n            filename = re.sub(r\'^.*?:\\s*\', \'\', title).strip()\n            if not filename or filename == "1fichier.com":\n                filename = "1fichier_download"\n        \n        # 1fichier requires a POST to download\n        # Check if there\'s a waiting time (free downloads)\n        if \'You must wait\' in resp.text or \'Please wait\' in resp.text:\n            print(f"   ⚠️ 1fichier: Rate limited, try later or use premium")\n            return []\n        \n        # Try to get the download link via POST\n        # Note: 1fichier may require CAPTCHA for free downloads\n        post_resp = session.post(url, data={\'dl_no_ssl\': \'on\', \'dlinline\': \'on\'}, timeout=30, allow_redirects=False)\n        \n        if post_resp.status_code == 302:\n            # Redirect to download URL\n            download_url = post_resp.headers.get(\'Location\', \'\')\n            if download_url:\n                print(f"   📁 1fichier: {filename}")\n                return [(download_url, sanitize_filename(filename))]\n        \n        # Check response for direct link\n        dl_match = re.search(r\'href="(https://[^"]*1fichier[^"]*)"[^>]*>Click here\', post_resp.text, re.IGNORECASE)\n        if dl_match:\n            return [(dl_match.group(1), sanitize_filename(filename))]\n        \n        print(f"   ⚠️ 1fichier: Could not extract download link (may require premium or CAPTCHA)")\n    except Exception as e:\n        print(f"   ❌ 1fichier Error: {str(e)[:80]}")\n    return []\n\n# --- FSHARE RESOLVER ---\n# FShare API credentials (used by legacy API if it still works)\nFSHARE_APP_KEY = "L2S7R6ZMagggC5wWkQhX2+aDi467PPuftWUMRFSn"\nFSHARE_API_URL = "https://api.fshare.vn/api"\n\ndef _fshare_api_login(email: str, password: str, session: requests.Session) -> Optional[Dict[str, str]]:\n    """Login to FShare via legacy API. Returns {\'token\': ..., \'session_id\': ...} or None."""\n    try:\n        data = {\n            "user_email": email,\n            "password": password,\n            "app_key": FSHARE_APP_KEY,\n        }\n        resp = session.post(f"{FSHARE_API_URL}/user/login",\n                           json=data,\n                           headers={"User-Agent": "okhttp/3.6.0", "Content-Type": "application/json"},\n                           timeout=REQUEST_TIMEOUT)\n        result = resp.json()\n        if result.get("code") == 200 and result.get("token"):\n            return {"token": result["token"], "session_id": result.get("session_id", "")}\n        else:\n            return None\n    except Exception:\n        return None\n\ndef _fshare_api_get_download_link(url: str, token: str, session: requests.Session) -> Optional[str]:\n    """Get direct download link from FShare API. Returns URL string or None."""\n    try:\n        data = {"token": token, "url": url}\n        resp = session.post(f"{FSHARE_API_URL}/session/download",\n                           json=data,\n                           headers={"User-Agent": "okhttp/3.6.0", "Content-Type": "application/json"},\n                           timeout=REQUEST_TIMEOUT)\n        if resp.status_code == 200:\n            result = resp.json()\n            download_url = result.get("location") or result.get("url") or result.get("download")\n            if download_url and download_url.startswith("http"):\n                return download_url\n        return None\n    except Exception:\n        return None\n\ndef _fshare_api_list_folder(url: str, token: str, session: requests.Session) -> List[Dict[str, Any]]:\n    """List files in an FShare folder via API. Returns list of file info dicts."""\n    try:\n        # Extract folder code from URL\n        match = re.search(r\'fshare\\.vn/folder/([a-zA-Z0-9]+)\', url)\n        if not match:\n            return []\n        folder_code = match.group(1)\n        \n        # FShare API folder listing\n        page = 1\n        all_files = []\n        while True:\n            data = {"token": token, "url": url, "dirOnly": 0, "pageIndex": page}\n            resp = session.post(f"{FSHARE_API_URL}/fileops/listDir",\n                               json=data,\n                               headers={"User-Agent": "okhttp/3.6.0", "Content-Type": "application/json"},\n                               timeout=REQUEST_TIMEOUT)\n            if resp.status_code != 200:\n                break\n            result = resp.json()\n            items = result if isinstance(result, list) else result.get("items", result.get("data", []))\n            if not items:\n                break\n            for item in items:\n                if item.get("type") == 1:  # type 1 = file (not subfolder)\n                    all_files.append({\n                        "name": item.get("name", "unknown"),\n                        "url": f"https://www.fshare.vn/file/{item.get(\'linkcode\', \'\')}",\n                        "size": item.get("size", 0),\n                    })\n            # Check if there are more pages\n            if len(items) < 50:  # Assume page size is ~50\n                break\n            page += 1\n        return all_files\n    except Exception:\n        return []\n\ndef _fshare_extract_filename(url: str, session: requests.Session) -> str:\n    """Extract filename from an FShare file page by scraping the HTML."""\n    try:\n        resp = session.get(url, timeout=REQUEST_TIMEOUT,\n                          headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"})\n        if resp.status_code == 200:\n            # Try to extract filename from the page title or file info section\n            title_match = re.search(r\'<title>([^<]+)</title>\', resp.text)\n            if title_match:\n                title = title_match.group(1).strip()\n                # FShare titles are typically "filename - Fshare" or just "filename"\n                title = re.sub(r\'\\s*[-|]\\s*[Ff]share.*$\', \'\', title).strip()\n                if title and title.lower() not in [\'fshare\', \'fshare.vn\', \'\']:\n                    return sanitize_filename(title)\n            # Try data attribute or download button text\n            name_match = re.search(r\'class="file-name[^"]*"[^>]*>([^<]+)<\', resp.text)\n            if name_match:\n                return sanitize_filename(name_match.group(1).strip())\n    except Exception:\n        pass\n    # Fallback: extract from URL\n    match = re.search(r\'fshare\\.vn/file/([a-zA-Z0-9]+)\', url)\n    return f"fshare_{match.group(1)}" if match else "fshare_download"\n\ndef save_fshare_cookies(session: requests.Session):\n    """Save FShare session cookies to json file."""\n    try:\n        if not os.path.exists(UD_CONFIG_PATH):\n            os.makedirs(UD_CONFIG_PATH, exist_ok=True)\n        cookies = requests.utils.dict_from_cookiejar(session.cookies)\n        with open(FSHARE_COOKIE_FILE, \'w\') as f:\n            json.dump(cookies, f)\n    except Exception as e:\n        print(f"   ⚠️ Could not save FShare cookies: {e}")\n\ndef load_fshare_cookies(session: requests.Session) -> bool:\n    """Load FShare session cookies from json file if it exists."""\n    try:\n        if os.path.exists(FSHARE_COOKIE_FILE):\n            with open(FSHARE_COOKIE_FILE, \'r\') as f:\n                cookies = json.load(f)\n            session.cookies.update(requests.utils.cookiejar_from_dict(cookies))\n            return True\n    except Exception as e:\n        print(f"   ⚠️ Could not load FShare cookies: {e}")\n    return False\n\ndef _is_fshare_logged_in(session: requests.Session) -> bool:\n    """Verify if FShare session is active by requesting the login page."""\n    try:\n        resp = session.get("https://www.fshare.vn/site/login",\n                           headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"},\n                           timeout=10,\n                           allow_redirects=True)\n        return "/site/login" not in resp.url\n    except Exception:\n        return False\n\ndef _fshare_web_login(email: str, password: str, session: requests.Session) -> bool:\n    """Login to FShare via web interface. Returns True on success."""\n    try:\n        # Get the login page to obtain CSRF token and session cookie\n        login_url = "https://www.fshare.vn/site/login"\n        login_page = session.get(login_url,\n                                headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"},\n                                timeout=REQUEST_TIMEOUT)\n        \n        # Check if CAPTCHA is present on page load\n        if \'robot\' in login_page.text or \'phép tính\' in login_page.text:\n            print(f"   ⚠️ FShare: Robot verification (CAPTCHA) detected on login page. Please log in to fshare.vn in your web browser first to clear the challenge.")\n            return False\n            \n        # Extract CSRF token - FShare uses "_csrf-app" as the parameter name\n        # Method 1: From hidden input field (most reliable)\n        csrf_input = re.search(r\'name="(_csrf-app)"\\s+value="([^"]+)"\', login_page.text)\n        if csrf_input:\n            csrf_param = csrf_input.group(1)\n            csrf_token = csrf_input.group(2)\n        else:\n            # Method 2: From meta tags\n            csrf_param_meta = re.search(r\'name="csrf-param"\\s+content="([^"]+)"\', login_page.text)\n            csrf_token_meta = re.search(r\'name="csrf-token"\\s+content="([^"]+)"\', login_page.text)\n            csrf_param = csrf_param_meta.group(1) if csrf_param_meta else "_csrf-app"\n            csrf_token = csrf_token_meta.group(1) if csrf_token_meta else ""\n        \n        if not csrf_token:\n            print(f"   ⚠️ FShare: Could not extract CSRF token from login page")\n            return False\n            \n        print(f"   ℹ️ FShare: Extracted CSRF token successfully ({csrf_param}={csrf_token[:15]}...)")\n        \n        # Submit login form\n        login_data = {\n            csrf_param: csrf_token,\n            "LoginForm[email]": email,\n            "LoginForm[password]": password,\n            "LoginForm[rememberMe]": "0",\n        }\n        \n        resp = session.post(login_url,\n                           data=login_data,\n                           headers={\n                               "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",\n                               "Referer": login_url,\n                               "Content-Type": "application/x-www-form-urlencoded",\n                           },\n                           timeout=REQUEST_TIMEOUT,\n                           allow_redirects=True)\n        \n        print(f"   ℹ️ FShare: Login POST status code: {resp.status_code}")\n        print(f"   ℹ️ FShare: Response URL after redirect: {resp.url}")\n        \n        # Check if login succeeded:\n        # - Failed login: stays on /site/login and contains LoginForm fields\n        # - Successful login: redirects away from /site/login (e.g. to homepage or dashboard)\n        if \'/site/login\' not in resp.url:\n            print(f"   ✅ FShare: Redirected away from login page to {resp.url}")\n            save_fshare_cookies(session)\n            return True\n        \n        # If still on login page, check if the login form is gone (another success indicator)\n        if \'LoginForm[email]\' not in resp.text:\n            print(f"   ✅ FShare: LoginForm[email] not found in response text. Assuming logged in.")\n            save_fshare_cookies(session)\n            return True\n            \n        # Check if CAPTCHA is present in failed POST response\n        if \'robot\' in resp.text or \'phép tính\' in resp.text:\n            print(f"   ⚠️ FShare: Robot verification (CAPTCHA) detected. Please log in to fshare.vn in your web browser first to clear the challenge.")\n        \n        # Let\'s print out potential errors shown on the page (mdc-text-field-helper-line or error classes)\n        val_msg_match = re.search(r\'class="[^"]*validation-msg[^"]*"[^>]*>([^<]+)\', resp.text)\n        if val_msg_match:\n            print(f"   ⚠️ FShare login page error: {val_msg_match.group(1).strip()}")\n        else:\n            error_match = re.search(r\'class="[^"]*error[^"]*"[^>]*>([^<]+)\', resp.text, re.IGNORECASE)\n            if error_match and error_match.group(1).strip():\n                print(f"   ⚠️ FShare login page error: {error_match.group(1).strip()}")\n            else:\n                # Let\'s check for yii validation errors\n                yii_errors = re.findall(r\'class="help-block"[^>]*>([^<]+)\', resp.text)\n                if yii_errors:\n                    print(f"   ⚠️ FShare validation errors: {yii_errors}")\n                else:\n                    invalid_fields = re.findall(r\'class="[^"]*mdc-text-field--invalid[^"]*"\', resp.text)\n                    if invalid_fields:\n                        print(f"   ⚠️ FShare login page has invalid input fields indicator")\n        \n        # Delete cookies on failure\n        if os.path.exists(FSHARE_COOKIE_FILE):\n            try:\n                os.remove(FSHARE_COOKIE_FILE)\n            except Exception:\n                pass\n        return False\n    except Exception as e:\n        print(f"   ❌ FShare login exception: {str(e)}")\n        # Delete cookies on failure\n        if os.path.exists(FSHARE_COOKIE_FILE):\n            try:\n                os.remove(FSHARE_COOKIE_FILE)\n            except Exception:\n                pass\n        return False\n\ndef _fshare_web_get_download_link(url: str, session: requests.Session) -> Optional[str]:\n    """Get direct download link from FShare file page (requires logged-in session).\n    \n    FShare\'s download mechanism works via an AJAX POST to /download/get with the\n    file\'s linkcode and CSRF token. The response is JSON: {url: "...", wait_time: N}.\n    For VIP users, wait_time is 0 and url is the direct download link.\n    """\n    try:\n        # Step 1: Visit the file page to get CSRF token and linkcode\n        resp = session.get(url,\n                          headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"},\n                          timeout=REQUEST_TIMEOUT)\n        \n        if resp.status_code != 200:\n            print(f"      🔍 File page returned status {resp.status_code}")\n            return None\n        \n        # Check for password protection\n        if \'password\' in resp.text.lower() and \'FilePasswordForm\' in resp.text:\n            print(f"   ⚠️ FShare: File is password-protected (not supported)")\n            return None\n        \n        # Extract CSRF token from the #form-download hidden input\n        csrf_input = re.search(r\'name="(_csrf-app)"\\s+value="([^"]+)"\', resp.text)\n        if csrf_input:\n            csrf_param = csrf_input.group(1)\n            csrf_token = csrf_input.group(2)\n        else:\n            # Fallback to meta tags\n            csrf_param_meta = re.search(r\'name="csrf-param"\\s+content="([^"]+)"\', resp.text)\n            csrf_token_meta = re.search(r\'name="csrf-token"\\s+content="([^"]+)"\', resp.text)\n            csrf_param = csrf_param_meta.group(1) if csrf_param_meta else "_csrf-app"\n            csrf_token = csrf_token_meta.group(1) if csrf_token_meta else ""\n        \n        if not csrf_token:\n            print(f"      🔍 No CSRF token found on file page (may not be logged in)")\n            return None\n        \n        # Extract linkcode from the form or URL\n        linkcode_input = re.search(r\'name="linkcode"\\s+value="([^"]+)"\', resp.text)\n        if linkcode_input:\n            linkcode = linkcode_input.group(1)\n        else:\n            # Fallback: extract from URL\n            linkcode_match = re.search(r\'fshare\\.vn/file/([a-zA-Z0-9]+)\', url)\n            if not linkcode_match:\n                return None\n            linkcode = linkcode_match.group(1)\n        \n        # Step 2: POST to /download/get (the AJAX endpoint used by download.js)\n        download_data = {\n            csrf_param: csrf_token,\n            "linkcode": linkcode,\n            "withFcode5": "0",\n        }\n        \n        dl_resp = session.post("https://www.fshare.vn/download/get",\n                              data=download_data,\n                              headers={\n                                  "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",\n                                  "Referer": url,\n                                  "X-Requested-With": "XMLHttpRequest",  # Mark as AJAX request\n                              },\n                              timeout=60)\n        \n        if dl_resp.status_code == 200:\n            try:\n                result = dl_resp.json()\n                if "url" in result and result["url"]:\n                    download_url = result["url"]\n                    wait_time = result.get("wait_time", 0)\n                    if wait_time > 0:\n                        print(f"      ⏳ Wait time {wait_time}s (free account speed limit)")\n                    return download_url\n                elif result.get("policydownload") or result.get("policydowload"):\n                    # FShare policy restriction (note: FShare API has a typo \'policydowload\')\n                    print(f"      ❌ FShare download policy restriction")\n                    return "FSHARE_LIMIT_REACHED"\n                elif "errors" in result:\n                    errors = result["errors"]\n                    if "linkcode" in errors:\n                        print(f"      ❌ {errors[\'linkcode\'][0]}")\n                    elif "fcode" in errors:\n                        print(f"      ❌ {errors[\'fcode\'][0]}")\n                    else:\n                        print(f"      ❌ Errors: {errors}")\n                else:\n                    print(f"      🔍 Unexpected response: {str(result)[:200]}")\n            except (ValueError, KeyError):\n                print(f"      🔍 Non-JSON response from /download/get: {dl_resp.text[:200]}")\n        else:\n            print(f"      🔍 /download/get returned status {dl_resp.status_code}")\n        \n        return None\n    except Exception as e:\n        print(f"   ❌ FShare download error: {str(e)[:80]}")\n        return None\n\n# Cached FShare web session — reused across all links in a batch to avoid repeated logins\n_fshare_cached_session: Optional[requests.Session] = None\n_fshare_cached_session_email: str = ""\n_fshare_cached_session_time: float = 0  # time.time() when session was last verified\n\ndef _get_fshare_web_session(email: str, password: str) -> Optional[requests.Session]:\n    """Get or create a cached FShare web session. Logs in only once per batch."""\n    global _fshare_cached_session, _fshare_cached_session_email, _fshare_cached_session_time\n    \n    # If we have a cached session for the same email, trust it if recently verified (<5 min)\n    if _fshare_cached_session and _fshare_cached_session_email == email:\n        elapsed = time.time() - _fshare_cached_session_time\n        if elapsed < 300:  # Trust cached session for 5 minutes without re-verifying\n            print(f"   ✅ FShare: Reusing active session (skipped login)")\n            return _fshare_cached_session\n        # Session is old, verify it\'s still active\n        if _is_fshare_logged_in(_fshare_cached_session):\n            print(f"   ✅ FShare: Reusing active session (skipped login)")\n            _fshare_cached_session_time = time.time()\n            return _fshare_cached_session\n        else:\n            print(f"   ℹ️ FShare: Cached session expired, logging in again...")\n            _fshare_cached_session = None\n    \n    # Create a new session\n    session = requests.Session()\n    \n    # Try to load saved cookies from disk\n    if load_fshare_cookies(session):\n        if _is_fshare_logged_in(session):\n            print(f"   ✅ FShare: Restored saved session (skipped login)")\n            _fshare_cached_session = session\n            _fshare_cached_session_email = email\n            _fshare_cached_session_time = time.time()\n            return session\n    \n    # Need a fresh login\n    if _fshare_web_login(email, password, session):\n        _fshare_cached_session = session\n        _fshare_cached_session_email = email\n        _fshare_cached_session_time = time.time()\n        return session\n    \n    # Login failed\n    _fshare_cached_session = None\n    return None\n\ndef resolve_fshare(url: str, email: str, password: str) -> List[Tuple[str, str]]:\n    """Resolve FShare URL to direct download link(s) via web scraping.\n    \n    Supports both file URLs (fshare.vn/file/...) and folder URLs (fshare.vn/folder/...).\n    Uses a cached session to avoid repeated logins when resolving multiple links.\n    """\n    if not email or not password:\n        print(f"   ❌ FShare credentials required (set in Settings → FShare Account)")\n        return []\n    \n    # Determine if this is a folder or file URL\n    is_folder = \'/folder/\' in url\n    \n    results: List[Tuple[str, str]] = []\n    \n    print(f"   🇻🇳 Resolving FShare: {url[:60]}...")\n    session = _get_fshare_web_session(email, password)\n    if not session:\n        print(f"   ❌ FShare: Login failed — click Resolve Links again to retry")\n        return []\n    \n    if is_folder:\n        # Use FShare\'s internal web API to list folder contents\n        try:\n            # Extract linkcode from URL (e.g., /folder/ABC123XYZ -> ABC123XYZ)\n            linkcode_match = re.search(r\'/folder/([a-zA-Z0-9]+)\', url)\n            if not linkcode_match:\n                print(f"   ❌ FShare: Could not extract folder linkcode from URL")\n                return []\n            linkcode = linkcode_match.group(1)\n            \n            all_files = []\n            page = 1\n            per_page = 50\n            \n            while True:\n                api_url = f"https://www.fshare.vn/api/v3/files/folder?linkcode={linkcode}&sort=type,name&page={page}&per-page={per_page}"\n                resp = session.get(api_url,\n                                   headers={\n                                       "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",\n                                       "Accept": "application/json",\n                                       "X-Requested-With": "XMLHttpRequest",\n                                       "Referer": url,\n                                   },\n                                   timeout=REQUEST_TIMEOUT)\n                \n                if resp.status_code != 200:\n                    print(f"   ❌ FShare folder API returned status {resp.status_code}")\n                    break\n                \n                try:\n                    data = resp.json()\n                except ValueError:\n                    print(f"   ❌ FShare folder API returned non-JSON response")\n                    break\n                \n                items = data if isinstance(data, list) else data.get("items", data.get("data", []))\n                if not items:\n                    break\n                \n                for item in items:\n                    if isinstance(item, dict):\n                        # FShare: type=1 is file, type=0 is folder\n                        item_type = item.get("type")\n                        if item_type == 0:\n                            continue  # Skip sub-folders\n                        \n                        name = item.get("name", "")\n                        item_linkcode = item.get("linkcode", "")\n                        size = item.get("size", 0)\n                        \n                        if name and item_linkcode:\n                            file_url = f"https://www.fshare.vn/file/{item_linkcode}"\n                            all_files.append({"url": file_url, "name": name, "size": int(size) if size else 0})\n                \n                # Check pagination via \'links\' dict\n                # links = {"self": "...?page=1", "first": "...?page=1", "last": "...?page=N"}\n                links = data.get("links", {}) if isinstance(data, dict) else {}\n                last_link = links.get("last", "")\n                has_more = False\n                \n                if last_link:\n                    last_page_match = re.search(r\'page=(\\d+)\', last_link)\n                    if last_page_match:\n                        total_pages = int(last_page_match.group(1))\n                        has_more = page < total_pages\n                \n                # Fallback: if we got exactly per_page items, there are likely more pages\n                if not has_more and len(items) >= per_page:\n                    has_more = True\n                \n                if not has_more:\n                    break\n                page += 1\n                time.sleep(0.5)  # Rate limit between pages\n            \n            if all_files:\n                total = len(all_files)\n                print(f"   📁 FShare folder: {total} file(s) listed")\n                print(f"   💡 Download links will be resolved when you click \'Start Download\'")\n                print(f"   💡 Remove unwanted files from the queue first to save your daily download limit")\n                for f_info in all_files:\n                    file_url = f_info["url"]\n                    filename = f_info["name"]\n                    results.append((file_url, sanitize_filename(filename)))\n                return results\n            else:\n                print(f"   ⚠️ FShare: No files found in folder (folder may be empty or require login)")\n        except Exception as e:\n            print(f"   ❌ FShare folder error: {str(e)[:80]}")\n        return []\n    else:\n        # Single file via web\n        filename = _fshare_extract_filename(url, session)\n        dl_link = _fshare_web_get_download_link(url, session)\n        if dl_link and dl_link != "FSHARE_LIMIT_REACHED":\n            print(f"   ✅ FShare (web): {filename}")\n            return [(dl_link, sanitize_filename(filename))]\n        elif dl_link == "FSHARE_LIMIT_REACHED":\n            print(f"   🛑 FShare daily download limit reached — try again tomorrow")\n            return []\n        else:\n            print(f"   ❌ FShare: Could not extract download link — VIP account may be required")\n            return []\n\ndef _tb_url_refresher(task: DownloadTask) -> Optional[Callable[[], str]]:\n    """Build a fresh-link minter for a TorBox file task, or None for other types.\n\n    convert_tb_tasks_to_parallel pre-requests every direct link before the first\n    download starts, so on a long batch the later files\' links expire before a\n    worker reaches them. original_url keeps the \'{endpoint}/{item_id}:{file_id}\'\n    handle those links were minted from, which is all _tb_requestdl needs to mint\n    another one on demand."""\n    if task.link_type != \'tb_magnet_file\' or not task.original_url or \':\' not in task.original_url:\n        return None\n    # split(\':\', 1) — same convention _group_tasks_by_torrent uses to read this field\n    item_ref, file_id = task.original_url.split(\':\', 1)\n    endpoint, _, item_id = item_ref.rpartition(\'/\')\n    endpoint = endpoint or \'torrents\'\n    id_param = _TB_REQUESTDL_ID_PARAM.get(endpoint, \'torrent_id\')\n\n    def _refresh() -> str:\n        tb_key = token_tb.value.strip()\n        if not tb_key:\n            return \'\'\n        url, _err = _tb_requestdl(endpoint, id_param, item_id, file_id, tb_key, attempts=2)\n        return url\n    return _refresh\n\n# --- PARALLEL DOWNLOAD WORKER ---\ndef download_worker(task: DownloadTask, gofile_token: str, move_queue: Optional[queue.Queue] = None) -> DownloadTask:\n    """Worker function for parallel downloads. Returns updated task.\n\n    With move_queue (the "Overlap Drive moves" setting), a finished download is\n    handed to the mover thread and this worker frees its pool slot for the next\n    download; the mover owns the task\'s done/failed transition from there."""\n    if _cancel_requested:\n        return task  # Never started — stays "pending" so resume picks it up untouched\n    task.status = "downloading"\n    try:\n        # update_bar=False: the batch monitor thread owns the shared progress bar\n        f = download_with_aria2(task.url, task.filename, COLAB_ROOT, task.cookie, task_id=task.id,\n                                update_bar=False, url_refresh=_tb_url_refresher(task))\n        if f is DUPLICATE_SKIP:\n            task.status = "skipped"  # Already in Drive — not a failure, don\'t retry on resume\n        elif f and move_queue is not None:\n            task.status = "moving"\n            while True:  # Bounded queue caps the local-disk backlog — block here until the mover catches up\n                try:\n                    move_queue.put((task, f), timeout=1)\n                    break\n                except queue.Full:\n                    if _cancel_requested:\n                        task.status = "failed"\n                        task.error = "Interrupted before Drive move (file kept locally)"\n                        break\n        elif f:\n            # Same "moving" state the overlapped path uses. Without it the task stays\n            # "downloading" for the whole (blocking) Drive transfer and keeps rendering\n            # at its last aria2 percentage — which reads as a download stuck at 99%.\n            task.status = "moving"\n            handle_file_processing(f, source=task.source,\n                                  relative_path=task.relative_path, torrent_name=task.torrent_name)\n            task.status = "done"\n        elif _cancel_requested:\n            task.status = "failed"\n            task.error = "Cancelled by user"\n        else:\n            task.status = "failed"\n            if download_stats.get(task.id, {}).get(\'disk_gave_up\'):\n                task.error = "Local disk full — Drive moves couldn\'t free space in time"\n            else:\n                task.error = "Download returned None"\n    except Exception as e:\n        task.status = "failed"\n        task.error = str(e)[:100]\n    return task\n\ndef _drive_mover(move_queue: queue.Queue, save_progress):\n    """Consume (task, file_path) items from the download pool and move each file to Drive.\n\n    Several of these run at once (drive_movers_slider). With the Drive API path these\n    threads do the uploading themselves, so their count sets real upload concurrency —\n    unlike a write to the FUSE mount, which only fills a local cache that drivefs\'s\n    own single background uploader then drains (adding writers there measurably bought\n    nothing). The download pool\'s size is dictated by debrid concurrency limits, so\n    mover count has to scale separately from it.\n\n    Each thread consumes exactly one None sentinel and exits; the shutdown path\n    puts one per thread."""\n    while True:\n        item = move_queue.get()\n        if item is None:\n            move_queue.task_done()\n            return\n        task, path = item\n        try:\n            handle_file_processing(path, source=task.source,\n                                  relative_path=task.relative_path, torrent_name=task.torrent_name)\n            task.status = "done"\n        except Exception as e:\n            task.status = "failed"\n            task.error = str(e)[:100]\n        finally:\n            move_queue.task_done()\n        save_progress()\n\n# --- URL CLASSIFICATION ---\nSTREAMING_HOSTS = (\'youtube.com\', \'youtu.be\', \'vimeo.com\', \'twitch.tv\', \'ok.ru\')\n\ndef _url_host(url: str) -> str:\n    """Return the lowercased hostname of a URL (\'\' if unparseable)."""\n    try:\n        return (urlparse(url).hostname or \'\').lower()\n    except Exception:\n        return \'\'\n\ndef url_matches_host(url: str, hosts) -> bool:\n    """True if the URL\'s hostname equals one of `hosts` or is a subdomain of one.\n    Hostname-based (not substring) so \'evil.com/?x=mega.nz\' does not match."""\n    host = _url_host(url)\n    if not host:\n        return False\n    return any(host == h or host.endswith(\'.\' + h) for h in hosts)\n\ndef _classify_url(url: str, has_debrid: bool) -> str:\n    """Map a URL to a resolver kind (mirrors the historical routing order)."""\n    if url_matches_host(url, (\'transfer.it\',)):\n        return \'transfer\'\n    if url_matches_host(url, (\'mega.nz\',)):\n        return \'mega\'\n    if url_matches_host(url, STREAMING_HOSTS):\n        return \'stream\'\n    if url_matches_host(url, (\'archive.org\',)) and \'/details/\' in urlparse(url).path:\n        return \'stream\'\n    if url_matches_host(url, (\'gofile.io\',)):\n        return \'gofile\'\n    if url_matches_host(url, (\'pixeldrain.com\',)):\n        return \'pixeldrain\'\n    if url_matches_host(url, (\'mediafire.com\',)):\n        return \'mediafire\'\n    if url_matches_host(url, (\'1fichier.com\',)):\n        return \'1fichier\'\n    if url.startswith(\'magnet:\'):\n        return \'magnet\'\n    if url_matches_host(url, (\'real-debrid.com\',)) and \'/d/\' in urlparse(url).path:\n        return \'rd_direct\'\n    if url_matches_host(url, (\'fshare.vn\',)):\n        return \'fshare\'\n    if url_matches_host(url, (\'torbox.app\',)) and \'id=\' in (urlparse(url).query or \'\'):\n        return \'tb_folder\'\n    if url_matches_host(url, (\'tb-cdn.io\',)):\n        return \'tb_cdn\'\n    if has_debrid and url_matches_host(url, DEBRID_SUPPORTED_HOSTS):\n        return \'debrid_host\'\n    if has_debrid and url.startswith(\'http\') and not url_matches_host(url, (\'archive.org\',)):\n        return \'debrid_generic\'\n    return \'direct\'\n\ndef resolve_all_links(urls: List[str], session: requests.Session, tokens: dict, rd_key: str, tb_key: str = "", debrid_service: str = "rd") -> Tuple[List[DownloadTask], List[str], List[str], List[str]]:\n    """\n    Pre-resolve all links into DownloadTasks.\n    Returns: (parallel_tasks, youtube_urls, mega_urls, debrid_urls)\n    debrid_service: \'rd\', \'tb\', or \'none\'\n\n    Independent HTTP resolvers (Gofile/Pixeldrain/TorBox-CDN and, when no debrid\n    service is active, MediaFire/1fichier) run concurrently. Rate-limited services\n    (debrid APIs, FShare) stay sequential. Results keep the original URL order.\n    """\n    global _rd_magnet_delay\n    _rd_magnet_delay = 0  # Reset adaptive pacing for each new batch\n    parallel_tasks: List[DownloadTask] = []\n    youtube_urls: List[str] = []\n    mega_urls: List[str] = []\n    debrid_urls: List[str] = []\n\n    # Determine which debrid key is active\n    has_debrid = bool((debrid_service == \'rd\' and rd_key) or (debrid_service == \'tb\' and tb_key))\n\n    classified = [(url, _classify_url(url, has_debrid)) for url in urls]\n\n    # Kick off independent resolvers concurrently (keyed by position - URLs may repeat)\n    futures = {}\n    with ThreadPoolExecutor(max_workers=4) as executor:\n        for i, (url, kind) in enumerate(classified):\n            if kind == \'gofile\':\n                futures[i] = executor.submit(resolve_gofile, url, session, tokens)\n            elif kind == \'pixeldrain\':\n                futures[i] = executor.submit(resolve_pixeldrain, url, session)\n            elif kind == \'mediafire\' and not has_debrid:\n                futures[i] = executor.submit(resolve_mediafire, url, session)\n            elif kind == \'1fichier\' and not has_debrid:\n                futures[i] = executor.submit(resolve_1fichier, url, session)\n            elif kind == \'tb_cdn\':\n                futures[i] = executor.submit(_resolve_tb_cdn, url, session)\n\n        for i, (url, kind) in enumerate(classified):\n            if kind == \'transfer\':\n                mega_urls.append(url)\n            elif kind == \'mega\':\n                if debrid_service == \'rd\' and rd_key:\n                    # Try RD for MEGA (handles all URL formats)\n                    resolved = resolve_rd_link(url, rd_key)\n                    if resolved:\n                        for u, n in resolved:\n                            parallel_tasks.append(DownloadTask(\n                                url=u, filename=n, source="mega", link_type="rd",\n                                original_url=url\n                            ))\n                    else:\n                        # RD failed (e.g. ip_not_allowed from Colab) - fall back to megadl\n                        print(f"   ⤴️ Falling back to megadl for: {url[:60]}...")\n                        mega_urls.append(url)\n                elif debrid_service == \'tb\' and tb_key:\n                    # Try TorBox for MEGA via web download\n                    resolved = resolve_tb_link(url, tb_key)\n                    if resolved:\n                        for u, n in resolved:\n                            parallel_tasks.append(DownloadTask(\n                                url=u, filename=n, source="mega", link_type="tb",\n                                original_url=url\n                            ))\n                    else:\n                        print(f"   ⤴️ Falling back to megadl for: {url[:60]}...")\n                        mega_urls.append(url)\n                else:\n                    mega_urls.append(url)\n            elif kind == \'stream\':\n                youtube_urls.append(url)\n            elif kind == \'gofile\':\n                for u, n in futures[i].result():\n                    parallel_tasks.append(DownloadTask(\n                        url=u, filename=n, source="gofile", link_type="gofile",\n                        cookie=tokens.get(\'token\'), original_url=url  # Store original for re-resolve\n                    ))\n            elif kind == \'pixeldrain\':\n                for u, n in futures[i].result():\n                    parallel_tasks.append(DownloadTask(\n                        url=u, filename=n, source="pixeldrain", link_type="pixeldrain",\n                        original_url=url  # Store original for re-resolve\n                    ))\n            elif kind in (\'mediafire\', \'1fichier\'):\n                # Prefer active debrid service, fall back to the direct resolver\n                if debrid_service == \'rd\' and rd_key:\n                    resolved = resolve_rd_link(url, rd_key)\n                    link_type = "rd"\n                elif debrid_service == \'tb\' and tb_key:\n                    resolved = resolve_tb_link(url, tb_key)\n                    link_type = "tb"\n                else:\n                    resolved = futures[i].result()\n                    link_type = kind\n                for u, n in resolved:\n                    parallel_tasks.append(DownloadTask(\n                        url=u, filename=n, source=kind, link_type=link_type,\n                        original_url=url\n                    ))\n            elif kind == \'magnet\':\n                # Resolve magnet to individual files via active debrid service\n                if debrid_service == \'rd\' and rd_key:\n                    # Adaptive pacing: delay only after a rate-limit has been hit\n                    if _rd_magnet_delay > 0:\n                        time.sleep(_rd_magnet_delay)\n                    parallel_tasks.extend(resolve_magnet_files(url, rd_key))\n                elif debrid_service == \'tb\' and tb_key:\n                    parallel_tasks.extend(resolve_tb_magnet_files(url, tb_key))\n                else:\n                    print(f"   ❌ Debrid token required for magnet links (select Real-Debrid or TorBox)")\n            elif kind == \'tb_folder\':\n                # torbox.app share links always resolve via TorBox regardless of toggle\n                tb_key_for_tb_links = tb_key or token_tb.value.strip()\n                parallel_tasks.extend(resolve_tb_folder_files(url, tb_key_for_tb_links))\n            elif kind == \'tb_cdn\':\n                for u, n in futures[i].result():\n                    parallel_tasks.append(DownloadTask(\n                        url=u, filename=n, source="torbox", link_type="direct",\n                        original_url=url\n                    ))\n            elif kind == \'rd_direct\':\n                # RD direct links - always use RD regardless of toggle\n                rd_key_for_rd_links = rd_key or token_rd.value.strip()\n                resolved = resolve_rd_link(url, rd_key_for_rd_links)\n                for u, n in resolved:\n                    parallel_tasks.append(DownloadTask(\n                        url=u, filename=n, source="rd", link_type="rd",\n                        original_url=url  # Store original for re-resolve\n                    ))\n            elif kind == \'fshare\':\n                # FShare file or folder links - resolve with VIP account\n                fshare_email = token_fshare_email.value.strip()\n                fshare_password = token_fshare_password.value.strip()\n                resolved = resolve_fshare(url, fshare_email, fshare_password)\n                for u, n in resolved:\n                    parallel_tasks.append(DownloadTask(\n                        url=u, filename=n, source="fshare", link_type="fshare",\n                        original_url=url\n                    ))\n                if resolved:\n                    time.sleep(1)  # Rate limit between FShare link resolutions\n            elif kind == \'debrid_host\':\n                # Route through active debrid service for supported premium hosts\n                if debrid_service == \'rd\' and rd_key:\n                    resolved = resolve_rd_link(url, rd_key)\n                    for u, n in resolved:\n                        parallel_tasks.append(DownloadTask(\n                            url=u, filename=n, source="rd_host", link_type="rd",\n                            original_url=url\n                        ))\n                elif debrid_service == \'tb\' and tb_key:\n                    resolved = resolve_tb_link(url, tb_key)\n                    for u, n in resolved:\n                        parallel_tasks.append(DownloadTask(\n                            url=u, filename=n, source="tb_host", link_type="tb",\n                            original_url=url\n                        ))\n            elif kind == \'debrid_generic\':\n                # Other links through debrid - try unrestricting\n                debrid_urls.append(url)\n            else:\n                # Direct URL (including archive.org/download/ links)\n                filename = os.path.basename(unquote(urlparse(url).path)) or "download"\n                source = "archive" if url_matches_host(url, (\'archive.org\',)) else "direct"\n                parallel_tasks.append(DownloadTask(\n                    url=url, filename=filename, source=source, link_type="direct"\n                ))\n\n    return parallel_tasks, youtube_urls, mega_urls, debrid_urls\n\n\ndef update_progress_display(tasks: List[DownloadTask]):\n    """Update overall progress bar and per-download progress bars."""\n    global last_display_speed\n    \n    active = [t for t in tasks if t.status == "downloading"]\n    moving = sum(1 for t in tasks if t.status == "moving")  # downloaded, Drive move pending (overlap mode)\n    done = sum(1 for t in tasks if t.status in ["done", "skipped"])\n    failed = sum(1 for t in tasks if t.status == "failed")\n    total = len(tasks)\n    now = time.time()\n    \n    # Collect aggregate speed and per-task progress from active downloads\n    total_speed_mbs = 0.0\n    active_pct_sum = 0.0\n    for t in active:\n        stats = download_stats.get(t.id)\n        if stats:\n            total_speed_mbs += stats[\'speed_mbs\']\n            active_pct_sum += stats[\'pct\'] / 100.0  # 0.0 – 1.0 contribution\n    \n    # Use current speed if available, otherwise keep last known speed\n    if total_speed_mbs > 0:\n        last_display_speed = total_speed_mbs\n    display_speed = last_display_speed if last_display_speed > 0 else total_speed_mbs\n    \n    # Disk-space guard readout (Colab local disk; inf when unmeasurable locally)\n    disk_free = _disk_free_gb()\n    if disk_free == float(\'inf\'):\n        disk_part = ""\n    elif disk_free < DISK_START_GB:\n        disk_part = f" | <b style=\'color:orange\'>💾 {disk_free:.1f} GB free</b>"\n    else:\n        disk_part = f" | 💾 {disk_free:.0f} GB free"\n\n    # --- OVERALL PROGRESS BAR ---\n    # Fractional progress: completed tasks + fractional progress of active tasks.\n    # Gives smooth continuous movement instead of staircase jumps.\n    display_progress = ((done + moving + active_pct_sum) / total) * 100 if total else 0\n    progress_bar.value = display_progress\n    progress_bar.bar_style = \'warning\' if active else \'success\' if done == total else \'info\'\n    \n    # Calculate ETA based on remaining tasks and current speed\n    remaining = total - done - failed\n    eta_str = ""\n    if active and batch_start_time:\n        elapsed = now - batch_start_time\n        if done > 0:\n            avg_time_per_task = elapsed / done\n            eta_seconds = avg_time_per_task * remaining\n            if eta_seconds < 60:\n                eta_str = f"{int(eta_seconds)}s"\n            elif eta_seconds < 3600:\n                eta_str = f"{int(eta_seconds // 60)}m {int(eta_seconds % 60)}s"\n            else:\n                eta_str = f"{int(eta_seconds // 3600)}h {int((eta_seconds % 3600) // 60)}m"\n    \n    if active:\n        speed_str = f"{display_speed:.1f} MB/s" if display_speed > 0 else "starting..."\n        eta_part = f" | ⏱️ {eta_str}" if eta_str else ""\n        moving_part = f" | 📤 {moving} moving" if moving else ""\n        progress_bar.description = f"⚡ {done}/{total}"\n        status_label.value = f"<small>📊 <b>{len(active)} downloading</b>{moving_part} | ⬇️ {speed_str}{eta_part}{disk_part}</small>"\n    elif moving:\n        progress_bar.description = f"📤 {done}/{total}"\n        status_label.value = f"<small>📤 <b>{moving} file{\'s\' if moving != 1 else \'\'} moving to Drive...</b>{disk_part}</small>"\n    elif done == total:\n        progress_bar.description = f"✅ {done}/{total}"\n        status_label.value = ""\n        last_display_speed = 0.0  # Reset for next batch\n    elif failed > 0:\n        progress_bar.description = f"⚠️ {done}/{total}"\n        status_label.value = f"<small style=\'color:orange\'>❌ {failed} failed</small>"\n    else:\n        progress_bar.description = f"DL {done}/{total}"\n    \n    # --- PER-DOWNLOAD PROGRESS BARS ---\n    finished_ids = {t.id for t in tasks if t.status in (\'done\', \'skipped\', \'failed\')}\n    \n    # Create bars for newly-active tasks\n    for t in active:\n        if t.id not in _per_task_bars:\n            name = t.filename[:40] if t.filename else t.url[:40]\n            bar = widgets.FloatProgress(\n                value=0.0, min=0.0, max=100.0,\n                description=name,\n                bar_style=\'info\',\n                style={\'description_width\': \'220px\'},\n                layout=widgets.Layout(width=\'100%\', height=\'22px\')\n            )\n            _per_task_bars[t.id] = bar\n    \n    # Update active bars with live stats\n    for t in active:\n        bar = _per_task_bars.get(t.id)\n        if not bar:\n            continue\n        stats = download_stats.get(t.id)\n        if stats:\n            pct = stats[\'pct\']\n            speed = stats[\'speed_mbs\']\n            bar.value = pct\n            name = t.filename[:30] if t.filename else \'download\'\n            if stats.get(\'disk_wait\'):\n                bar.bar_style = \'info\'\n                bar.description = f"⏸️ {name}  {int(pct)}% (low disk)"\n            else:\n                bar.bar_style = \'warning\'\n                task_speed = f"{speed:.1f} MB/s" if speed > 0 else "starting..."\n                bar.description = f"{name}  {int(pct)}% ({task_speed})"\n\n    # Bars for tasks handed to the mover thread (download done, Drive move pending)\n    for t in tasks:\n        if t.status == "moving":\n            bar = _per_task_bars.get(t.id)\n            if bar:\n                bar.value = 100\n                bar.bar_style = \'info\'\n                name = t.filename[:35] if t.filename else \'download\'\n                bar.description = f"📤 {name}"\n\n    # Mark completed/failed/skipped bars for linger\n    for task_id in list(_per_task_bars.keys()):\n        if task_id in finished_ids and task_id not in _per_task_done_at:\n            _per_task_done_at[task_id] = now\n            bar = _per_task_bars[task_id]\n            task = next((t for t in tasks if t.id == task_id), None)\n            if not task:\n                continue\n            bar.value = 100\n            name = task.filename[:35] if task.filename else \'download\'\n            if task.status == \'failed\':\n                bar.bar_style = \'danger\'\n                bar.description = f"❌ {name}"\n            elif task.status == \'skipped\':\n                bar.bar_style = \'success\'\n                bar.description = f"⏭️ {name}"\n            else:\n                bar.bar_style = \'success\'\n                bar.description = f"✅ {name}"\n    \n    # Remove bars that have lingered long enough (race-safe: .pop instead of del)\n    for task_id in list(_per_task_done_at.keys()):\n        if now - _per_task_done_at.get(task_id, now) >= _PER_TASK_LINGER:\n            bar = _per_task_bars.pop(task_id, None)\n            if bar:\n                bar.close()\n            _per_task_done_at.pop(task_id, None)\n    \n    # Sync container children and accordion visibility\n    visible_bars = list(_per_task_bars.values())\n    if visible_bars:\n        _per_task_box.children = visible_bars\n        active_count = len(active)\n        if active_count > 0:\n            agg_speed = f"{display_speed:.1f} MB/s" if display_speed > 0 else "starting..."\n            _per_task_accordion.set_title(0, f"📥 {active_count} active download{\'s\' if active_count != 1 else \'\'} ({agg_speed})")\n        else:\n            _per_task_accordion.set_title(0, "📥 Finishing...")\n        _per_task_accordion.layout.display = \'block\'\n    else:\n        _per_task_box.children = []\n        _per_task_accordion.layout.display = \'none\'\n\ndef progress_monitor(tasks: List[DownloadTask], interval: float = 0.5):\n    """Background thread to update progress display periodically."""\n    global stop_monitor\n    while not stop_monitor:\n        try:\n            update_progress_display(tasks)\n            time.sleep(interval)\n        except Exception:\n            pass\n\n\ndef _run_download_pipeline(\n    all_tasks: List[DownloadTask],\n    parallel_tasks: List[DownloadTask],\n    youtube_urls: List[str],\n    mega_urls: List[str],\n    debrid_urls: List[str],\n    mode: str,\n    gofile_token: str,\n    rd_key: str,\n    max_workers: int,\n    magnet_file_tasks: Optional[List[DownloadTask]] = None,\n    tb_magnet_file_tasks: Optional[List[DownloadTask]] = None,\n    tb_key: str = ""\n) -> Tuple[int, int]:\n    """\n    Shared download orchestration for parallel and sequential downloads.\n\n    Returns: (total_success, total_failed) counts\n    """\n    global yt_success_cumulative, yt_fail_cumulative, stop_monitor, batch_start_time\n    import threading\n\n    start_keep_alive()\n    download_stats.clear()  # Drop progress entries from any previous batch\n    _clear_per_task_bars()\n    _reset_cancel_state()\n    # Point the user at the kernel interrupt for stopping (hidden in the finally).\n    # There is no per-cell ■ button during a download because it runs inside a widget\n    # callback, not a cell execution — the menu/shortcut interrupt is the way.\n    stop_hint.value = ("<div style=\'padding:4px 8px;background:#5a1e1e;border-radius:4px;"\n                       "display:inline-block\'>⏹ <b>To stop:</b> menu <b>Runtime → Interrupt execution</b> "\n                       "&nbsp;(shortcut <b>Ctrl+M&nbsp;I</b>, Mac <b>⌘+M&nbsp;I</b>) — progress is saved for Resume/Retry.</div>")\n    stop_hint.layout.display = \'block\'\n\n    def save_progress(throttle: bool = True):\n        """Persist batch state; throttled by default so per-task completions\n        don\'t hammer the (slow) Drive FUSE mount with JSON writes."""\n        save_session(all_tasks,\n                     playlist_range=playlist_selection.value.strip(),\n                     yt_success=yt_success_cumulative,\n                     yt_fail=yt_fail_cumulative,\n                     subtitle_langs_value=subtitle_langs.value,\n                     throttle=throttle)\n\n    # tb_magnet_file tasks come from torbox.app share links (which resolve via\n    # TorBox regardless of the debrid toggle) or TorBox magnet selections — so\n    # downloading them must not depend on the toggle either.\n    tb_file_key = tb_key or token_tb.value.strip()\n\n    # --- TORBOX: cached items download in parallel, not one-by-one ---\n    if tb_magnet_file_tasks and tb_file_key:\n        tb_ready, tb_magnet_file_tasks = convert_tb_tasks_to_parallel(tb_magnet_file_tasks, tb_file_key)\n        if tb_ready:\n            print(f"🔗 TorBox: {len(tb_ready)} cached file(s) moved to the parallel queue")\n            parallel_tasks = list(parallel_tasks) + tb_ready\n\n    # --- PARALLEL DOWNLOADS ---\n    if parallel_tasks:\n        total_parallel = len(parallel_tasks)\n        _reset_drive_xfer_stats()  # Per-batch, so the summary compares like with like\n        move_queue = None\n        mover_threads = []\n        if async_moves_checkbox.value:\n            # Overlap mode: workers hand finished files to the mover pool and immediately\n            # start the next download. Mover count is deliberately independent of\n            # max_workers: the download pool is sized by the debrid plan\'s concurrent\n            # slots (3 on the TorBox entry tier), which has nothing to do with how many\n            # uploads Drive will take. Measured at 3 concurrent API uploads: ~45 MB/s\n            # each, i.e. per-stream throughput holds as streams are added, so this is\n            # where the batch time actually comes from.\n            # Local-disk cost is (movers + MOVE_QUEUE_DEPTH) finished files waiting in\n            # /content; the disk guard still pauses downloads if that runs the disk down.\n            mover_count = max(1, int(drive_movers_slider.value))\n            move_queue = queue.Queue(maxsize=MOVE_QUEUE_DEPTH)\n            for _ in range(mover_count):\n                _mt = threading.Thread(target=_drive_mover, args=(move_queue, save_progress), daemon=True)\n                _mt.start()\n                mover_threads.append(_mt)\n            print(f"⚡ Starting {total_parallel} parallel downloads (max {max_workers} concurrent, {mover_count} Drive mover(s) overlapped)...")\n        else:\n            print(f"⚡ Starting {total_parallel} parallel downloads (max {max_workers} concurrent)...")\n\n        stop_monitor = False\n        batch_start_time = time.time()\n        \n        monitor_thread = threading.Thread(target=progress_monitor, args=(parallel_tasks,), daemon=True)\n        monitor_thread.start()\n        \n        try:\n            with ThreadPoolExecutor(max_workers=max_workers) as executor:\n                future_to_task = {\n                    executor.submit(download_worker, task, gofile_token, move_queue): task\n                    for task in parallel_tasks\n                }\n                try:\n                    for future in as_completed(future_to_task):\n                        task = future_to_task[future]\n                        try:\n                            result = future.result()\n                            for i, t in enumerate(all_tasks):\n                                if t.id == result.id:\n                                    all_tasks[i] = result\n                                    break\n                            save_progress()\n                        except Exception as e:\n                            print(f"   ❌ Task failed: {str(e)[:80]}")\n                            task.status = "failed"\n                            task.error = str(e)[:100]\n                except KeyboardInterrupt:\n                    # Terminate the aria2 subprocesses HERE, before the executor\'s\n                    # shutdown(wait=True) on with-exit would otherwise block on them.\n                    # Workers see the dead process + cancel flag and return promptly.\n                    print("\\n🛑 Interrupt received — stopping active downloads (progress is saved)...")\n                    stop_active_downloads()\n            if mover_threads:\n                if _cancel_requested:\n                    # Don\'t sit through queued multi-GB moves after an interrupt. The local\n                    # files survive in /content, so Retry re-moves them without re-downloading.\n                    skipped_moves = 0\n                    while True:\n                        try:\n                            q_task, _ = move_queue.get_nowait()\n                        except queue.Empty:\n                            break\n                        q_task.status = "failed"\n                        q_task.error = "Interrupted before Drive move (file kept locally)"\n                        move_queue.task_done()\n                        skipped_moves += 1\n                    if skipped_moves:\n                        print(f"   📤 {skipped_moves} queued move(s) skipped — Retry re-moves them without re-downloading")\n                    if move_queue.unfinished_tasks:\n                        print("   ⏳ Waiting for in-flight Drive upload(s) to finish...")\n                elif move_queue.unfinished_tasks:\n                    print("📤 Downloads finished — waiting for remaining Drive uploads...")\n                # One sentinel per mover: each thread consumes exactly one and exits, so a\n                # single None would strand every other mover blocked on get() forever.\n                for _ in mover_threads:\n                    move_queue.put(None)\n                try:\n                    for _mt in mover_threads:\n                        _mt.join()\n                except KeyboardInterrupt:\n                    # Second interrupt: stop waiting. The daemon movers finish their current\n                    # file; anything they never reach stays "moving", which resume retries.\n                    print("\\n🛑 Interrupt — not waiting for Drive moves; unfinished items can be retried")\n        finally:\n            stop_monitor = True\n            monitor_thread.join(timeout=2)  # Wait for monitor\'s final tick before touching shared state\n        \n        update_progress_display(parallel_tasks)\n        _clear_per_task_bars()  # Clean up before sequential phase\n        print(f"✅ Parallel downloads complete!")\n        _xfer = _drive_xfer_summary()\n        if _xfer: print(_xfer)\n    \n    # --- SEQUENTIAL DOWNLOADS ---\n    yt_success = 0\n    yt_fail = 0\n    \n    if youtube_urls:\n        print(f"\\n▶️ Processing {len(youtube_urls)} YouTube links...")\n        playlist_url_count = sum(1 for u in youtube_urls if \'list=\' in u or \'/playlist\' in u)\n        use_playlist_range = playlist_url_count <= 1\n        if not use_playlist_range and playlist_selection.value.strip():\n            print("   ℹ️ Multiple playlist URLs detected - playlist range ignored (downloading all videos)")\n        \n        for i, url in enumerate(youtube_urls, 1):\n            if _cancel_requested:\n                print("   🛑 Stopped — remaining YouTube links left pending for resume")\n                break\n            print(f"   [{i}/{len(youtube_urls)}] {url[:60]}...")\n            s, f, total = process_youtube_link(url, mode, apply_playlist_range=use_playlist_range)\n            yt_success += s\n            yt_fail += f\n            yt_success_cumulative += s\n            yt_fail_cumulative += f\n            \n            for task in all_tasks:\n                if task.url == url and task.link_type == \'youtube\':\n                    task.status = "done" if f == 0 else "failed"\n                    break\n            save_progress()\n        \n        if yt_fail > 0:\n            print(f"   📊 YouTube: {yt_success_cumulative} succeeded, {yt_fail_cumulative} failed")\n        else:\n            print(f"   📊 YouTube: {yt_success_cumulative} succeeded")\n    \n    if mega_urls:\n        print(f"\\n☁️ Processing {len(mega_urls)} Mega links...")\n        for i, url in enumerate(mega_urls, 1):\n            if _cancel_requested:\n                print("   🛑 Stopped — remaining Mega links left pending for resume")\n                break\n            print(f"   [{i}/{len(mega_urls)}] {url[:60]}...")\n            mega_success = process_mega_link(url)\n            for t in all_tasks:\n                if t.url == url and t.link_type == \'mega\':\n                    t.status = "done" if mega_success else "failed"\n                    break\n            save_progress()\n    \n    if debrid_urls:\n        if rd_key:\n            print(f"\\n🔓 Processing {len(debrid_urls)} RD links...")\n            for i, url in enumerate(debrid_urls, 1):\n                if _cancel_requested:\n                    print("   🛑 Stopped — remaining links left pending for resume")\n                    break\n                print(f"   [{i}/{len(debrid_urls)}] {url[:60]}...")\n                ok = process_rd_link(url, rd_key)\n                for t in all_tasks:\n                    if t.url == url and t.link_type == \'magnet\':\n                        t.status = "done" if ok else "failed"\n                        break\n                save_progress()\n        elif tb_key:\n            print(f"\\n🔓 Processing {len(debrid_urls)} TorBox links...")\n            for i, url in enumerate(debrid_urls, 1):\n                if _cancel_requested:\n                    print("   🛑 Stopped — remaining links left pending for resume")\n                    break\n                print(f"   [{i}/{len(debrid_urls)}] {url[:60]}...")\n                ok = process_tb_link(url, tb_key)\n                for t in all_tasks:\n                    if t.url == url:\n                        t.status = "done" if ok else "failed"\n                        break\n                save_progress()\n\n    # --- MAGNET FILE TASKS (RD) --- (process_* sets each task\'s status per file)\n    if magnet_file_tasks and rd_key and not _cancel_requested:\n        print(f"\\n🧲 Processing {len(magnet_file_tasks)} selected magnet files (RD)...")\n        process_magnet_file_tasks(magnet_file_tasks, rd_key)\n\n    # --- MAGNET FILE TASKS (TorBox) ---\n    if tb_magnet_file_tasks and tb_file_key and not _cancel_requested:\n        print(f"\\n🧲 Processing {len(tb_magnet_file_tasks)} selected magnet files (TorBox)...")\n        process_tb_magnet_file_tasks(tb_magnet_file_tasks, tb_file_key)\n    elif tb_magnet_file_tasks and not tb_file_key:\n        print(f"\\n❌ TorBox token required for {len(tb_magnet_file_tasks)} TorBox file(s) — enter TB Token, then Resume/Retry")\n    \n    # --- SUMMARY ---\n    save_progress(throttle=False)  # Final state must always hit disk (throttled saves may have skipped it)\n    done_count = sum(1 for t in all_tasks if t.status == \'done\')\n    failed_count = sum(1 for t in all_tasks if t.status == \'failed\')\n    \n    # Adjust for YouTube individual counts\n    total_success = done_count - len([t for t in all_tasks if t.link_type == \'youtube\' and t.status == \'done\']) + yt_success_cumulative\n    total_failed = failed_count - len([t for t in all_tasks if t.link_type == \'youtube\' and t.status == \'failed\']) + yt_fail_cumulative\n    \n    return total_success, total_failed\n\n\n# --- AUTO RETRY ---\n# Optional "Auto Retry" field: when set to N, a batch that ends with failures\n# automatically re-runs the 🔁 Retry Failed path until nothing is left failed or\n# N extra passes have run, whichever comes first. State is module-level because\n# one batch spans several functions (queue preview → execute_selected_tasks →\n# chained execute_batch resumes): the failure handlers set \'pending\', and each\n# batch function consumes it at its tail — after the finally block — so the next\n# pass starts with keep-alive, cancel state, and buttons fully reset.\n_auto_retry_state = {\'remaining\': 0, \'total\': 0, \'pending\': False}\n\ndef _arm_auto_retry():\n    """Read the Auto Retry field at a user-initiated batch start (fresh budget).\n    Chained retries skip this so they consume the budget instead of refreshing it."""\n    try:\n        n = max(0, int(auto_retry_input.value.strip()))\n    except ValueError:\n        n = 0  # empty or non-numeric = feature off\n    _auto_retry_state.update(remaining=n, total=n, pending=False)\n\ndef _run_auto_retry_chain(mode: str):\n    """Fire a queued auto retry — equivalent to clicking 🔁 Retry Failed. The grace\n    pause gives the kernel interrupt a window to cancel the chain between passes."""\n    if not _auto_retry_state[\'pending\']:\n        return\n    _auto_retry_state[\'pending\'] = False\n    _auto_retry_state[\'remaining\'] -= 1\n    attempt = _auto_retry_state[\'total\'] - _auto_retry_state[\'remaining\']\n    try:\n        print(f"\\n🔁 Auto Retry {attempt}/{_auto_retry_state[\'total\']} starting in 5s — interrupt (Ctrl+M I, Mac ⌘+M I) to cancel...")\n        for _ in range(5):\n            time.sleep(1)  # 1s steps so an interrupt lands promptly\n    except KeyboardInterrupt:\n        _auto_retry_state[\'remaining\'] = 0\n        print("\\n🛑 Auto Retry cancelled — session kept, use 🔁 Retry Failed to continue manually.")\n        return\n    execute_batch(mode, resume=True, auto_retry_chain=True)\n\n\ndef execute_selected_tasks(selected_tasks: List[DownloadTask], mode: str):\n    """Execute download for selected tasks from queue."""\n    global yt_success_cumulative, yt_fail_cumulative\n    yt_success_cumulative = 0\n    yt_fail_cumulative = 0\n    _arm_auto_retry()\n\n    clear_output(wait=True)\n    display(input_ui)\n    settings_ui.layout.display = \'none\'\n    btn.disabled = True\n    btn_quick.disabled = True\n    btn_resume.disabled = True\n    btn_retry.layout.display = \'none\'\n\n    start_keep_alive()\n    try:\n        gofile_token = token_gf.value.strip()\n        debrid_service, rd_key, tb_key = get_active_debrid()\n        max_workers = concurrent_slider.value\n\n        # Quick Download skips the queue preview — run TMDB matching here if it hasn\'t run.\n        # (Do NOT re-run analyze_batch_episodes here: it would re-analyze on the selected\n        # subset and could lose the batch detection computed at preview time.)\n        if tmdb_is_enabled():\n            if not _tmdb_match_cache:  # Quick Download path — no queue preview ran\n                analyze_batch_metadata([t.filename for t in selected_tasks if t.filename])\n            _apply_tmdb_overrides(selected_tasks)  # honor any manual corrections\n        _apply_queue_overrides(selected_tasks)  # independent of TMDB — works either way\n\n        # Separate by type: everything without a sequential processor goes parallel\n        parallel_tasks = [t for t in selected_tasks if t.link_type not in SEQUENTIAL_LINK_TYPES]\n        youtube_urls = [t.url for t in selected_tasks if t.link_type == \'youtube\']\n        mega_urls = [t.url for t in selected_tasks if t.link_type == \'mega\']\n        debrid_urls = [t.url for t in selected_tasks if t.link_type == \'magnet\']\n        magnet_file_tasks = [t for t in selected_tasks if t.link_type == \'magnet_file\']\n        tb_magnet_file_tasks = [t for t in selected_tasks if t.link_type == \'tb_magnet_file\']\n        \n        # Resolve FShare download links sequentially before parallel download\n        # (deferred from Resolve Links phase so user can review queue first)\n        fshare_unresolved = [t for t in parallel_tasks if t.link_type == \'fshare\' and \'fshare.vn/file/\' in t.url]\n        if fshare_unresolved:\n            fshare_email = token_fshare_email.value.strip()\n            fshare_password = token_fshare_password.value.strip()\n            if fshare_email and fshare_password:\n                session = _get_fshare_web_session(fshare_email, fshare_password)\n                if session:\n                    total_fshare = len(fshare_unresolved)\n                    print(f"🔄 Resolving {total_fshare} FShare download link(s)...")\n                    print(f"   ⚠️ Each resolved link counts toward your daily FShare download limit")\n                    resolved_count = 0\n                    limit_reached = False\n                    consecutive_failures = 0\n                    for i, task in enumerate(fshare_unresolved, 1):\n                        print(f"   [{i}/{total_fshare}] {task.filename[:60]}{\'...\' if len(task.filename) > 60 else \'\'}")\n                        dl_link = _fshare_web_get_download_link(task.url, session)\n                        if dl_link == "FSHARE_LIMIT_REACHED":\n                            print(f"   🛑 FShare download policy restriction — stopping resolution")\n                            print(f"   💡 Try again later or check your FShare account status")\n                            task.status = "failed"\n                            task.error = "FShare policy restriction"\n                            # Mark remaining tasks as failed too\n                            for remaining in fshare_unresolved[i:]:\n                                remaining.status = "failed"\n                                remaining.error = "FShare policy restriction"\n                            limit_reached = True\n                            break\n                        elif dl_link:\n                            task.url = dl_link\n                            resolved_count += 1\n                            consecutive_failures = 0\n                            print(f"   ✅ Resolved")\n                        else:\n                            print(f"   ⚠️ Could not resolve — will skip")\n                            task.status = "failed"\n                            task.error = "Could not resolve FShare download link"\n                            consecutive_failures += 1\n                            if consecutive_failures >= 3:\n                                print(f"   🛑 3 consecutive failures — stopping resolution")\n                                print(f"   💡 FShare may be temporarily blocking requests")\n                                for remaining in fshare_unresolved[i:]:\n                                    remaining.status = "failed"\n                                    remaining.error = "Skipped (consecutive failures)"\n                                limit_reached = True\n                                break\n                        time.sleep(1)  # Rate limiting\n                    # Remove failed FShare tasks from parallel\n                    parallel_tasks = [t for t in parallel_tasks if t.status != "failed"]\n                    if limit_reached:\n                        print(f"   📊 {resolved_count}/{total_fshare} resolved before limit was reached\\n")\n                    else:\n                        print(f"   📊 {resolved_count}/{total_fshare} FShare links resolved\\n")\n                else:\n                    print("   ❌ FShare login failed — skipping FShare downloads")\n                    parallel_tasks = [t for t in parallel_tasks if t.link_type != \'fshare\']\n        \n        all_tasks = selected_tasks.copy()\n        \n        total_parallel = len(parallel_tasks)\n        total_sequential = len(youtube_urls) + len(mega_urls) + len(debrid_urls) + len(magnet_file_tasks) + len(tb_magnet_file_tasks)\n        print(f"📊 Starting: {total_parallel} parallel + {total_sequential} sequential\\n")\n        \n        # Run shared download pipeline\n        total_success, total_failed = _run_download_pipeline(\n            all_tasks=all_tasks,\n            parallel_tasks=parallel_tasks,\n            youtube_urls=youtube_urls,\n            mega_urls=mega_urls,\n            debrid_urls=debrid_urls,\n            mode=mode,\n            gofile_token=gofile_token,\n            rd_key=rd_key,\n            max_workers=max_workers,\n            magnet_file_tasks=magnet_file_tasks,\n            tb_magnet_file_tasks=tb_magnet_file_tasks,\n            tb_key=tb_key\n        )\n        \n        # Handle results\n        if total_failed > 0:\n            failed_files = [t.filename for t in all_tasks if t.status == \'failed\']\n            if cancel_requested():\n                print(f"\\n🛑 Batch stopped by user — {total_success} completed, {total_failed} cancelled/failed.")\n                _auto_retry_state[\'remaining\'] = 0  # user stopped — don\'t auto-retry\n            else:\n                print(f"\\n⚠️ Completed with {total_success} success, {total_failed} failed after 3 attempts:")\n                for f in failed_files[:5]:\n                    print(f"   ❌ {f[:60]}")\n                if len(failed_files) > 5:\n                    print(f"   ... and {len(failed_files) - 5} more")\n\n            save_session(all_tasks,\n                        playlist_range=playlist_selection.value.strip(),\n                        yt_success=yt_success_cumulative, yt_fail=yt_fail_cumulative)\n            print(f"\\n💾 Session saved. Click \'🔁 Retry Failed\' to try again, or \'Clear Session\' in Settings to mark complete.")\n            btn_restart.layout.display = \'inline-block\'\n            btn_retry.layout.display = \'inline-block\'\n            if _auto_retry_state[\'remaining\'] > 0:\n                _auto_retry_state[\'pending\'] = True  # chain fires after cleanup (see function tail)\n        else:\n            print(f"\\n✅ All {total_success} downloads completed successfully!")\n            clear_session()\n            btn_restart.layout.display = \'none\'\n            btn_retry.layout.display = \'none\'\n            yt_success_cumulative = 0\n            yt_fail_cumulative = 0\n\n    except KeyboardInterrupt:\n        # User hit the kernel interrupt — stop cleanly and keep the session for retry.\n        # selected_tasks holds the same task objects the pipeline mutated (shallow copy).\n        stop_active_downloads()\n        _auto_retry_state[\'remaining\'] = 0  # user stopped — don\'t auto-retry\n        print(f"\\n🛑 Stopped by user (kernel interrupt). Progress saved.")\n        save_session(selected_tasks,\n                    playlist_range=playlist_selection.value.strip(),\n                    yt_success=yt_success_cumulative, yt_fail=yt_fail_cumulative)\n        print(f"💾 Session saved. Click \'🔁 Retry Failed\' to continue.")\n        btn_restart.layout.display = \'inline-block\'\n        btn_retry.layout.display = \'inline-block\'\n    except Exception as e:\n        print(f"\\n❌ Critical Error: {e}")\n    finally:\n        stop_keep_alive()\n        stop_hint.layout.display = \'none\'\n        _reset_cancel_state()\n        btn.disabled = False\n        btn_quick.disabled = False\n        btn_resume.disabled = False\n        reset_progress()\n        check_resume_available()\n\n    # Auto Retry fires outside try/finally so keep-alive, cancel state, and buttons\n    # are fully reset before the next pass (mirrors clicking 🔁 Retry Failed).\n    _run_auto_retry_chain(mode)\n\n\ndef execute_batch(mode: str, resume: bool = False, quick_mode: bool = False, auto_retry_chain: bool = False):\n    global yt_success_cumulative, yt_fail_cumulative  # Must be at function start\n    if not auto_retry_chain:\n        _arm_auto_retry()  # user-initiated (Resolve/Quick/Resume/Retry) — fresh retry budget\n    queue_open = False  # True while the queue preview is waiting for user input\n    all_tasks = []  # Ensure defined for the KeyboardInterrupt handler even if we stop early\n    clear_output(wait=True)\n    display(input_ui)\n    settings_ui.layout.display = \'none\'  # Close settings panel if open\n    btn.disabled = True\n    btn_quick.disabled = True\n    btn_resume.disabled = True\n    btn_retry.layout.display = \'none\'\n    print(f"\\n🚀 Initializing... (Mode: {mode}, Resume: {resume})")\n    \n    start_keep_alive()\n    try:\n        gofile_token = token_gf.value.strip()\n        debrid_service, rd_key, tb_key = get_active_debrid()\n        max_workers = concurrent_slider.value\n        \n        # Load from session or parse new URLs\n        if resume:\n            session_data = load_session()\n            if not session_data:\n                print("❌ No session to resume!")\n                return\n            \n            # Tokens are not persisted in the session (plaintext on Drive) — use the\n            # current widget/Colab Secrets values. Legacy sessions may still carry\n            # tokens; fall back to those only when nothing is configured now.\n            gofile_token = gofile_token or session_data.get(\'gofile_token\', \'\')\n            rd_key = rd_key or session_data.get(\'rd_token\', \'\')\n            tb_key = tb_key or session_data.get(\'tb_token\', \'\')\n            # Restore playlist range from session\n            saved_playlist_range = session_data.get(\'playlist_range\', \'\')\n            if saved_playlist_range:\n                playlist_selection.value = saved_playlist_range\n                print(f"   🎯 Restored playlist range: {saved_playlist_range}")\n            # Restore subtitle language selection from session\n            saved_subtitle_langs = session_data.get(\'subtitle_langs\', None)\n            if saved_subtitle_langs:\n                subtitle_langs.value = tuple(saved_subtitle_langs)\n                print(f"   🔤 Restored subtitle languages: {\', \'.join(saved_subtitle_langs)}")\n            # Restore cumulative YouTube counters\n            # Only restore success count - reset fail count so previous 403s don\'t persist\n            yt_success_cumulative = session_data.get(\'yt_success\', 0)\n            yt_fail_cumulative = 0  # Reset failures - only count failures in current run\n            all_tasks = [task_from_dict(t) for t in session_data.get(\'tasks\', [])]\n            \n            # Filter to unfinished tasks (downloading/moving = was active when the runtime\n            # crashed or was interrupted; retrying a \'moving\' task is cheap — the duplicate\n            # check finds the already-moved Drive copy, or the local file skips the download)\n            pending_tasks = [t for t in all_tasks if t.status in [\'pending\', \'failed\', \'downloading\', \'moving\']]\n            print(f"📂 Resuming {len(pending_tasks)} of {len(all_tasks)} tasks...")\n\n            # Fresh runtime — re-run TMDB matching, then reapply saved manual corrections\n            if tmdb_is_enabled():\n                analyze_batch_metadata([t.filename for t in pending_tasks if t.filename])\n                _apply_tmdb_overrides(pending_tasks)\n            _apply_queue_overrides(pending_tasks)  # forced seasons/episodes persist across resume\n\n            # Install required tools first\n            needs_pixeldrain_gofile_rd_tb = any(t.link_type in [\'gofile\', \'pixeldrain\', \'rd\', \'tb\'] for t in pending_tasks)\n            needs_fshare = any(t.link_type == \'fshare\' for t in pending_tasks)\n            needs_ytdlp = any(t.link_type in [\'youtube\', \'archive\'] for t in pending_tasks)\n            needs_mega = any(t.link_type == \'mega\' for t in pending_tasks)\n            needs_aria = any(t.link_type not in [\'youtube\', \'mega\', \'magnet\'] for t in pending_tasks)\n            setup_environment(needs_mega, needs_ytdlp, needs_aria)\n            \n            # Re-resolve Gofile/Pixeldrain/RD/TB URLs to get fresh API tokens (bypasses IP rate limits)\n            if needs_pixeldrain_gofile_rd_tb:\n                print("🔄 Re-resolving links with fresh session...")\n                s, t = get_gofile_session(gofile_token)\n                # rd/tb tasks keep their original service on resume regardless of\n                # the debrid toggle (mirrors the resolve/download phases) — fall\n                # back to the token widgets when the toggle-derived key is empty.\n                rd_refresh_key = rd_key or token_rd.value.strip()\n                tb_refresh_key = tb_key or token_tb.value.strip()\n                \n                for task in pending_tasks:\n                    if task.original_url and task.link_type in [\'gofile\', \'pixeldrain\', \'rd\', \'tb\']:\n                        try:\n                            if task.link_type == \'gofile\':\n                                resolved = resolve_gofile(task.original_url, s, t)\n                                if resolved:\n                                    task.url = resolved[0][0]  # Update with fresh API URL\n                                    task.cookie = t.get(\'token\')\n                            elif task.link_type == \'pixeldrain\':\n                                resolved = resolve_pixeldrain(task.original_url, s)\n                                if resolved:\n                                    task.url = resolved[0][0]  # Update with fresh API URL\n                            elif task.link_type == \'rd\' and rd_refresh_key:\n                                resolved = resolve_rd_link(task.original_url, rd_refresh_key)\n                                if resolved:\n                                    task.url = resolved[0][0]  # Update with fresh API URL\n                            elif task.link_type == \'tb\' and tb_refresh_key:\n                                resolved = resolve_tb_link(task.original_url, tb_refresh_key)\n                                if resolved:\n                                    task.url = resolved[0][0]  # Update with fresh API URL\n                        except Exception as e:\n                            print(f"   ⚠️ Could not re-resolve {task.filename}: {e}")\n            \n            # Re-resolve FShare URLs with fresh login session\n            if needs_fshare:\n                fshare_email = token_fshare_email.value.strip()\n                fshare_password = token_fshare_password.value.strip()\n                if fshare_email and fshare_password:\n                    fshare_tasks = [t for t in pending_tasks if t.original_url and t.link_type == \'fshare\']\n                    if fshare_tasks:\n                        print(f"🔄 Re-resolving {len(fshare_tasks)} FShare link(s)...")\n                        print(f"   ⚠️ Note: Each resolved link counts toward your daily FShare download limit")\n                        for task in fshare_tasks:\n                            try:\n                                resolved = resolve_fshare(task.original_url, fshare_email, fshare_password)\n                                if resolved:\n                                    task.url = resolved[0][0]\n                            except Exception as e:\n                                print(f"   ⚠️ Could not re-resolve FShare {task.filename}: {e}")\n            \n            # Separate by type: everything without a sequential processor goes parallel\n            parallel_tasks = [t for t in pending_tasks if t.link_type not in SEQUENTIAL_LINK_TYPES]\n            youtube_urls = [t.url for t in pending_tasks if t.link_type == \'youtube\']\n            mega_urls = [t.url for t in pending_tasks if t.link_type == \'mega\']\n            debrid_urls = [t.url for t in pending_tasks if t.link_type == \'magnet\']\n            magnet_file_tasks = [t for t in pending_tasks if t.link_type == \'magnet_file\']\n            tb_magnet_file_tasks = [t for t in pending_tasks if t.link_type == \'tb_magnet_file\']\n        else:\n            urls = [x.strip() for x in text_area.value.split(\'\\n\') if x.strip()]\n            if not urls:\n                print("❌ No links provided!")\n                btn.disabled = False\n                return\n\n            ytdlp_hosts = STREAMING_HOSTS + (\'tiktok.com\', \'dailymotion.com\', \'soundcloud.com\')\n            needs_ytdlp = any(url_matches_host(u, ytdlp_hosts) for u in urls) or \\\n                          any(url_matches_host(u, (\'archive.org\',)) and \'/details/\' in u for u in urls)\n            needs_mega = any(url_matches_host(u, (\'mega.nz\', \'transfer.it\')) for u in urls)\n            needs_aria = not (needs_ytdlp and not needs_mega) or any(\n                url_matches_host(u, (\'gofile.io\', \'pixeldrain.com\', \'real-debrid.com\', \'mega.nz\', \'fshare.vn\'))\n                or u.startswith(\'magnet:\') for u in urls)\n\n            setup_environment(needs_mega, needs_ytdlp, needs_aria)\n            \n            s, t = get_gofile_session(gofile_token)\n            \n            print(f"🔍 Resolving {len(urls)} links...")\n            parallel_tasks, youtube_urls, mega_urls, debrid_urls = resolve_all_links(urls, s, t, rd_key, tb_key, debrid_service)\n            \n            # Create session-compatible task list for saving\n            all_tasks = parallel_tasks.copy()\n            \n            # Expand YouTube playlists into individual video tasks for queue display\n            for url in youtube_urls:\n                yt_tasks = resolve_youtube_playlist(url)\n                all_tasks.extend(yt_tasks)\n            \n            for url in mega_urls:\n                all_tasks.append(DownloadTask(url=url, filename="", source="mega", link_type="mega"))\n            for url in debrid_urls:\n                # Distinguish between magnet links and other debrid-related links\n                if url.startswith("magnet:"):\n                    all_tasks.append(DownloadTask(url=url, filename="", source="debrid", link_type="magnet"))\n                else:\n                    debrid_type = "tb" if debrid_service == \'tb\' else "rd"\n                    all_tasks.append(DownloadTask(url=url, filename="", source="debrid", link_type=debrid_type))\n\n            # Save initial session\n            save_session(all_tasks,\n                        playlist_range=playlist_selection.value.strip())\n\n            if quick_mode:\n                # Quick Download: Skip queue preview, start immediately\n                print(f"⚡ Quick Download: Starting {len(all_tasks)} items...")\n                execute_selected_tasks(all_tasks, mode)\n            else:\n                # Show queue preview instead of immediate download\n                show_queue_preview(all_tasks, mode)\n                queue_open = True  # Keep buttons disabled until the queue is acted on\n            return  # Wait for user to click "Start Selected"\n        \n        # This code only runs for RESUME mode (preview was skipped)\n        total_parallel = len(parallel_tasks)\n        total_sequential = len(youtube_urls) + len(mega_urls) + len(debrid_urls) + len(magnet_file_tasks) + len(tb_magnet_file_tasks)\n        print(f"📊 Tasks: {total_parallel} parallel + {total_sequential} sequential\\n")\n\n        # Run shared download pipeline\n        total_success, total_failed = _run_download_pipeline(\n            all_tasks=all_tasks,\n            parallel_tasks=parallel_tasks,\n            youtube_urls=youtube_urls,\n            mega_urls=mega_urls,\n            debrid_urls=debrid_urls,\n            mode=mode,\n            gofile_token=gofile_token,\n            rd_key=rd_key,\n            max_workers=max_workers,\n            magnet_file_tasks=magnet_file_tasks,\n            tb_magnet_file_tasks=tb_magnet_file_tasks,\n            tb_key=tb_key\n        )\n        \n        # Handle results\n        if total_failed > 0:\n            if cancel_requested():\n                print(f"\\n🛑 Batch stopped by user — {total_success} completed, {total_failed} cancelled/failed (session saved)")\n                _auto_retry_state[\'remaining\'] = 0  # user stopped — don\'t auto-retry\n            else:\n                print(f"\\n⚠️ Completed with {total_success} success, {total_failed} failed (session saved for retry)")\n            btn_restart.layout.display = \'inline-block\'\n            btn_retry.layout.display = \'inline-block\'\n            if _auto_retry_state[\'remaining\'] > 0:\n                _auto_retry_state[\'pending\'] = True  # chain fires after cleanup (see function tail)\n        else:\n            print(f"\\n✅ All {total_success} downloads completed successfully!")\n            clear_session()\n            btn_restart.layout.display = \'none\'\n            btn_retry.layout.display = \'none\'\n            yt_success_cumulative = 0\n            yt_fail_cumulative = 0\n\n    except KeyboardInterrupt:\n        # User hit the kernel interrupt (resume path) — stop cleanly, keep session\n        stop_active_downloads()\n        _auto_retry_state[\'remaining\'] = 0  # user stopped — don\'t auto-retry\n        print(f"\\n🛑 Stopped by user (kernel interrupt). Progress saved.")\n        if all_tasks:\n            save_session(all_tasks,\n                        playlist_range=playlist_selection.value.strip(),\n                        yt_success=yt_success_cumulative, yt_fail=yt_fail_cumulative)\n        print(f"💾 Session saved. Click \'🔁 Retry Failed\' to continue.")\n        btn_restart.layout.display = \'inline-block\'\n        btn_retry.layout.display = \'inline-block\'\n    except Exception as e:\n        print(f"\\n❌ Critical Error: {e}")\n    finally:\n        stop_keep_alive()\n        stop_hint.layout.display = \'none\'\n        _reset_cancel_state()\n        if not queue_open:\n            # While the queue preview is open the buttons stay disabled;\n            # hide_queue()/start_from_queue() re-enable them later.\n            btn.disabled = False\n            btn_quick.disabled = False\n            btn_resume.disabled = False\n            reset_progress()\n        check_resume_available()\n\n    # Auto Retry fires outside try/finally so keep-alive, cancel state, and buttons\n    # are fully reset before the next pass (mirrors clicking 🔁 Retry Failed).\n    _run_auto_retry_chain(mode)\n\n\n\n# --- QUICK DOWNLOAD ---\ndef on_quick_download(b=None):\n    """Quick Download - bypass queue preview and download immediately."""\n    urls = text_area.value.strip()\n    if not urls:\n        print("⚠️ No links to download!")\n        return\n    \n    # Setup subtitle languages for Quick Download if enabled\n    if quick_dl_subs_checkbox.value:\n        subtitle_langs.value = quick_dl_subtitle_langs.value\n\n    # Call execute_batch with quick_mode to skip queue preview\n    execute_batch("video", quick_mode=True)\n\n# --- BINDINGS ---\nbtn.on_click(lambda b: execute_batch("video"))\nbtn_quick.on_click(on_quick_download)\nbtn_resume.on_click(lambda b: execute_batch("video", resume=True))\nbtn_retry.on_click(lambda b: execute_batch("video", resume=True))  # Retry = resume machinery, no restart needed\nbtn_restart.on_click(restart_runtime)\nbtn_history.on_click(view_history)\n\n# Queue control bindings\nbtn_queue_up.on_click(queue_move_up)\nbtn_queue_down.on_click(queue_move_down)\nbtn_queue_select_all.on_click(queue_select_all)\nbtn_queue_select_none.on_click(queue_select_none)\nbtn_queue_remove.on_click(queue_remove_selected)\nbtn_queue_cancel.on_click(queue_cancel)\nbtn_queue_sort.on_click(queue_sort_alpha)\nbtn_tmdb_match.on_click(apply_tmdb_override)\nbtn_tmdb_clear.on_click(clear_tmdb_override)\nbtn_season_apply.on_click(apply_season_override)\nbtn_season_clear.on_click(clear_season_override)\nbtn_renumber.on_click(apply_renumber)\nbtn_renumber_clear.on_click(clear_renumber)\nbtn_part_apply.on_click(apply_part_override)\nbtn_part_remove.on_click(remove_part_override)\nbtn_name_apply.on_click(apply_name_override)\nbtn_name_clear.on_click(clear_name_override)\nbtn_route_apply.on_click(apply_route_override)\nbtn_route_clear.on_click(clear_route_override)\nbtn_queue_start.on_click(lambda b: start_from_queue(mode="video"))\nbtn_queue_start_subs.on_click(lambda b: start_from_queue(mode="subs_only"))\n\n# Settings control bindings\nbtn_settings.on_click(toggle_settings)\nbtn_settings_close.on_click(close_settings)\nbtn_upload_cookies.on_click(upload_cookies)\nbtn_clear_cookies.on_click(clear_cookies)\nbtn_clear_history.on_click(request_clear_history)\nbtn_clear_ytarchive.on_click(request_clear_ytarchive)\nbtn_clear_session.on_click(request_clear_session)\nbtn_confirm_yes.on_click(confirm_action)\nbtn_confirm_cancel.on_click(cancel_confirmation)\n\n# Embedded subtitle extraction bindings\nbtn_extract_library.on_click(run_library_extract)\n\n# --- INITIAL SETUP ---\ndef early_mount_drive():\n    """Mount Drive on script load to enable session resume detection."""\n    drive_path = f"{COLAB_ROOT}drive"\n    if drive is not None and not os.path.exists(drive_path):\n        try:\n            print("📂 Mounting Google Drive for session detection...")\n            drive.mount(drive_path)\n        except Exception as e:\n            print(f"⚠️ Could not mount Drive: {e}")\n    check_resume_available()\n\n# Display UI first (so it shows even if mount hangs), then mount drive\ndisplay(input_ui)\nearly_mount_drive()'
exec(script)
